In [ ]:

# Configuration O+ (CORA only): CAST v1 core, then the reviewed R29 package, on CPU; no neural model, no GPU, no internet.
# Cell 1 of 2: write the embedded code (hash-checked).
import hashlib, json, os
O_ENV = {'SF_OP_PACKAGES': '[{"name": "cast_v1", "entry": "cast_candidates", "cpu_s": 45, "wall_s": 90, "details": 1}, {"name": "r29_cpu_package_20261101", "entry": "candidates", "cpu_s": 12, "wall_s": 24, "details": 0}]', 'SF_O_DEADLINE_S': '39600', 'SF_O_GRACE_S': '10'}
for _k, _v in O_ENV.items():
    os.environ[_k] = _v
CODE_DIR = os.environ.get("SF_KAGGLE_ROOT", "/kaggle") + "/working/o_code"
CODE_SHA = '525344270a1bc56ea708792761e5017737cf49a75d317cd36a79ad890b46b19f'
FILES = json.loads('{"cast_v1/__init__.py": "\\"\\"\\"CAST v1 core (CORA-CAST): executable program candidates for ARC tasks. See README.md.\\"\\"\\"\\nfrom .api import VERSION, cast_candidates\\n\\n__all__ = [\\"cast_candidates\\", \\"VERSION\\"]\\n", "cast_v1/_nd.py": "\\"\\"\\"The four image operations CAST needs, in pure numpy, with scipy used as an accelerator when it imports.\\n\\n    label(mask, structure)          connected components (labels 1..n), 4- or 8-connectivity from a 3x3 structure\\n    binary_dilation(mask, structure) dilation by a 3x3 structure, cells outside the grid count as False\\n    binary_fill_holes(mask)         mask plus every background cell not 4-connected to the border\\n    components_from_edges(n, a, b)  connected components of an undirected graph given as edge arrays\\nCallers never depend on label numbering (objects are sorted afterwards), so both backends give identical results;\\ntests/test_pkg.py checks the two backends against each other. Set CAST_FORCE_NUMPY=1 to disable scipy.\\n\\"\\"\\"\\nimport os\\n\\nimport numpy as np\\n\\n_SCIPY = None\\nif os.environ.get(\\"CAST_FORCE_NUMPY\\") != \\"1\\":\\n    try:                                                     # optional accelerator\\n        from scipy import ndimage as _sn\\n        from scipy.sparse import coo_matrix as _coo\\n        from scipy.sparse.csgraph import connected_components as _cc\\n        _SCIPY = True\\n    except Exception:                                        # noqa: BLE001\\n        _SCIPY = None\\n\\n\\ndef _offsets(structure):\\n    s = np.asarray(structure, dtype=bool)\\n    return [(dr - 1, dc - 1) for dr in range(3) for dc in range(3) if s[dr, dc] and (dr, dc) != (1, 1)]\\n\\n\\ndef label(mask, structure=None):\\n    mask = np.asarray(mask, dtype=bool)\\n    if structure is None:\\n        structure = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=bool)\\n    if _SCIPY:\\n        return _sn.label(mask, structure=structure)\\n    H, W = mask.shape\\n    lab = np.zeros((H, W), dtype=np.int32)\\n    offs = _offsets(structure)\\n    n = 0\\n    for r0 in range(H):\\n        for c0 in range(W):\\n            if not mask[r0, c0] or lab[r0, c0]:\\n                continue\\n            n += 1\\n            lab[r0, c0] = n\\n            stack = [(r0, c0)]\\n            while stack:\\n                r, c = stack.pop()\\n                for dr, dc in offs:\\n                    rr, cc = r + dr, c + dc\\n                    if 0 <= rr < H and 0 <= cc < W and mask[rr, cc] and not lab[rr, cc]:\\n                        lab[rr, cc] = n\\n                        stack.append((rr, cc))\\n    return lab, n\\n\\n\\ndef binary_dilation(mask, structure=None):\\n    mask = np.asarray(mask, dtype=bool)\\n    if structure is None:\\n        structure = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=bool)\\n    if _SCIPY:\\n        return _sn.binary_dilation(mask, structure=structure)\\n    out = mask.copy()\\n    H, W = mask.shape\\n    for dr, dc in _offsets(structure):\\n        sh = np.zeros_like(mask)\\n        r0, r1 = max(0, dr), min(H, H + dr)\\n        c0, c1 = max(0, dc), min(W, W + dc)\\n        sh[r0:r1, c0:c1] = mask[r0 - dr:r1 - dr, c0 - dc:c1 - dc]\\n        out |= sh\\n    return out\\n\\n\\ndef binary_fill_holes(mask):\\n    mask = np.asarray(mask, dtype=bool)\\n    if _SCIPY:\\n        return _sn.binary_fill_holes(mask)\\n    comp = ~mask\\n    lab, n = label(comp)\\n    border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])).tolist()) - {0}\\n    holes = comp & ~np.isin(lab, list(border))\\n    return mask | holes\\n\\n\\ndef components_from_edges(n, rows, cols):\\n    rows, cols = np.asarray(rows), np.asarray(cols)\\n    if _SCIPY:\\n        g = _coo((np.ones(len(rows), dtype=np.int8), (rows, cols)), shape=(n, n))\\n        return _cc(g, directed=False)\\n    parent = np.arange(n)\\n\\n    def find(x):\\n        while parent[x] != x:\\n            parent[x] = parent[parent[x]]\\n            x = parent[x]\\n        return x\\n    for a, b in zip(rows.tolist(), cols.tolist()):\\n        ra, rb = find(a), find(b)\\n        if ra != rb:\\n            parent[max(ra, rb)] = min(ra, rb)\\n    roots = np.array([find(i) for i in range(n)])\\n    _, lab = np.unique(roots, return_inverse=True)\\n    return int(lab.max()) + 1 if n else 0, lab\\n", "cast_v1/absdom.py": "\\"\\"\\"M3 abstract domain: a reduced product of cheap domains over sets of grids.\\n\\nAn abstract value `Abs` describes a SET of grids (its concretization gamma):\\n    shapes    finite set of (h, w) the grid may have, or None = any shape with h >= shape_lo[0], w >= shape_lo[1]\\n    cells     (only when exactly one shape is possible) an (h, w) array of 10-bit masks: the colours cell p may take.\\n              This one domain subsumes the write footprint (cells whose mask is not the singleton of the reference\\n              grid\'s value may change; all others are protected) and the per-cell palette.\\n    pal       may-palette: every colour that occurs is in pal\\n    need      must-palette: every colour in need occurs\\n    lo, hi    per-colour cell-count intervals\\ngamma(A) = {y : shape(y) in shapes (or >= shape_lo), y[p] in cells[p] for all p, colours(y) <= pal,\\n            need <= colours(y), lo[k] <= count_k(y) <= hi[k]}.\\n`bottom` is the empty set. join over-approximates the union; meet is exact on each component (the product is\\nreduced by `normalize`, which only removes grids that cannot be in gamma).\\n\\nSoundness obligation of every operator contract (cast/ops.py): for an abstract input A and parameter theta,\\n    {op(theta, z) : z in gamma(A), op defined on z} is a subset of gamma(fwd(A, theta)),\\nand for a target y, {w : op(theta, w) = y} is a subset of gamma(bwd(y, theta)). Tests: tests/test_contracts_*.py.\\n\\"\\"\\"\\nimport numpy as np\\n\\nALL = (1 << 10) - 1\\nMAXC = 900\\nBITS = np.array([1 << k for k in range(10)], dtype=np.uint16)\\nPOPC = np.array([bin(i).count(\\"1\\") for i in range(1 << 10)], dtype=np.int64)\\n\\n\\ndef bit(c):\\n    return 1 << int(c)\\n\\n\\ndef mask_of(colours):\\n    m = 0\\n    for c in colours:\\n        m |= 1 << int(c)\\n    return m\\n\\n\\ndef colours_of(m):\\n    return [k for k in range(10) if m >> k & 1]\\n\\n\\nclass Abs:\\n    __slots__ = (\\"shapes\\", \\"shape_lo\\", \\"cells\\", \\"pal\\", \\"need\\", \\"lo\\", \\"hi\\", \\"bottom\\")\\n\\n    def __init__(self, shapes=None, shape_lo=(1, 1), cells=None, pal=ALL, need=0, lo=None, hi=None, bottom=False):\\n        self.shapes = None if shapes is None else frozenset(shapes)\\n        self.shape_lo = tuple(shape_lo)\\n        self.cells = cells\\n        self.pal = int(pal)\\n        self.need = int(need)\\n        self.lo = np.zeros(10, dtype=np.int64) if lo is None else np.asarray(lo, dtype=np.int64).copy()\\n        self.hi = np.full(10, MAXC, dtype=np.int64) if hi is None else np.asarray(hi, dtype=np.int64).copy()\\n        self.bottom = bottom\\n        if not bottom:\\n            normalize(self)\\n\\n    def __repr__(self):\\n        if self.bottom:\\n            return \\"Abs(BOTTOM)\\"\\n        sh = \\"any>=%s\\" % (self.shape_lo,) if self.shapes is None else sorted(self.shapes)[:4]\\n        return (f\\"Abs(shapes={sh}{\'...\' if self.shapes and len(self.shapes) > 4 else \'\'}, \\"\\n                f\\"cells={\'yes\' if self.cells is not None else \'no\'}, pal={colours_of(self.pal)}, \\"\\n                f\\"need={colours_of(self.need)})\\")\\n\\n    def copy(self):\\n        if self.bottom:\\n            return BOTTOM()\\n        return Abs(self.shapes, self.shape_lo, None if self.cells is None else self.cells.copy(), self.pal,\\n                   self.need, self.lo, self.hi)\\n\\n    @property\\n    def single_shape(self):\\n        if self.shapes is not None and len(self.shapes) == 1:\\n            return next(iter(self.shapes))\\n        return None\\n\\n    def footprint(self, ref):\\n        \\"\\"\\"Cells that may differ from the reference grid `ref` (None if not aligned).\\"\\"\\"\\n        if self.cells is None or self.cells.shape != np.asarray(ref).shape:\\n            return None\\n        return self.cells != BITS[np.asarray(ref)]\\n\\n\\ndef BOTTOM():\\n    a = Abs.__new__(Abs)\\n    a.shapes, a.shape_lo, a.cells, a.pal, a.need = frozenset(), (1, 1), None, 0, 0\\n    a.lo, a.hi, a.bottom = np.zeros(10, dtype=np.int64), np.zeros(10, dtype=np.int64), True\\n    return a\\n\\n\\ndef TOP():\\n    return Abs()\\n\\n\\nLOG2 = np.zeros(1 << 10, dtype=np.int64)\\nfor _k in range(10):\\n    LOG2[1 << _k] = _k\\nKS = np.arange(10)\\n\\n\\ndef _bits(m):\\n    return (int(m) >> KS) & 1\\n\\n\\ndef normalize(a):\\n    \\"\\"\\"Tighten the reduced product in place; set bottom when gamma is provably empty.\\"\\"\\"\\n    if a.shapes is not None:\\n        a.shapes = frozenset(s for s in a.shapes if s[0] >= a.shape_lo[0] and s[1] >= a.shape_lo[1]\\n                             and 1 <= s[0] <= 30 and 1 <= s[1] <= 30)\\n        if not a.shapes:\\n            return _bot(a)\\n        if len(a.shapes) > 1:\\n            a.cells = None\\n    if a.cells is not None:\\n        if a.shapes is None or len(a.shapes) != 1 or a.cells.shape != next(iter(a.shapes)):\\n            a.cells = None\\n    if a.cells is not None:\\n        a.cells = (a.cells & np.uint16(a.pal)).astype(np.uint16)\\n        flat = a.cells.ravel().astype(np.int64)\\n        if (flat == 0).any():\\n            return _bot(a)\\n        a.pal &= int(np.bitwise_or.reduce(flat))\\n        single = POPC[flat] == 1\\n        fixed = np.bincount(LOG2[flat[single]], minlength=10)[:10]\\n        may = ((flat[:, None] >> KS) & 1).sum(0)\\n        a.lo = np.maximum(a.lo, fixed)\\n        a.hi = np.minimum(a.hi, may)\\n    a.hi = np.where(_bits(a.pal) == 1, a.hi, 0)\\n    a.lo = np.where(_bits(a.need) == 1, np.maximum(a.lo, 1), a.lo)\\n    pos = np.nonzero(a.lo > 0)[0]\\n    for k in pos:\\n        a.need |= 1 << int(k)\\n    if a.need & ~a.pal or (a.lo > a.hi).any():\\n        return _bot(a)\\n    if a.shapes is not None:\\n        nmax = max(h * w for h, w in a.shapes)\\n        nmin = min(h * w for h, w in a.shapes)\\n        if a.lo.sum() > nmax or a.hi.sum() < nmin:\\n            return _bot(a)\\n    return a\\n\\n\\ndef _bot(a):\\n    b = BOTTOM()\\n    for s in Abs.__slots__:\\n        setattr(a, s, getattr(b, s))\\n    return a\\n\\n\\nBITW = np.array([1 << k for k in range(10)], dtype=np.int64)\\n\\n\\ndef alpha(g):\\n    \\"\\"\\"Exact abstraction of one concrete grid (already reduced; built without normalize).\\"\\"\\"\\n    g = np.asarray(g, dtype=np.int64)\\n    cnt = np.bincount(g.ravel(), minlength=10)[:10]\\n    m = int(BITW[cnt > 0].sum())\\n    a = Abs.__new__(Abs)\\n    a.shapes, a.shape_lo, a.cells = frozenset([g.shape]), (1, 1), BITS[g]\\n    a.pal = a.need = m\\n    a.lo, a.hi, a.bottom = cnt.copy(), cnt.copy(), False\\n    return a\\n\\n\\ndef contains(a, y):\\n    \\"\\"\\"y in gamma(a).\\"\\"\\"\\n    if a.bottom:\\n        return False\\n    y = np.asarray(y, dtype=np.int64)\\n    if a.shapes is not None and y.shape not in a.shapes:\\n        return False\\n    if a.shapes is None and (y.shape[0] < a.shape_lo[0] or y.shape[1] < a.shape_lo[1]):\\n        return False\\n    cnt = np.bincount(y.ravel(), minlength=10)[:10]\\n    cy = int(BITW[cnt > 0].sum())\\n    if cy & ~a.pal or a.need & ~cy:\\n        return False\\n    if (cnt < a.lo).any() or (cnt > a.hi).any():\\n        return False\\n    if a.cells is not None and not ((a.cells & BITS[y]) != 0).all():\\n        return False\\n    return True\\n\\n\\ndef join(a, b):\\n    if a.bottom:\\n        return b.copy()\\n    if b.bottom:\\n        return a.copy()\\n    if a.shapes is None or b.shapes is None:\\n        shapes = None\\n        lo_a = a.shape_lo if a.shapes is None else (min(s[0] for s in a.shapes), min(s[1] for s in a.shapes))\\n        lo_b = b.shape_lo if b.shapes is None else (min(s[0] for s in b.shapes), min(s[1] for s in b.shapes))\\n        shape_lo = (min(lo_a[0], lo_b[0]), min(lo_a[1], lo_b[1]))\\n    else:\\n        shapes = a.shapes | b.shapes\\n        shape_lo = (1, 1)\\n    cells = None\\n    if (a.cells is not None and b.cells is not None and a.cells.shape == b.cells.shape and shapes is not None\\n            and len(shapes) == 1):\\n        cells = a.cells | b.cells\\n    return Abs(shapes, shape_lo, cells, a.pal | b.pal, a.need & b.need, np.minimum(a.lo, b.lo),\\n               np.maximum(a.hi, b.hi))\\n\\n\\ndef meet(a, b):\\n    if a.bottom or b.bottom:\\n        return BOTTOM()\\n    if a.shapes is None and b.shapes is None:\\n        shapes = None\\n    elif a.shapes is None:\\n        shapes = b.shapes\\n    elif b.shapes is None:\\n        shapes = a.shapes\\n    else:\\n        shapes = a.shapes & b.shapes\\n    shape_lo = (max(a.shape_lo[0], b.shape_lo[0]), max(a.shape_lo[1], b.shape_lo[1]))\\n    cells = None\\n    if a.cells is not None and b.cells is not None:\\n        if a.cells.shape != b.cells.shape:\\n            return BOTTOM()\\n        cells = a.cells & b.cells\\n    elif a.cells is not None:\\n        cells = a.cells.copy()\\n    elif b.cells is not None:\\n        cells = b.cells.copy()\\n    return Abs(shapes, shape_lo, cells, a.pal & b.pal, a.need | b.need, np.maximum(a.lo, b.lo),\\n               np.minimum(a.hi, b.hi))\\n\\n\\ndef join_all(items):\\n    out = BOTTOM()\\n    for x in items:\\n        out = join(out, x)\\n    return out\\n\\n\\ndef is_concrete(a):\\n    return (not a.bottom and a.cells is not None and bool((POPC[a.cells] == 1).all()))\\n\\n\\ndef concretize_one(a):\\n    \\"\\"\\"The single grid of a concrete abstract value.\\"\\"\\"\\n    assert is_concrete(a)\\n    return np.log2(a.cells.astype(np.float64)).round().astype(np.int64)\\n\\n\\ndef sample(a, rng, tries=50):\\n    \\"\\"\\"A random member of gamma(a) when cells are known (rejection on the count/need constraints); else None.\\"\\"\\"\\n    if a.bottom or a.cells is None:\\n        return None\\n    opts = [[k for k in range(10) if m >> k & 1] for m in a.cells.ravel()]\\n    for _ in range(tries):\\n        y = np.array([rng.choice(o) for o in opts], dtype=np.int64).reshape(a.cells.shape)\\n        if contains(a, y):\\n            return y\\n    return None\\n", "cast_v1/api.py": "\\"\\"\\"CAST v1 core for the Kaggle entry (configuration B): one call, hard caps, deterministic, fail-safe.\\n\\n    from cast_v1 import cast_candidates\\n    grids = cast_candidates(task)              # list over test inputs: a grid (list of lists) or None\\n    info = cast_candidates(task, details=True) # {\\"grids\\": [...], \\"module\\": [...], \\"program\\": [...], \\"status\\": ...}\\n\\n`task` is a Kaggle/ARC task dict ({\\"train\\": [{\\"input\\", \\"output\\"}, ...], \\"test\\": [{\\"input\\"}, ...]}). Test outputs are\\nnever read. Modules, in this fixed order (the frozen CAST v1 core): M4 general symmetry completion (complete\\npredictions only), M2 typed guarded rewriting and guarded frames, L0 contract-guided search (depth 2, 1,000 executions).\\nGeneral acceptance rule (task-independent; no task ids, no special cases). A module\'s first attempt for a query is\\nreturned only if\\n  1. its program reproduces EVERY demonstration exactly (by construction of every module);\\n  2. the answer is complete: a valid grid (1..30 x 1..30, colours 0..9), not the unchanged query input, and for M4 no\\n     undetermined cell;\\n  3. leave-one-demonstration-out passes: with each demonstration held out in turn, the same module rebuilt from the\\n     remaining demonstrations ranks that demonstration\'s output among its top two attempts (tasks with fewer than two\\n     demonstrations are never accepted).\\nQueries take the first accepting module in the order above; others get None. Hard caps per call: `cpu_s` of process\\nCPU (ITIMER_VIRTUAL) and `wall_s` of wall time (ITIMER_REAL), both from the main thread; any error, timeout or invalid\\ninput returns None for every query. The caller\'s signal handlers and timers are restored afterwards.\\n\\"\\"\\"\\nimport signal\\nimport threading\\nimport time\\n\\nimport numpy as np\\n\\nfrom . import absdom as D\\nfrom . import candidates as CA\\nfrom . import grids as G\\nfrom . import ops as O\\nfrom . import rewrite as RW\\nfrom . import search as SR\\nfrom . import symmetry as S\\n\\nVERSION = \\"cast-v1-core-1\\"\\nORDER = (\\"m4\\", \\"m2\\", \\"l0\\")\\n\\n\\nclass _Timeout(BaseException):\\n    pass\\n\\n\\n# ------------------------------------------------------------------ the three frozen modules\\ndef _m4_decide(op, thetas, demos):\\n    fits = []\\n    for th in thetas:\\n        if any(not D.contains(op.fwd(D.alpha(x), th), y) for x, y in demos):\\n            continue\\n        ok = True\\n        for x, y in demos:\\n            out = op.apply(x, th)\\n            if out is None or out.shape != y.shape or not np.array_equal(out, y):\\n                ok = False\\n                break\\n        if ok:\\n            fits.append(th)\\n    return fits\\n\\n\\ndef _m4(task):\\n    op = S.SymComplete(0.5, 0.5)\\n    demos = [(np.asarray(x), np.asarray(y)) for x, y in task[\\"train\\"]]\\n    qs = [np.asarray(q) for q in task[\\"test_inputs\\"]]\\n    ctx = O.TaskCtx(task[\\"train\\"], task[\\"test_inputs\\"])\\n    fits = []\\n    for k, th in enumerate(_m4_decide(op, op.domain(ctx), demos)):\\n        preds, complete = [], []\\n        for q in qs:\\n            out = op.apply(q, th)\\n            preds.append(out)\\n            complete.append(out is not None and (op.last_trace or {}).get(\\"status\\") == \\"COMPLETE\\")\\n        fits.append({\\"program\\": f\\"(symcomplete mask={th[0]} crop={th[1]})\\", \\"preds\\": preds, \\"complete\\": complete,\\n                     \\"cost\\": 1.0, \\"order\\": k, \\"loo\\": 0.0})\\n    return fits\\n\\n\\ndef _m2(task):\\n    return RW.search(task)\\n\\n\\ndef _l0(task):\\n    ctx = O.TaskCtx(task[\\"train\\"], task[\\"test_inputs\\"])\\n    r = SR.Searcher(ctx, O.L0, depth=2, mode=\\"fwdbwd\\", max_exec=1000, oe=True, max_fits=32).run()\\n    return [{\\"program\\": \\" ; \\".join(f\\"{n}{th!r}\\" for n, th in f[\\"program\\"]),\\n             \\"preds\\": [None if p is None else np.asarray(p) for p in f[\\"preds\\"]],\\n             \\"complete\\": [p is not None for p in f[\\"preds\\"]], \\"cost\\": float(len(f[\\"program\\"])), \\"order\\": i}\\n            for i, f in enumerate(r[\\"fits\\"])]\\n\\n\\nMODULES = {\\"m4\\": _m4, \\"m2\\": _m2, \\"l0\\": _l0}\\n\\n\\ndef _ranked(module, task):\\n    fits = MODULES[module](task)\\n    if module == \\"m4\\":\\n        fits = [f for f in fits if all(f[\\"complete\\"])] or [f for f in fits if any(f[\\"complete\\"])]\\n    demos = [(np.asarray(x), np.asarray(y)) for x, y in task[\\"train\\"]]\\n    qs = [np.asarray(q) for q in task[\\"test_inputs\\"]]\\n    groups = CA.rank(fits, {\\"train\\": demos, \\"test_inputs\\": qs})\\n    return groups, CA.attempts(groups, len(qs))\\n\\n\\ndef _loo_pass(module, task):\\n    if len(task[\\"train\\"]) < 2:\\n        return False\\n    for i in range(len(task[\\"train\\"])):\\n        rest = [d for j, d in enumerate(task[\\"train\\"]) if j != i]\\n        sub = {\\"train\\": rest, \\"test_inputs\\": [task[\\"train\\"][i][0]]}\\n        _, atts = _ranked(module, sub)\\n        y = np.asarray(task[\\"train\\"][i][1])\\n        if not any(np.asarray(a).shape == y.shape and np.array_equal(np.asarray(a), y) for a in atts[0]):\\n            return False\\n    return True\\n\\n\\n# ------------------------------------------------------------------ input, caps, entry point\\ndef _norm(task):\\n    train = [(p[\\"input\\"], p[\\"output\\"]) if isinstance(p, dict) else (p[0], p[1]) for p in task[\\"train\\"]]\\n    tests = task.get(\\"test_inputs\\")\\n    if tests is None:\\n        tests = [t[\\"input\\"] if isinstance(t, dict) else t for t in task[\\"test\\"]]\\n    for g in [a for a, _ in train] + [b for _, b in train] + list(tests):\\n        if not G.valid_grid(np.asarray(g)):\\n            raise ValueError(\\"invalid grid\\")\\n    return {\\"task_id\\": \\"task\\", \\"train\\": train, \\"test_inputs\\": list(tests)}\\n\\n\\nclass _Caps:\\n    def __init__(self, cpu_s, wall_s):\\n        self.cpu_s, self.wall_s, self.on = cpu_s, wall_s, threading.current_thread() is threading.main_thread()\\n\\n    def __enter__(self):\\n        if self.on:\\n            def boom(signum, frame):                          # noqa: ARG001\\n                raise _Timeout()\\n            self.old = (signal.signal(signal.SIGVTALRM, boom), signal.signal(signal.SIGALRM, boom))\\n            self.old_t = (signal.setitimer(signal.ITIMER_VIRTUAL, self.cpu_s),\\n                          signal.setitimer(signal.ITIMER_REAL, self.wall_s))\\n        return self\\n\\n    def __exit__(self, *exc):\\n        if self.on:\\n            signal.setitimer(signal.ITIMER_VIRTUAL, 0)\\n            signal.setitimer(signal.ITIMER_REAL, 0)\\n            signal.signal(signal.SIGVTALRM, self.old[0])\\n            signal.signal(signal.SIGALRM, self.old[1])\\n            for which, (val, interval) in ((signal.ITIMER_VIRTUAL, self.old_t[0]), (signal.ITIMER_REAL, self.old_t[1])):\\n                if val > 0:\\n                    signal.setitimer(which, val, interval)\\n        return False\\n\\n\\ndef cast_candidates(task, cpu_s=15.0, wall_s=30.0, details=False):\\n    n_test = len(task.get(\\"test\\", task.get(\\"test_inputs\\", [])) or [])\\n    info = {\\"grids\\": [None] * n_test, \\"module\\": [None] * n_test, \\"program\\": [None] * n_test, \\"status\\": \\"NONE\\",\\n            \\"version\\": VERSION, \\"cpu_s\\": 0.0}\\n    t0 = time.process_time()\\n    try:\\n        with _Caps(cpu_s, wall_s):\\n            t = _norm(task)\\n            changes = all(np.asarray(x).shape != np.asarray(y).shape or not np.array_equal(x, y) for x, y in t[\\"train\\"])\\n            for module in ORDER:\\n                if all(g is not None for g in info[\\"grids\\"]):\\n                    break\\n                groups, atts = _ranked(module, t)\\n                if not any(atts) or not _loo_pass(module, t):\\n                    continue\\n                for j, q in enumerate(t[\\"test_inputs\\"]):\\n                    if info[\\"grids\\"][j] is not None or not atts[j]:\\n                        continue\\n                    g = np.asarray(atts[j][0])\\n                    if not G.valid_grid(g) or (changes and g.shape == np.asarray(q).shape and np.array_equal(g, q)):\\n                        continue\\n                    info[\\"grids\\"][j] = g.astype(int).tolist()\\n                    info[\\"module\\"][j] = module\\n                    info[\\"program\\"][j] = groups[0][\\"members\\"][0][\\"program\\"] if groups else None\\n            info[\\"status\\"] = \\"OK\\"\\n    except _Timeout:\\n        info.update(grids=[None] * n_test, module=[None] * n_test, program=[None] * n_test, status=\\"TIMEOUT\\")\\n    except Exception as e:                                    # noqa: BLE001  fail-safe by contract\\n        info.update(grids=[None] * n_test, module=[None] * n_test, program=[None] * n_test,\\n                    status=f\\"ERROR:{type(e).__name__}\\")\\n    info[\\"cpu_s\\"] = round(time.process_time() - t0, 3)\\n    return info if details else info[\\"grids\\"]\\n", "cast_v1/candidates.py": "\\"\\"\\"Fitting programs -> ranked behaviours -> two attempts per query, and export in the score-first branch\'s format.\\n\\nSelection rule (pre-declared for every CAST module; frozen 27 Oct before any evaluation run):\\n  1. Only programs that reproduce EVERY demonstration are candidates. Group them by their predictions on the queries\\n     (two programs that fit but disagree on a query stay separate behaviours).\\n  2. Non-vacuity: a query prediction equal to that query\'s input is dropped when every demonstration changes its\\n     input (the synthesis review\'s identity-attempt defect).\\n  3. Rank behaviours by: complete (no undetermined cell on any query) first; then leave-one-demonstration-out\\n     support (higher first); then group size (more agreeing programs first); then program cost (lower first); then\\n     enumeration order. Deterministic; no randomness; no answer access.\\n  4. Attempts per query: the first two distinct valid grids in that order.\\nExport (`to_program_candidates`) follows kaggle/hybrid_2026/llm_writer/program_candidates.py:\\n  {\\"per_test\\": [[{\\"grid\\", \\"tier\\", \\"score\\", \\"certified\\", \\"loo\\", \\"group_size\\", \\"program\\", \\"candidate\\"}, ...], ...],\\n   \\"n_programs\\", \\"n_fits\\", \\"n_groups\\", \\"failures\\"}; tier is \\"CAST_FIT\\" or \\"CAST_PARTIAL\\" (incomplete completion).\\nWhether the score-first entry imports these is the author\'s decision through the SUBMISSION_READY gate.\\n\\"\\"\\"\\nimport numpy as np\\n\\nfrom . import grids as G\\n\\n\\ndef _key(p):\\n    return None if p is None else (tuple(np.asarray(p).shape), np.asarray(p).tobytes())\\n\\n\\ndef rank(fits, task):\\n    \\"\\"\\"fits: [{\\"program\\", \\"preds\\", \\"complete\\": [bool per query], \\"loo\\": float, \\"cost\\": float, \\"order\\": int}]\\"\\"\\"\\n    every_changes = all(np.asarray(x).shape != np.asarray(y).shape or not np.array_equal(x, y)\\n                        for x, y in task[\\"train\\"])\\n    groups = {}\\n    for f in fits:\\n        preds = []\\n        for q, p in zip(task[\\"test_inputs\\"], f[\\"preds\\"]):\\n            if p is None or not G.valid_grid(p):\\n                preds.append(None)\\n                continue\\n            if every_changes and np.asarray(p).shape == np.asarray(q).shape and np.array_equal(p, q):\\n                preds.append(None)                    # vacuous\\n                continue\\n            preds.append(p)\\n        k = tuple(_key(p) for p in preds)\\n        if all(x is None for x in k):\\n            continue\\n        g = groups.setdefault(k, {\\"preds\\": preds, \\"members\\": [], \\"complete\\": True, \\"loo\\": 0.0, \\"cost\\": 1e9,\\n                                  \\"order\\": 1e9})\\n        g[\\"members\\"].append(f)\\n        g[\\"complete\\"] = g[\\"complete\\"] and all(f.get(\\"complete\\", [True] * len(preds)))\\n        g[\\"loo\\"] = max(g[\\"loo\\"], f.get(\\"loo\\", 0.0))\\n        g[\\"cost\\"] = min(g[\\"cost\\"], f.get(\\"cost\\", 1.0))\\n        g[\\"order\\"] = min(g[\\"order\\"], f.get(\\"order\\", 0))\\n    out = sorted(groups.values(), key=lambda g: (not g[\\"complete\\"], -g[\\"loo\\"], -len(g[\\"members\\"]), g[\\"cost\\"],\\n                                                  g[\\"order\\"]))\\n    return out\\n\\n\\ndef attempts(groups, n_queries):\\n    per = []\\n    for j in range(n_queries):\\n        atts, seen = [], set()\\n        for g in groups:\\n            p = g[\\"preds\\"][j] if j < len(g[\\"preds\\"]) else None\\n            k = _key(p)\\n            if p is None or k in seen:\\n                continue\\n            seen.add(k)\\n            atts.append(np.asarray(p).tolist())\\n            if len(atts) == 2:\\n                break\\n        per.append(atts)\\n    return per\\n\\n\\ndef to_program_candidates(groups, n_queries, n_programs, failures=None, limit=2):\\n    per = [[] for _ in range(n_queries)]\\n    for gi, g in enumerate(groups):\\n        best = g[\\"members\\"][0]\\n        for j in range(n_queries):\\n            p = g[\\"preds\\"][j] if j < len(g[\\"preds\\"]) else None\\n            if p is None or len(per[j]) >= limit or any(np.asarray(p).tolist() == x[\\"grid\\"] for x in per[j]):\\n                continue\\n            per[j].append({\\"grid\\": np.asarray(p).tolist(), \\"tier\\": \\"CAST_FIT\\" if g[\\"complete\\"] else \\"CAST_PARTIAL\\",\\n                           \\"score\\": round(float(len(per[j]) == 0) + g[\\"loo\\"], 3), \\"certified\\": False,\\n                           \\"loo\\": round(float(g[\\"loo\\"]), 3), \\"group_size\\": len(g[\\"members\\"]),\\n                           \\"program\\": best[\\"program\\"], \\"candidate\\": f\\"cast-{gi}\\"})\\n    return {\\"per_test\\": per, \\"n_programs\\": n_programs, \\"n_fits\\": sum(len(g[\\"members\\"]) for g in groups),\\n            \\"n_groups\\": len(groups), \\"failures\\": failures or {}}\\n", "cast_v1/grids.py": "\\"\\"\\"Grid utilities and pixel-grounded views for CAST (numpy, deterministic, no background-0 assumption).\\n\\nConventions (declared once, used everywhere):\\n- A grid is a 2-D numpy int array with values 0..9. Coordinates are (row, col), row 0 at the top.\\n- Background of a grid = its majority colour; ties go to the smallest colour value. Never assumed to be 0.\\n- Connectivity: 4 = edge neighbours; 8 = edge or corner neighbours. A \\"mono\\" object is a connected set of cells of one\\n  colour; a \\"multi\\" object is a connected set of non-background cells of any colours.\\n- An enclosed region (hole) of a mask M is a 4-connected component of the complement of M that does not touch the\\n  grid border. With 8-connected objects, holes use 4-connectivity of the complement (the standard digital-topology\\n  pairing (8, 4)); with 4-connected objects the pairing is (4, 8). Both are offered; callers state which.\\nEvery entity keeps its source mask, its box and its cells, so rendering back to a grid is exact.\\n\\"\\"\\"\\nfrom dataclasses import dataclass, field\\n\\nimport numpy as np\\nfrom . import _nd as ndimage\\n\\nS4 = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=bool)\\nS8 = np.ones((3, 3), dtype=bool)\\n\\n\\ndef A(g):\\n    return np.asarray(g, dtype=np.int64)\\n\\n\\ndef bg_of(g):\\n    \\"\\"\\"Majority colour; ties -> smallest colour value.\\"\\"\\"\\n    cnt = np.bincount(A(g).ravel(), minlength=10)\\n    return int(np.argmax(cnt))\\n\\n\\ndef palette(g):\\n    return frozenset(int(v) for v in np.unique(A(g)))\\n\\n\\ndef counts(g):\\n    return np.bincount(A(g).ravel(), minlength=10)[:10]\\n\\n\\ndef label(mask, conn=4):\\n    lab, n = ndimage.label(mask, structure=S4 if conn == 4 else S8)\\n    return lab, n\\n\\n\\n@dataclass\\nclass Obj:\\n    \\"\\"\\"A connected entity: its mask on the source grid, box (r0, c0, r1, c1) inclusive, colours and cell values.\\"\\"\\"\\n    mask: np.ndarray\\n    box: tuple\\n    colours: frozenset\\n    size: int\\n    idx: int = 0\\n    tags: dict = field(default_factory=dict)\\n\\n    @property\\n    def h(self):\\n        return self.box[2] - self.box[0] + 1\\n\\n    @property\\n    def w(self):\\n        return self.box[3] - self.box[1] + 1\\n\\n    def patch(self, g, fill=-1):\\n        \\"\\"\\"The object\'s box cut from g, cells outside the mask set to `fill`.\\"\\"\\"\\n        r0, c0, r1, c1 = self.box\\n        p = A(g)[r0:r1 + 1, c0:c1 + 1].copy()\\n        p[~self.mask[r0:r1 + 1, c0:c1 + 1]] = fill\\n        return p\\n\\n    def shape_key(self):\\n        r0, c0, r1, c1 = self.box\\n        return self.mask[r0:r1 + 1, c0:c1 + 1].tobytes() + bytes([self.h, self.w])\\n\\n    @property\\n    def colour(self):\\n        return min(self.colours) if len(self.colours) == 1 else -1\\n\\n\\ndef _box(mask):\\n    rows = np.where(mask.any(1))[0]\\n    cols = np.where(mask.any(0))[0]\\n    return (int(rows[0]), int(cols[0]), int(rows[-1]), int(cols[-1]))\\n\\n\\ndef objects(g, bg=None, conn=4, mono=True):\\n    \\"\\"\\"Objects of g under a declared view; bg defaults to the grid\'s majority colour.\\"\\"\\"\\n    g = A(g)\\n    if bg is None:\\n        bg = bg_of(g)\\n    out = []\\n    if mono:\\n        for c in sorted(set(np.unique(g).tolist()) - {bg}):\\n            lab, n = label(g == c, conn)\\n            for i in range(1, n + 1):\\n                m = lab == i\\n                out.append(Obj(m, _box(m), frozenset([c]), int(m.sum())))\\n    else:\\n        lab, n = label(g != bg, conn)\\n        for i in range(1, n + 1):\\n            m = lab == i\\n            out.append(Obj(m, _box(m), frozenset(int(v) for v in np.unique(g[m])), int(m.sum())))\\n    out.sort(key=lambda o: (o.box[0], o.box[1], o.colour, o.size))\\n    for i, o in enumerate(out):\\n        o.idx = i\\n    return out\\n\\n\\ndef holes_of_mask(mask, conn_bg=4):\\n    \\"\\"\\"Enclosed regions of the complement of mask (components not touching the border).\\"\\"\\"\\n    comp = ~mask\\n    lab, n = label(comp, conn_bg)\\n    border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])).tolist())\\n    return [lab == i for i in range(1, n + 1) if i not in border]\\n\\n\\ndef hole_count(obj, conn_bg=4):\\n    r0, c0, r1, c1 = obj.box\\n    m = np.pad(obj.mask[r0:r1 + 1, c0:c1 + 1], 1)\\n    return len(holes_of_mask(m, conn_bg))\\n\\n\\ndef enclosed_background(g, bg=None, conn_bg=4):\\n    \\"\\"\\"Background cells not connected (conn_bg) to the border through background cells.\\"\\"\\"\\n    g = A(g)\\n    if bg is None:\\n        bg = bg_of(g)\\n    lab, n = label(g == bg, conn_bg)\\n    border = set(np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]])).tolist())\\n    keep = np.zeros(g.shape, dtype=bool)\\n    for i in range(1, n + 1):\\n        if i not in border:\\n            keep |= lab == i\\n    return keep\\n\\n\\ndef separators(g):\\n    \\"\\"\\"Full rows/columns of one non-background-or-background colour that cut the grid into panels.\\n    Returns (colour, row indices, col indices) for the colour with the most full lines, or None.\\"\\"\\"\\n    g = A(g)\\n    h, w = g.shape\\n    best = None\\n    for c in np.unique(g).tolist():\\n        rows = [r for r in range(h) if (g[r] == c).all()]\\n        cols = [k for k in range(w) if (g[:, k] == c).all()]\\n        if (rows or cols) and len(rows) < h and len(cols) < w:\\n            score = len(rows) + len(cols)\\n            if best is None or score > best[0]:\\n                best = (score, c, rows, cols)\\n    if best is None:\\n        return None\\n    return best[1], best[2], best[3]\\n\\n\\ndef panels(g):\\n    \\"\\"\\"Panels cut by the separator lines (row-major order), as (r0, c0, r1, c1) inclusive boxes; [] if none.\\"\\"\\"\\n    g = A(g)\\n    s = separators(g)\\n    if s is None:\\n        return []\\n    _, rows, cols = s\\n    h, w = g.shape\\n\\n    def spans(lines, n):\\n        out, start = [], 0\\n        for x in sorted(lines) + [n]:\\n            if x > start:\\n                out.append((start, x - 1))\\n            start = x + 1\\n        return out\\n    rs, cs = spans(rows, h), spans(cols, w)\\n    if len(rs) * len(cs) < 2:\\n        return []\\n    return [(a, b, c, d) for a, c in rs for b, d in cs]\\n\\n\\ndef crop(g, box):\\n    r0, c0, r1, c1 = box\\n    return A(g)[r0:r1 + 1, c0:c1 + 1].copy()\\n\\n\\ndef bbox_of(mask):\\n    if not mask.any():\\n        return None\\n    return _box(mask)\\n\\n\\ndef grid_key(g):\\n    g = A(g)\\n    return g.shape, g.tobytes()\\n\\n\\ndef valid_grid(g):\\n    if g is None:\\n        return False\\n    g = np.asarray(g)\\n    return g.ndim == 2 and 1 <= g.shape[0] <= 30 and 1 <= g.shape[1] <= 30 and g.min() >= 0 and g.max() <= 9\\n", "cast_v1/ops.py": "\\"\\"\\"CAST operator library L0 with declared contracts (M3) and the task context that sets parameter domains.\\n\\nEvery operator class declares:\\n    domain(ctx)        finite parameter domain for a task (built from the demonstrations\' colours and shapes only:\\n                       an OBSERVED_DEMONSTRATION_CONSTRAINT that defines the grammar instance; never from queries\'\\n                       outputs, which the solver cannot read)\\n    apply(g, th)       concrete semantics; returns a grid or None (undefined); deterministic, bounded, no I/O\\n    fwd(a, th)         cheap forward contract (PROVED_OPERATOR_CONTRACT): gamma(fwd(a, th)) contains apply(z, th) for\\n                       every z in gamma(a) where apply is defined. It must not run the operator\'s expensive part.\\n    bwd(y, th)         predecessor relation: gamma(bwd(y, th)) contains every w with apply(w, th) == y\\n    cost               nominal execution cost unit (CPU is measured separately)\\nContracts are validated exhaustively on tiny grids and by randomized property tests (tests/test_contracts_*.py);\\nthe declared scope is the finite parameter domain and grids up to 30 x 30.\\nOverlap semantics: every L0 operator is a whole-grid function (one synchronous write of the full output).\\n\\"\\"\\"\\nimport numpy as np\\n\\nfrom . import absdom as D\\nfrom . import grids as G\\n\\nGEO = (\\"fliplr\\", \\"flipud\\", \\"rot90\\", \\"rot180\\", \\"rot270\\", \\"transpose\\", \\"antitranspose\\")\\n\\n\\ndef geo_apply(g, t):\\n    g = np.asarray(g)\\n    return {\\"fliplr\\": lambda x: x[:, ::-1], \\"flipud\\": lambda x: x[::-1, :], \\"rot90\\": lambda x: np.rot90(x, 1),\\n            \\"rot180\\": lambda x: np.rot90(x, 2), \\"rot270\\": lambda x: np.rot90(x, 3), \\"transpose\\": lambda x: x.T,\\n            \\"antitranspose\\": lambda x: np.rot90(x, 2).T}[t](g).copy()\\n\\n\\nGEO_INV = {\\"fliplr\\": \\"fliplr\\", \\"flipud\\": \\"flipud\\", \\"rot90\\": \\"rot270\\", \\"rot180\\": \\"rot180\\", \\"rot270\\": \\"rot90\\",\\n           \\"transpose\\": \\"transpose\\", \\"antitranspose\\": \\"antitranspose\\"}\\n\\n\\ndef geo_shape(s, t):\\n    return (s[1], s[0]) if t in (\\"rot90\\", \\"rot270\\", \\"transpose\\", \\"antitranspose\\") else s\\n\\n\\nclass TaskCtx:\\n    \\"\\"\\"What a grammar instance may depend on: demonstrations (inputs and outputs) and query INPUTS only.\\"\\"\\"\\n\\n    def __init__(self, train, test_inputs=()):\\n        self.train = [(np.asarray(x, dtype=np.int64), np.asarray(y, dtype=np.int64)) for x, y in train]\\n        self.test_inputs = [np.asarray(x, dtype=np.int64) for x in test_inputs]\\n        self.in_pal = sorted(set().union(*[G.palette(x) for x, _ in self.train]))\\n        self.out_pal = sorted(set().union(*[G.palette(y) for _, y in self.train]))\\n        self.all_pal = sorted(set(self.in_pal) | set(self.out_pal))\\n        self.same_shape = all(x.shape == y.shape for x, y in self.train)\\n\\n\\nclass Op:\\n    name = \\"op\\"\\n    cost = 1\\n    evidence = \\"PROVED_OPERATOR_CONTRACT\\"\\n    scope = \\"grids up to 30x30, the listed parameter domain\\"\\n    same_shape = True\\n\\n    def domain(self, ctx):\\n        return [()]\\n\\n    def apply(self, g, th):\\n        raise NotImplementedError\\n\\n    def fwd(self, a, th):\\n        return D.TOP()\\n\\n    def bwd(self, y, th):\\n        return D.TOP()\\n\\n    def label(self, th):\\n        return f\\"{self.name}{th}\\"\\n\\n\\ndef _same_shape_cells_or_top(a, newbits_fn):\\n    \\"\\"\\"Helper: per-cell transfer when cells are known; otherwise only shapes and palette are kept.\\"\\"\\"\\n    if a.bottom:\\n        return D.BOTTOM()\\n    if a.cells is None:\\n        return D.Abs(a.shapes, a.shape_lo, None, D.ALL)\\n    return D.Abs(a.shapes, a.shape_lo, newbits_fn(a.cells), D.ALL)\\n\\n\\nclass Identity(Op):\\n    name = \\"id\\"\\n\\n    def apply(self, g, th):\\n        return np.asarray(g).copy()\\n\\n    def fwd(self, a, th):\\n        return a.copy()\\n\\n    def bwd(self, y, th):\\n        return D.alpha(y)\\n\\n\\nclass Geo(Op):\\n    name = \\"geo\\"\\n    same_shape = False\\n\\n    def domain(self, ctx):\\n        return [(t,) for t in GEO]\\n\\n    def apply(self, g, th):\\n        return geo_apply(g, th[0])\\n\\n    def fwd(self, a, th):\\n        t = th[0]\\n        if a.bottom:\\n            return D.BOTTOM()\\n        shapes = None if a.shapes is None else {geo_shape(s, t) for s in a.shapes}\\n        lo = geo_shape(a.shape_lo, t)\\n        cells = None if a.cells is None else geo_apply(a.cells, t)\\n        return D.Abs(shapes, lo, cells, a.pal, a.need, a.lo, a.hi)\\n\\n    def bwd(self, y, th):\\n        return D.alpha(geo_apply(y, GEO_INV[th[0]]))\\n\\n\\nclass Recolor(Op):\\n    \\"\\"\\"Every cell of colour a becomes b (a != b).\\"\\"\\"\\n    name = \\"recolor\\"\\n\\n    def domain(self, ctx):\\n        return [(a, b) for a in ctx.in_pal for b in ctx.out_pal if a != b]\\n\\n    def apply(self, g, th):\\n        a, b = th\\n        g = np.asarray(g).copy()\\n        g[g == a] = b\\n        return g\\n\\n    def fwd(self, x, th):\\n        a, b = th\\n        if x.bottom:\\n            return D.BOTTOM()\\n\\n        def mp(m):\\n            return (m & ~D.bit(a)) | (D.bit(b) if m >> a & 1 else 0)\\n        lo, hi = x.lo.copy(), x.hi.copy()\\n        lo[b], hi[b] = lo[b] + lo[a], hi[b] + hi[a]\\n        lo[a] = hi[a] = 0\\n        cells = None\\n        if x.cells is not None:\\n            has = (x.cells >> a) & 1\\n            cells = ((x.cells & ~np.uint16(D.bit(a))) | (has * np.uint16(D.bit(b)))).astype(np.uint16)\\n        return D.Abs(x.shapes, x.shape_lo, cells, mp(x.pal), mp(x.need), lo, hi)\\n\\n    def bwd(self, y, th):\\n        a, b = th\\n        y = np.asarray(y)\\n        if (y == a).any():\\n            return D.BOTTOM()\\n        cells = D.BITS[y].astype(np.uint16)\\n        cells[y == b] = D.bit(a) | D.bit(b)\\n        return D.Abs({y.shape}, (1, 1), cells, D.ALL)\\n\\n\\nSELECTORS = (\\"nonbg\\", \\"largest4\\", \\"smallest4\\", \\"largest8\\", \\"smallest8\\", \\"largest_multi8\\", \\"colour\\")\\n\\n\\ndef select_box(g, sel):\\n    g = np.asarray(g)\\n    bg = G.bg_of(g)\\n    if sel[0] == \\"nonbg\\":\\n        return G.bbox_of(g != bg)\\n    if sel[0] == \\"colour\\":\\n        return G.bbox_of(g == sel[1])\\n    conn = 8 if sel[0].endswith(\\"8\\") else 4\\n    objs = G.objects(g, bg, conn=conn, mono=not sel[0].startswith(\\"largest_multi\\"))\\n    if not objs:\\n        return None\\n    sizes = [o.size for o in objs]\\n    target = max(sizes) if sel[0].startswith(\\"largest\\") else min(sizes)\\n    cand = [o for o in objs if o.size == target]\\n    if len(cand) != 1:\\n        return None                                   # ambiguous extremum: undefined (no tie-break by position)\\n    return cand[0].box\\n\\n\\nclass CropBox(Op):\\n    \\"\\"\\"Crop to the box of a selected region: the non-background cells, one colour, or the unique largest/smallest\\n    object under a declared view.\\"\\"\\"\\n    name = \\"crop\\"\\n    same_shape = False\\n\\n    def domain(self, ctx):\\n        return [(s,) for s in SELECTORS if s != \\"colour\\"] + [(\\"colour\\", c) for c in ctx.in_pal]\\n\\n    def apply(self, g, th):\\n        box = select_box(g, th)\\n        if box is None:\\n            return None\\n        return G.crop(g, box)\\n\\n    def fwd(self, a, th):\\n        if a.bottom:\\n            return D.BOTTOM()\\n        if a.shapes is None:\\n            return D.Abs(None, (1, 1), None, a.pal, 0, None, a.hi)\\n        if D.is_concrete(a) and th[0] == \\"colour\\":            # cheap and exact: the box of one colour\\n            g = D.concretize_one(a)\\n            out = self.apply(g, th)\\n            return D.BOTTOM() if out is None else D.alpha(out)\\n        shapes = {(h, w) for H, W in a.shapes for h in range(1, H + 1) for w in range(1, W + 1)}\\n        return D.Abs(shapes, (1, 1), None, a.pal, 0, None, a.hi)\\n\\n    def bwd(self, y, th):\\n        y = np.asarray(y)\\n        cnt = G.counts(y)\\n        need = D.mask_of(np.unique(y).tolist())\\n        return D.Abs(None, y.shape, None, D.ALL, need, cnt, None)\\n\\n\\nclass Scale(Op):\\n    name = \\"scale\\"\\n    same_shape = False\\n\\n    def domain(self, ctx):\\n        return [(k,) for k in (2, 3, 4, 5)]\\n\\n    def apply(self, g, th):\\n        k = th[0]\\n        g = np.asarray(g)\\n        if g.shape[0] * k > 30 or g.shape[1] * k > 30:\\n            return None\\n        return np.kron(g, np.ones((k, k), dtype=g.dtype))\\n\\n    def fwd(self, a, th):\\n        k = th[0]\\n        if a.bottom:\\n            return D.BOTTOM()\\n        shapes = None if a.shapes is None else {(h * k, w * k) for h, w in a.shapes}\\n        cells = None if a.cells is None else np.kron(a.cells, np.ones((k, k), dtype=np.uint16)).astype(np.uint16)\\n        return D.Abs(shapes, (a.shape_lo[0] * k, a.shape_lo[1] * k), cells, a.pal, a.need, a.lo * k * k,\\n                     np.minimum(a.hi * k * k, D.MAXC))\\n\\n    def bwd(self, y, th):\\n        k = th[0]\\n        y = np.asarray(y)\\n        h, w = y.shape\\n        if h % k or w % k:\\n            return D.BOTTOM()\\n        small = y[::k, ::k]\\n        if not np.array_equal(np.kron(small, np.ones((k, k), dtype=y.dtype)), y):\\n            return D.BOTTOM()\\n        return D.alpha(small)\\n\\n\\nclass Downscale(Op):\\n    \\"\\"\\"Exact block downscale: defined only when the grid is constant on every k x k block.\\"\\"\\"\\n    name = \\"down\\"\\n    same_shape = False\\n\\n    def domain(self, ctx):\\n        return [(k,) for k in (2, 3, 4, 5)]\\n\\n    def apply(self, g, th):\\n        k = th[0]\\n        g = np.asarray(g)\\n        h, w = g.shape\\n        if h % k or w % k:\\n            return None\\n        small = g[::k, ::k]\\n        if not np.array_equal(np.kron(small, np.ones((k, k), dtype=g.dtype)), g):\\n            return None\\n        return small.copy()\\n\\n    def fwd(self, a, th):\\n        k = th[0]\\n        if a.bottom:\\n            return D.BOTTOM()\\n        if a.shapes is None:\\n            return D.Abs(None, (1, 1), None, a.pal, a.need)\\n        shapes = {(h // k, w // k) for h, w in a.shapes if h % k == 0 and w % k == 0 and h >= k and w >= k}\\n        if not shapes:\\n            return D.BOTTOM()\\n        cells = None\\n        if a.cells is not None and a.cells.shape[0] % k == 0 and a.cells.shape[1] % k == 0:\\n            h, w = a.cells.shape\\n            cells = np.bitwise_and.reduce(a.cells.reshape(h // k, k, w // k, k).transpose(0, 2, 1, 3)\\n                                          .reshape(h // k, w // k, k * k), axis=2).astype(np.uint16)\\n        return D.Abs(shapes, (1, 1), cells, a.pal, a.need, (a.lo + k * k - 1) // (k * k), a.hi // (k * k))\\n\\n    def bwd(self, y, th):\\n        k = th[0]\\n        y = np.asarray(y)\\n        if y.shape[0] * k > 30 or y.shape[1] * k > 30:\\n            return D.BOTTOM()\\n        return D.alpha(np.kron(y, np.ones((k, k), dtype=y.dtype)))\\n\\n\\nclass Tile(Op):\\n    name = \\"tile\\"\\n    same_shape = False\\n\\n    def domain(self, ctx):\\n        return [(n, m) for n in range(1, 5) for m in range(1, 5) if (n, m) != (1, 1)]\\n\\n    def apply(self, g, th):\\n        n, m = th\\n        g = np.asarray(g)\\n        if g.shape[0] * n > 30 or g.shape[1] * m > 30:\\n            return None\\n        return np.tile(g, (n, m))\\n\\n    def fwd(self, a, th):\\n        n, m = th\\n        if a.bottom:\\n            return D.BOTTOM()\\n        shapes = None if a.shapes is None else {(h * n, w * m) for h, w in a.shapes}\\n        cells = None if a.cells is None else np.tile(a.cells, (n, m)).astype(np.uint16)\\n        return D.Abs(shapes, (a.shape_lo[0] * n, a.shape_lo[1] * m), cells, a.pal, a.need, a.lo * n * m,\\n                     np.minimum(a.hi * n * m, D.MAXC))\\n\\n    def bwd(self, y, th):\\n        n, m = th\\n        y = np.asarray(y)\\n        h, w = y.shape\\n        if h % n or w % m:\\n            return D.BOTTOM()\\n        u = y[:h // n, :w // m]\\n        if not np.array_equal(np.tile(u, (n, m)), y):\\n            return D.BOTTOM()\\n        return D.alpha(u)\\n\\n\\nclass Concat(Op):\\n    \\"\\"\\"g next to t(g): side \'h\' (left|right) or \'v\' (top/bottom), t a same-shape symmetry.\\"\\"\\"\\n    name = \\"concat\\"\\n    same_shape = False\\n    TS = (\\"id\\", \\"fliplr\\", \\"flipud\\", \\"rot180\\")\\n\\n    def domain(self, ctx):\\n        return [(s, t) for s in \\"hv\\" for t in self.TS]\\n\\n    @staticmethod\\n    def _t(g, t):\\n        return np.asarray(g).copy() if t == \\"id\\" else geo_apply(g, t)\\n\\n    def apply(self, g, th):\\n        s, t = th\\n        g = np.asarray(g)\\n        if (s == \\"h\\" and g.shape[1] * 2 > 30) or (s == \\"v\\" and g.shape[0] * 2 > 30):\\n            return None\\n        return np.concatenate([g, self._t(g, t)], axis=1 if s == \\"h\\" else 0)\\n\\n    def fwd(self, a, th):\\n        s, t = th\\n        if a.bottom:\\n            return D.BOTTOM()\\n        f = (lambda h, w: (h, 2 * w)) if s == \\"h\\" else (lambda h, w: (2 * h, w))\\n        shapes = None if a.shapes is None else {f(*sh) for sh in a.shapes}\\n        cells = None\\n        if a.cells is not None:\\n            cells = np.concatenate([a.cells, self._t(a.cells, t)], axis=1 if s == \\"h\\" else 0).astype(np.uint16)\\n        return D.Abs(shapes, f(*a.shape_lo), cells, a.pal, a.need, a.lo * 2, np.minimum(a.hi * 2, D.MAXC))\\n\\n    def bwd(self, y, th):\\n        s, t = th\\n        y = np.asarray(y)\\n        h, w = y.shape\\n        if (s == \\"h\\" and w % 2) or (s == \\"v\\" and h % 2):\\n            return D.BOTTOM()\\n        p, q = (y[:, :w // 2], y[:, w // 2:]) if s == \\"h\\" else (y[:h // 2], y[h // 2:])\\n        inv = q if t == \\"id\\" else geo_apply(q, GEO_INV[t])\\n        return D.meet(D.alpha(p), D.alpha(inv))\\n\\n\\nclass FillEnclosed(Op):\\n    \\"\\"\\"Background cells that cannot reach the border through background (4-connected) become colour c.\\"\\"\\"\\n    name = \\"fill_enclosed\\"\\n\\n    def domain(self, ctx):\\n        return [(c,) for c in ctx.out_pal]\\n\\n    def apply(self, g, th):\\n        g = np.asarray(g).copy()\\n        m = G.enclosed_background(g)\\n        if not m.any():\\n            return None\\n        g[m] = th[0]\\n        return g\\n\\n    def fwd(self, a, th):\\n        c = th[0]\\n        if a.bottom:\\n            return D.BOTTOM()\\n        if a.cells is None:\\n            return D.Abs(a.shapes, a.shape_lo, None, a.pal | D.bit(c))\\n        cells = a.cells.copy()\\n        inner = np.zeros(cells.shape, dtype=bool)\\n        inner[1:-1, 1:-1] = True                        # border cells can never be enclosed\\n        cells[inner] |= np.uint16(D.bit(c))\\n        return D.Abs(a.shapes, a.shape_lo, cells, a.pal | D.bit(c))\\n\\n    def bwd(self, y, th):\\n        y = np.asarray(y)\\n        cells = D.BITS[y].astype(np.uint16)\\n        cells[y == th[0]] = D.ALL\\n        return D.Abs({y.shape}, (1, 1), cells, D.ALL)\\n\\n\\nclass Outline(Op):\\n    \\"\\"\\"Background cells 4- or 8-adjacent to a non-background cell become colour c.\\"\\"\\"\\n    name = \\"outline\\"\\n\\n    def domain(self, ctx):\\n        return [(c, k) for c in ctx.out_pal for k in (4, 8)]\\n\\n    def apply(self, g, th):\\n        c, k = th\\n        g = np.asarray(g)\\n        bg = G.bg_of(g)\\n        from . import _nd as ndimage\\n        fg = g != bg\\n        if not fg.any():\\n            return None\\n        near = ndimage.binary_dilation(fg, structure=G.S4 if k == 4 else G.S8) & ~fg\\n        out = g.copy()\\n        out[near] = c\\n        return out\\n\\n    def fwd(self, a, th):\\n        c = th[0]\\n        if a.bottom:\\n            return D.BOTTOM()\\n        if a.cells is None:\\n            return D.Abs(a.shapes, a.shape_lo, None, a.pal | D.bit(c))\\n        return D.Abs(a.shapes, a.shape_lo, (a.cells | np.uint16(D.bit(c))).astype(np.uint16), a.pal | D.bit(c))\\n\\n    def bwd(self, y, th):\\n        y = np.asarray(y)\\n        cells = D.BITS[y].astype(np.uint16)\\n        cells[y == th[0]] = D.ALL\\n        return D.Abs({y.shape}, (1, 1), cells, D.ALL)\\n\\n\\nclass Gravity(Op):\\n    \\"\\"\\"Cell gravity: in every column (up/down) or row (left/right) the non-background cells slide to that side,\\n    keeping their order (one synchronous write of the full grid).\\"\\"\\"\\n    name = \\"gravity\\"\\n\\n    def domain(self, ctx):\\n        return [(d,) for d in (\\"down\\", \\"up\\", \\"left\\", \\"right\\")]\\n\\n    def apply(self, g, th):\\n        d = th[0]\\n        g = np.asarray(g)\\n        bg = G.bg_of(g)\\n        x = g if d in (\\"down\\", \\"up\\") else g.T\\n        out = np.full_like(x, bg)\\n        for j in range(x.shape[1]):\\n            col = x[:, j]\\n            fg = col[col != bg]\\n            if d in (\\"down\\", \\"right\\"):\\n                out[x.shape[0] - len(fg):, j] = fg\\n            else:\\n                out[:len(fg), j] = fg\\n        return out if d in (\\"down\\", \\"up\\") else out.T\\n\\n    def fwd(self, a, th):\\n        d = th[0]\\n        if a.bottom:\\n            return D.BOTTOM()\\n        if a.cells is None:\\n            return D.Abs(a.shapes, a.shape_lo, None, a.pal, a.need, a.lo, a.hi)\\n        x = a.cells if d in (\\"down\\", \\"up\\") else a.cells.T\\n        u = np.bitwise_or.reduce(x, axis=0)\\n        out = np.broadcast_to(u, x.shape).copy()\\n        out = out if d in (\\"down\\", \\"up\\") else out.T\\n        return D.Abs(a.shapes, a.shape_lo, out.astype(np.uint16), a.pal, a.need, a.lo, a.hi)\\n\\n    def bwd(self, y, th):\\n        d = th[0]\\n        y = np.asarray(y)\\n        x = y if d in (\\"down\\", \\"up\\") else y.T\\n        u = np.bitwise_or.reduce(D.BITS[x], axis=0)\\n        out = np.broadcast_to(u, x.shape).copy()\\n        out = out if d in (\\"down\\", \\"up\\") else out.T\\n        cnt = G.counts(y)\\n        return D.Abs({y.shape}, (1, 1), out.astype(np.uint16), D.ALL, D.mask_of(np.unique(y).tolist()), cnt, cnt)\\n\\n\\nL0 = [Geo(), Recolor(), CropBox(), Scale(), Downscale(), Tile(), Concat(), FillEnclosed(), Outline(), Gravity()]\\n\\n\\ndef by_name(name, lib=None):\\n    for op in (lib or L0):\\n        if op.name == name:\\n            return op\\n    raise KeyError(name)\\n", "cast_v1/rewrite.py": "\\"\\"\\"M2: typed guarded rewriting over a grounded scene, and guarded output frames.\\n\\nScene (per grid, per view): background = majority colour; entities = objects under a declared view\\n(mono-4, mono-8, multi-8), each with its source mask, box, colours and features; relation tables computed from the\\nmasks (8-adjacency, enclosure of another object\'s cells inside the object\'s holes). Rendering is exact: an entity is\\nits cells.\\n\\nA rule is a typed guarded rewrite (NOT a double-pushout construction: no categorical interface is checked; this is an\\nordinary guarded graph rewrite with explicit semantics):\\n    pattern     one entity of the view\\n    guard       an atom over the entity\'s own features or its relations (or TRUE)\\n    preserved   every entity the rule does not match, and every cell outside the action\'s write set\\n    changed     the matched entity\'s write set (below)\\n    action      KEEP | DELETE | RECOLOR(c) | RECOLOR_REF(adjacent | enclosed) | FILL_HOLES(c) | OUTLINE(c) |\\n                MOVE(direction, until_blocked | to_border)\\n    schedule    SYNC for every action except MOVE: all matches and write values are computed on the INPUT scene,\\n                writes are applied together; two writes of different values to one cell make the program UNDEFINED.\\n                SEQ for MOVE: entities move one at a time, nearest to the target wall first, each reading the grid\\n                left by the previous moves (an entity stops at the border or before the first non-background cell).\\nA rule list is first-match: an entity takes the action of the first rule whose guard holds; no rule = KEEP.\\nWrite sets (the contract): DELETE/RECOLOR: the entity\'s cells; FILL_HOLES: its enclosed background cells; OUTLINE:\\nbackground cells 8-adjacent to it; MOVE: cells in its corridor along the direction. Canvas unchanged.\\nPrograms are found by construction from evidence (every demonstration object\'s observed fate proposes actions; guards\\nthat separate the objects of different fates are enumerated) and are ALWAYS verified by exact execution of the whole\\nrule list on every demonstration. Frame rules (CropEntity) crop to the box of the unique entity satisfying a guard.\\n\\"\\"\\"\\nimport itertools\\n\\nimport numpy as np\\nfrom . import _nd as ndimage\\n\\nfrom . import absdom as D\\nfrom . import grids as G\\nfrom .ops import Op\\n\\nVIEWS = ((\\"mono\\", 4), (\\"mono\\", 8), (\\"multi\\", 8))\\nMAX_ACTS = 8           # actions considered per view: the 8 that explain the most changed objects (declared bound)\\nMAX_OBJS = 120         # scenes with more objects over all demonstrations are skipped (declared bound)\\nTABLE_FEATURES = (\\"colour\\", \\"size\\", \\"holes\\", \\"shape\\", \\"height\\", \\"width\\", \\"n_colours\\")\\nDIRS = {\\"up\\": (-1, 0), \\"down\\": (1, 0), \\"left\\": (0, -1), \\"right\\": (0, 1)}\\n\\n\\n# ------------------------------------------------------------------ scene\\nclass Scene:\\n    def __init__(self, g, view):\\n        self.g = G.A(g)\\n        self.bg = G.bg_of(self.g)\\n        kind, conn = view\\n        self.view = view\\n        self.objs = G.objects(self.g, self.bg, conn=conn, mono=(kind == \\"mono\\"))\\n        H, W = self.g.shape\\n        lab = np.full(self.g.shape, -1)\\n        for o in self.objs:\\n            lab[o.mask] = o.idx\\n        self.lab = lab\\n        sizes = [o.size for o in self.objs]\\n        shapes = [o.shape_key() for o in self.objs]\\n        cols = [o.colour for o in self.objs]\\n        for o in self.objs:\\n            f = o.tags\\n            f[\\"colour\\"] = o.colour\\n            f[\\"size\\"] = o.size\\n            f[\\"holes\\"] = G.hole_count(o, 4)\\n            f[\\"has_hole\\"] = f[\\"holes\\"] > 0\\n            f[\\"border\\"] = bool(o.box[0] == 0 or o.box[1] == 0 or o.box[2] == H - 1 or o.box[3] == W - 1)\\n            f[\\"largest\\"] = o.size == max(sizes)\\n            f[\\"smallest\\"] = o.size == min(sizes)\\n            f[\\"size_unique\\"] = sizes.count(o.size) == 1\\n            f[\\"shape_unique\\"] = shapes.count(o.shape_key()) == 1\\n            f[\\"colour_unique\\"] = cols.count(o.colour) == 1\\n            f[\\"single_cell\\"] = o.size == 1\\n            f[\\"n_colours\\"] = len(o.colours)\\n            dil = ndimage.binary_dilation(o.mask, structure=G.S8) & ~o.mask\\n            nb = set(np.unique(lab[dil]).tolist()) - {-1}\\n            f[\\"adj_ids\\"] = sorted(nb)\\n            f[\\"adj_colours\\"] = sorted({self.objs[i].colour for i in nb})\\n            holes = self.hole_mask(o)\\n            inner = set(np.unique(lab[holes]).tolist()) - {-1} if holes.any() else set()\\n            f[\\"enc_ids\\"] = sorted(inner)\\n            f[\\"enc_colours\\"] = sorted({self.objs[i].colour for i in inner})\\n\\n    def hole_mask(self, o):\\n        r0, c0, r1, c1 = o.box\\n        sub = o.mask[r0:r1 + 1, c0:c1 + 1]\\n        filled = ndimage.binary_fill_holes(sub)\\n        m = np.zeros(self.g.shape, dtype=bool)\\n        m[r0:r1 + 1, c0:c1 + 1] = filled & ~sub\\n        return m\\n\\n\\n# ------------------------------------------------------------------ guards\\n_TRUE_ONLY = False\\n\\n\\ndef guard_space(scenes):\\n    \\"\\"\\"Finite guard atoms from the demonstration scenes (values seen in the demonstrations only).\\"\\"\\"\\n    out = [(\\"true\\",)]\\n    if _TRUE_ONLY:\\n        return out\\n    cols = sorted({o.tags[\\"colour\\"] for s in scenes for o in s.objs})\\n    out += [(\\"colour\\", c) for c in cols]\\n    out += [(\\"not_colour\\", c) for c in cols]\\n    for b in (\\"has_hole\\", \\"border\\", \\"largest\\", \\"smallest\\", \\"size_unique\\", \\"shape_unique\\", \\"colour_unique\\",\\n              \\"single_cell\\"):\\n        out += [(b, True), (b, False)]\\n    sizes = sorted({o.tags[\\"size\\"] for s in scenes for o in s.objs})\\n    out += [(\\"size\\", k) for k in sizes[:12]]\\n    hs = sorted({o.tags[\\"holes\\"] for s in scenes for o in s.objs})\\n    out += [(\\"holes\\", k) for k in hs if k <= 4]\\n    adj = sorted({c for s in scenes for o in s.objs for c in o.tags[\\"adj_colours\\"]})\\n    out += [(\\"adj_colour\\", c) for c in adj] + [(\\"adj_any\\", True), (\\"adj_any\\", False)]\\n    enc = sorted({c for s in scenes for o in s.objs for c in o.tags[\\"enc_colours\\"]})\\n    out += [(\\"enc_colour\\", c) for c in enc]\\n    return out\\n\\n\\ndef holds(o, gd):\\n    k = gd[0]\\n    f = o.tags\\n    if k == \\"true\\":\\n        return True\\n    if k == \\"colour\\":\\n        return f[\\"colour\\"] == gd[1]\\n    if k == \\"not_colour\\":\\n        return f[\\"colour\\"] != gd[1]\\n    if k in (\\"has_hole\\", \\"border\\", \\"largest\\", \\"smallest\\", \\"size_unique\\", \\"shape_unique\\", \\"colour_unique\\",\\n             \\"single_cell\\"):\\n        return f[k] == gd[1]\\n    if k == \\"size\\":\\n        return f[\\"size\\"] == gd[1]\\n    if k == \\"holes\\":\\n        return f[\\"holes\\"] == gd[1]\\n    if k == \\"adj_colour\\":\\n        return gd[1] in f[\\"adj_colours\\"]\\n    if k == \\"adj_any\\":\\n        return bool(f[\\"adj_ids\\"]) == gd[1]\\n    if k == \\"enc_colour\\":\\n        return gd[1] in f[\\"enc_colours\\"]\\n    raise KeyError(k)\\n\\n\\n# ------------------------------------------------------------------ actions (SYNC write sets and values)\\ndef writes(scene, o, act):\\n    \\"\\"\\"(mask, value array or scalar) written by a SYNC action on entity o; None = action undefined on o.\\"\\"\\"\\n    k = act[0]\\n    if k == \\"keep\\":\\n        return np.zeros(scene.g.shape, dtype=bool), 0\\n    if k == \\"delete\\":\\n        return o.mask, scene.bg\\n    if k == \\"recolor\\":\\n        return o.mask, act[1]\\n    if k == \\"recolor_ref\\":\\n        cs = o.tags[\\"adj_colours\\"] if act[1] == \\"adjacent\\" else o.tags[\\"enc_colours\\"]\\n        cs = [c for c in cs if c != o.tags[\\"colour\\"]]\\n        if len(cs) != 1:\\n            return None\\n        return o.mask, cs[0]\\n    if k == \\"fill_holes\\":\\n        h = scene.hole_mask(o) & (scene.g == scene.bg)\\n        if not h.any():\\n            return None\\n        return h, act[1]\\n    if k == \\"outline\\":\\n        m = ndimage.binary_dilation(o.mask, structure=G.S8) & (scene.g == scene.bg)\\n        return m, act[1]\\n    raise KeyError(k)\\n\\n\\ndef run_sync(scene, rules):\\n    out = scene.g.copy()\\n    wrote = np.zeros(scene.g.shape, dtype=bool)\\n    for o in scene.objs:\\n        act = next((a for gd, a in rules if holds(o, gd)), (\\"keep\\",))\\n        if act[0] == \\"keep\\":\\n            continue\\n        w = writes(scene, o, act)\\n        if w is None:\\n            return None\\n        m, v = w\\n        clash = wrote & m & (out != v)\\n        if clash.any():\\n            return None\\n        out[m] = v\\n        wrote |= m\\n    return out\\n\\n\\ndef run_moves(scene, rules):\\n    \\"\\"\\"SEQ schedule for MOVE rules (other entities keep their place).\\"\\"\\"\\n    g = scene.g.copy()\\n    plan = []\\n    for o in scene.objs:\\n        act = next((a for gd, a in rules if holds(o, gd)), (\\"keep\\",))\\n        if act[0] == \\"move\\":\\n            plan.append((o, act))\\n        elif act[0] != \\"keep\\":\\n            return None\\n    if not plan:\\n        return None\\n\\n    def key(item):\\n        o, act = item\\n        dr, dc = DIRS[act[1]]\\n        r0, c0, r1, c1 = o.box\\n        return -(r1 if dr > 0 else (-r0 if dr < 0 else (c1 if dc > 0 else -c0)))\\n    plan.sort(key=key)\\n    H, W = g.shape\\n    for o, act in plan:\\n        dr, dc = DIRS[act[1]]\\n        vals = g[o.mask]\\n        ys, xs = np.nonzero(o.mask)\\n        g[o.mask] = scene.bg\\n        step = 0\\n        while True:\\n            ny, nx = ys + (step + 1) * dr, xs + (step + 1) * dc\\n            if ny.min() < 0 or nx.min() < 0 or ny.max() >= H or nx.max() >= W:\\n                break\\n            if act[2] == \\"until_blocked\\" and (g[ny, nx] != scene.bg).any():\\n                break\\n            step += 1\\n        g[ys + step * dr, xs + step * dc] = vals\\n    return g\\n\\n\\n_SCENES = {}\\n\\n\\ndef scene_of(g, view):\\n    g = G.A(g)\\n    key = (g.shape, g.tobytes(), view)\\n    s = _SCENES.get(key)\\n    if s is None:\\n        if len(_SCENES) > 512:\\n            _SCENES.clear()\\n        s = Scene(g, view)\\n        _SCENES[key] = s\\n    return s\\n\\n\\ndef execute(g, view, rules):\\n    s = scene_of(g, view)\\n    if any(a[0] == \\"move\\" for _, a in rules):\\n        return run_moves(s, rules)\\n    return run_sync(s, rules)\\n\\n\\n# ------------------------------------------------------------------ construction from evidence\\ndef fates(scene, y):\\n    \\"\\"\\"Actions that explain each object of a demonstration in isolation (SYNC actions), by object index.\\"\\"\\"\\n    out = {}\\n    if y.shape != scene.g.shape:\\n        return None\\n    for o in scene.objs:\\n        ex = set()\\n        if (y[o.mask] == scene.g[o.mask]).all():\\n            ex.add((\\"keep\\",))\\n        vals = np.unique(y[o.mask])\\n        if len(vals) == 1:\\n            v = int(vals[0])\\n            if v == scene.bg:\\n                ex.add((\\"delete\\",))\\n            elif not (len(o.colours) == 1 and v == o.colour):\\n                ex.add((\\"recolor\\", v))\\n                for rel in (\\"adjacent\\", \\"enclosed\\"):\\n                    w = writes(scene, o, (\\"recolor_ref\\", rel))\\n                    if w is not None and w[1] == v:\\n                        ex.add((\\"recolor_ref\\", rel))\\n        h = scene.hole_mask(o) & (scene.g == scene.bg)\\n        if h.any():\\n            hv = np.unique(y[h])\\n            if len(hv) == 1 and int(hv[0]) != scene.bg:\\n                ex.add((\\"fill_holes\\", int(hv[0])))\\n        m = ndimage.binary_dilation(o.mask, structure=G.S8) & (scene.g == scene.bg)\\n        if m.any():\\n            mv = np.unique(y[m])\\n            if len(mv) == 1 and int(mv[0]) != scene.bg:\\n                ex.add((\\"outline\\", int(mv[0])))\\n        out[o.idx] = ex\\n    return out\\n\\n\\ndef construct(demos, view, max_lists=64):\\n    \\"\\"\\"Candidate rule lists (first-match, <= 2 rules + KEEP default, or 1 rule + another default) from evidence,\\n    each verified by exact execution on every demonstration. Returns verified rule lists.\\"\\"\\"\\n    if any(x.shape != y.shape for x, y in demos):\\n        return []\\n    scenes = [scene_of(x, view) for x, _ in demos]\\n    fs = [fates(s, y) for s, (_, y) in zip(scenes, demos)]\\n    cover = {}\\n    for f in fs:\\n        for e in f.values():\\n            if (\\"keep\\",) in e:\\n                continue\\n            for a in e:\\n                cover[a] = cover.get(a, 0) + 1\\n    acts = sorted(cover, key=lambda a: (-cover[a], repr(a)))[:MAX_ACTS]\\n    objs = [(i, o) for i, s in enumerate(scenes) for o in s.objs]\\n    if len(objs) > MAX_OBJS:\\n        return []\\n    guards = guard_space(scenes)\\n    # which guards are consistent with an action: every matching object is explained by it\\n    match = {gd: [holds(o, gd) for _, o in objs] for gd in guards}\\n    expl = {a: [a in fs[i][o.idx] for i, o in objs] for a in acts + [(\\"keep\\",)]}\\n    cands = []\\n    for a in acts:\\n        for gd in guards:\\n            m = match[gd]\\n            if not any(m) or not all(e for e, mm in zip(expl[a], m) if mm):\\n                continue\\n            rest = [k for k, mm in enumerate(m) if not mm]\\n            if all(expl[(\\"keep\\",)][k] for k in rest):\\n                cands.append([(gd, a)])\\n                continue\\n            for a2 in acts:\\n                if a2 == a:\\n                    continue\\n                if all(expl[a2][k] for k in rest):\\n                    cands.append([(gd, a), ((\\"true\\",), a2)])\\n                    continue\\n                for g2 in guards:\\n                    m2 = match[g2]\\n                    if not all((not m2[k]) or expl[a2][k] for k in rest):\\n                        continue\\n                    if all(m2[k] or expl[(\\"keep\\",)][k] for k in rest) and any(m2[k] for k in rest):\\n                        cands.append([(gd, a), (g2, a2)])\\n                        break\\n            if len(cands) > 4 * max_lists:\\n                break\\n    verified, seen = [], set()\\n    for rules in cands:\\n        key = repr(rules)\\n        if key in seen:\\n            continue\\n        seen.add(key)\\n        ok = True\\n        for x, y in demos:\\n            out = execute(x, view, rules)\\n            if out is None or not np.array_equal(out, y):\\n                ok = False\\n                break\\n        if ok:\\n            verified.append(rules)\\n            if len(verified) >= max_lists:\\n                break\\n    return verified\\n\\n\\ndef tfeat(o, f):\\n    if f == \\"shape\\":\\n        return o.shape_key()\\n    if f == \\"height\\":\\n        return o.h\\n    if f == \\"width\\":\\n        return o.w\\n    return o.tags[f]\\n\\n\\ndef run_table(scene, f, table):\\n    out = scene.g.copy()\\n    for o in scene.objs:\\n        k = tfeat(o, f)\\n        if k in table:\\n            out[o.mask] = table[k]\\n    return out\\n\\n\\ndef construct_tables(demos, view):\\n    \\"\\"\\"N-guard table rewrite: one categorical feature f; every object whose f-value is in the table is recoloured\\n    to the table\'s colour (guards f == v_i, actions RECOLOR(c_i)); all other objects KEEP. The table is a\\n    task-level constant learned from the demonstrations (an OBSERVED_DEMONSTRATION_CONSTRAINT), unlike M1\'s tables\\n    which are re-read from every input. Verified by exact execution.\\"\\"\\"\\n    if any(x.shape != y.shape for x, y in demos):\\n        return []\\n    out = []\\n    for f in TABLE_FEATURES:\\n        table, ok = {}, True\\n        for x, y in demos:\\n            s = scene_of(x, view)\\n            for o in s.objs:\\n                vals = np.unique(y[o.mask])\\n                k = tfeat(o, f)\\n                if len(vals) == 1 and not (len(o.colours) == 1 and int(vals[0]) == o.colour):\\n                    v = int(vals[0])\\n                    if table.get(k, v) != v:\\n                        ok = False\\n                        break\\n                    table[k] = v\\n            if not ok:\\n                break\\n        if not ok or not table:\\n            continue\\n        good = True\\n        for x, y in demos:\\n            if not np.array_equal(run_table(scene_of(x, view), f, table), y):\\n                good = False\\n                break\\n        if good:\\n            out.append((f, table))\\n    return out\\n\\n\\ndef construct_moves(demos, view, max_lists=16):\\n    \\"\\"\\"MOVE rule lists (SEQ): [(guard, move)] + KEEP, or [(guard, move d1), (true, move d2)]; verified by execution.\\"\\"\\"\\n    if any(x.shape != y.shape for x, y in demos):\\n        return []\\n    # declared precondition of the move constructor: every demonstration changes and keeps its colour counts\\n    if any(np.array_equal(x, y) or not np.array_equal(G.counts(x), G.counts(y)) for x, y in demos):\\n        return []\\n    scenes = [scene_of(x, view) for x, _ in demos]\\n    guards = guard_space(scenes)\\n    moves = [(\\"move\\", d, m) for d in DIRS for m in (\\"until_blocked\\", \\"to_border\\")]\\n    out = []\\n    for mv in moves:\\n        for gd in guards:\\n            for default in [None] + [m2 for m2 in moves if m2 != mv]:\\n                rules = [(gd, mv)] + ([((\\"true\\",), default)] if default else [])\\n                if gd == (\\"true\\",) and default:\\n                    continue\\n                ok = True\\n                for x, y in demos:\\n                    o = execute(x, view, rules)\\n                    if o is None or not np.array_equal(o, y):\\n                        ok = False\\n                        break\\n                if ok:\\n                    out.append(rules)\\n                    if len(out) >= max_lists:\\n                        return out\\n    return out\\n\\n\\n# ------------------------------------------------------------------ guarded output frames\\nclass CropEntity(Op):\\n    \\"\\"\\"Crop the input to the box of the UNIQUE entity (under a view) whose guard holds; undefined otherwise.\\"\\"\\"\\n    name = \\"crop_entity\\"\\n    same_shape = False\\n    scope = \\"grids up to 30x30; views mono-4, mono-8, multi-8; guard atoms from the demonstrations\\"\\n\\n    def domain(self, ctx):\\n        scenes = {v: [scene_of(x, v) for x, _ in ctx.train] for v in VIEWS}\\n        return [(v, gd) for v in VIEWS for gd in guard_space(scenes[v]) if gd != (\\"true\\",)]\\n\\n    def apply(self, g, th):\\n        view, gd = th\\n        s = scene_of(g, view)\\n        hits = [o for o in s.objs if holds(o, gd)]\\n        if len(hits) != 1:\\n            return None\\n        return G.crop(s.g, hits[0].box)\\n\\n    def fwd(self, a, th):\\n        if a.bottom:\\n            return D.BOTTOM()\\n        if a.shapes is None:\\n            return D.Abs(None, (1, 1), None, a.pal, 0, None, a.hi)\\n        shapes = {(h, w) for H, W in a.shapes for h in range(1, H + 1) for w in range(1, W + 1)}\\n        return D.Abs(shapes, (1, 1), None, a.pal, 0, None, a.hi)\\n\\n    def bwd(self, y, th):\\n        y = np.asarray(y)\\n        return D.Abs(None, y.shape, None, D.ALL, D.mask_of(np.unique(y).tolist()), G.counts(y), None)\\n\\n\\ndef program_text(view, rules):\\n    return f\\"(rewrite view={view[0]}{view[1]} \\" + \\" \\".join(f\\"[{gd} -> {a}]\\" for gd, a in rules) + \\")\\"\\n\\n\\ndef search(task, max_lists=64, views=VIEWS, true_only=False, frames=True):\\n    \\"\\"\\"All verified rewrite programs (SYNC rule lists, SEQ moves, table rewrites, guarded crop frames) over the given\\n    views, with query predictions. true_only=True restricts every guard to TRUE (control: no guards).\\"\\"\\"\\n    demos = [(np.asarray(x, dtype=np.int64), np.asarray(y, dtype=np.int64)) for x, y in task[\\"train\\"]]\\n    qs = [np.asarray(q, dtype=np.int64) for q in task[\\"test_inputs\\"]]\\n    fits, execs = [], 0\\n    global _TRUE_ONLY\\n    _TRUE_ONLY = true_only\\n    for view in views:\\n        for rules in construct(demos, view, max_lists) + construct_moves(demos, view):\\n            preds = [execute(q, view, rules) for q in qs]\\n            fits.append({\\"program\\": program_text(view, rules), \\"preds\\": preds,\\n                         \\"complete\\": [p is not None for p in preds], \\"cost\\": float(len(rules)),\\n                         \\"order\\": len(fits), \\"view\\": view, \\"rules\\": repr(rules)})\\n        for f, table in construct_tables(demos, view):\\n            preds = [run_table(scene_of(q, view), f, table) for q in qs]\\n            if true_only:\\n                continue\\n            fits.append({\\"program\\": f\\"(table-rewrite view={view[0]}{view[1]} feature={f} n={len(table)})\\",\\n                         \\"preds\\": preds, \\"complete\\": [True] * len(preds), \\"cost\\": float(len(table)),\\n                         \\"order\\": len(fits), \\"view\\": view, \\"rules\\": f\\"table:{f}\\"})\\n    if frames and not true_only and all(np.asarray(x).shape != np.asarray(y).shape for x, y in task[\\"train\\"]):\\n        op = CropEntity()\\n        ctx = type(\\"C\\", (), {\\"train\\": demos})\\n        for th in op.domain(ctx):\\n            if th[0] not in views:\\n                continue\\n            ok = True\\n            for x, y in demos:\\n                o = op.apply(x, th)\\n                if o is None or o.shape != y.shape or not np.array_equal(o, y):\\n                    ok = False\\n                    break\\n            if ok:\\n                preds = [op.apply(q, th) for q in qs]\\n                fits.append({\\"program\\": f\\"(crop-entity view={th[0][0]}{th[0][1]} guard={th[1]})\\", \\"preds\\": preds,\\n                             \\"complete\\": [p is not None for p in preds], \\"cost\\": 1.0, \\"order\\": len(fits),\\n                             \\"view\\": th[0], \\"rules\\": repr(th)})\\n    _TRUE_ONLY = False\\n    return fits\\n\\n\\ndef footprint_ok(demos, view, rules):\\n    \\"\\"\\"M3 contract check for a SYNC rule list on concrete demonstrations: every changed cell must lie in the union of\\n    the matched entities\' write sets (computed without evaluating values).\\"\\"\\"\\n    for x, y in demos:\\n        s = scene_of(x, view)\\n        m = np.zeros(x.shape, dtype=bool)\\n        for o in s.objs:\\n            act = next((a for gd, a in rules if holds(o, gd)), (\\"keep\\",))\\n            if act[0] in (\\"keep\\",):\\n                continue\\n            if act[0] == \\"move\\":\\n                return True                                  # corridor footprint: no cheap rejection here\\n            w = writes(s, o, act)\\n            if w is not None:\\n                m |= w[0]\\n        if ((x != y) & ~m).any():\\n            return False\\n    return True\\n\\n\\n__all__ = [\\"Scene\\", \\"VIEWS\\", \\"construct\\", \\"construct_moves\\", \\"execute\\", \\"search\\", \\"CropEntity\\", \\"guard_space\\",\\n           \\"holds\\", \\"footprint_ok\\"]\\n_unused = itertools\\n", "cast_v1/search.py": "\\"\\"\\"Bounded enumerative search over operator sequences with sound contract pruning (M3).\\n\\nA program is a sequence of operator instances (op, theta) applied left to right to the input grid. A partial program\\n(prefix) is executed concretely on every demonstration input; its remaining completions are the sequences of at\\nmost r further instances from the same grammar instance (stopping early is allowed).\\n\\nModes (the directive\'s mandatory controls; identical enumeration order and grammar):\\n    exact     no pruning: execute everything (exact execution only)\\n    typed     prune with the dimension domain only (ordinary typed constraints)\\n    fwd       sound forward analysis: hole(alpha(z), r) must contain the demonstrated output\\n    fwdbwd    fwd plus a backward predecessor check: meet(hole(alpha(z), r - 1), pre(y)) must be non-empty, where\\n              pre(y) is the join over the grammar of every operator\'s predecessor relation (and y itself)\\nA prune is recorded with its scope (prefix, remaining depth, grammar hash, demonstration index, domain, evidence\\nclass). A prune is never generalized beyond that scope. Search outcome statuses are kept distinct:\\n    FIT                      at least one program reproduces every demonstration\\n    NO_FIT_EXHAUSTED         every program of the bounded grammar was decided (executed or soundly pruned) and none\\n                             fits: \\"bounded-unrealizable in this grammar instance\\", NOT a proof that the task is\\n                             unexpressible\\n    BUDGET_EXHAUSTED         the execution or CPU budget ran out first (unknown)\\nObservational equivalence (optional): prefixes whose outputs on every demonstration AND query input equal an earlier\\nprefix\'s (at the same or larger remaining depth) are skipped; two programs that fit but disagree on a query are never\\nmerged.\\n\\"\\"\\"\\nimport hashlib\\nimport time\\n\\nimport numpy as np\\n\\nfrom . import absdom as D\\nfrom .ops import Identity\\n\\nMODES = (\\"exact\\", \\"typed\\", \\"fwd\\", \\"fwdbwd\\")\\n\\n\\ndef shapes_only(a):\\n    if a.bottom:\\n        return a\\n    return D.Abs(a.shapes, a.shape_lo, None, D.ALL)\\n\\n\\nclass Budget:\\n    def __init__(self, max_exec=None, max_cpu_s=None):\\n        self.max_exec, self.max_cpu_s = max_exec, max_cpu_s\\n        self.t0 = time.process_time()\\n        self.execs = 0\\n        self.transfers = 0\\n\\n    def out(self):\\n        if self.max_exec is not None and self.execs >= self.max_exec:\\n            return True\\n        if self.max_cpu_s is not None and time.process_time() - self.t0 >= self.max_cpu_s:\\n            return True\\n        return False\\n\\n    @property\\n    def cpu_s(self):\\n        return time.process_time() - self.t0\\n\\n\\ndef grammar_hash(instances):\\n    h = hashlib.sha256()\\n    for op, th in instances:\\n        h.update(f\\"{op.name}{th};\\".encode())\\n    return h.hexdigest()[:12]\\n\\n\\nclass Searcher:\\n    def __init__(self, ctx, lib, depth=2, mode=\\"fwdbwd\\", max_exec=None, max_cpu_s=None, oe=True,\\n                 keep_prunes=200, max_fits=64):\\n        assert mode in MODES\\n        self.ctx, self.lib, self.depth, self.mode, self.oe = ctx, lib, depth, mode, oe\\n        self.instances = [(op, th) for op in lib for th in op.domain(ctx)]\\n        self.ghash = grammar_hash(self.instances)\\n        self.budget = Budget(max_exec, max_cpu_s)\\n        self.fits, self.prunes, self.keep_prunes, self.max_fits = [], [], keep_prunes, max_fits\\n        self.n_pruned = {\\"subtree\\": 0, \\"subtree_bwd\\": 0, \\"leaf\\": 0}\\n        self.seen = {}\\n        self.ys = [y for _, y in ctx.train]\\n        self.queries = list(ctx.test_inputs)\\n        self._pre = None\\n        self._hcache = {}\\n        self.complete = True\\n\\n    # ---------------------------------------------------------------- abstract helpers\\n    def _proj(self, a):\\n        return shapes_only(a) if self.mode == \\"typed\\" else a\\n\\n    def hole(self, a, r):\\n        \\"\\"\\"Over-approximation of every completion of length <= r from abstract a (stop included).\\"\\"\\"\\n        cur = self._proj(a)\\n        for _ in range(r):\\n            nxt = cur\\n            for op, th in self.instances:\\n                self.budget.transfers += 1\\n                nxt = D.join(nxt, self._proj(op.fwd(cur, th)))\\n            if _same(nxt, cur):\\n                break\\n            cur = nxt\\n        return cur\\n\\n    def hole_summary(self, z, r):\\n        \\"\\"\\"Sound, cached over-approximation for subtree checks: the hole transfer started from the shape-and-palette\\n        abstraction of z (gamma contains alpha(z), so every completion\'s output is covered). Cached per\\n        (shape, palette, r); per-cell precision is kept for the leaf (instance) checks.\\"\\"\\"\\n        pal = int(D.BITW[np.bincount(z.ravel(), minlength=10)[:10] > 0].sum())\\n        key = (z.shape, pal, r, self.mode)\\n        hit = self._hcache.get(key)\\n        if hit is None:\\n            hit = self.hole(D.Abs({z.shape}, (1, 1), None, pal), r)\\n            self._hcache[key] = hit\\n        return hit\\n\\n    def pre(self, i):\\n        \\"\\"\\"Join of the predecessor relations of every grammar instance for demonstration output i, plus y_i.\\"\\"\\"\\n        if self._pre is None:\\n            self._pre = []\\n            for y in self.ys:\\n                acc = D.alpha(y)\\n                for op, th in self.instances:\\n                    self.budget.transfers += 1\\n                    acc = D.join(acc, op.bwd(y, th))\\n                self._pre.append(acc)\\n        return self._pre[i]\\n\\n    def _record(self, kind, prefix, r, i, why):\\n        self.n_pruned[kind] += 1\\n        if len(self.prunes) < self.keep_prunes:\\n            self.prunes.append({\\"kind\\": kind, \\"prefix\\": [f\\"{op.name}{th}\\" for op, th in prefix], \\"remaining\\": r,\\n                                \\"demo\\": i, \\"grammar\\": self.ghash, \\"domain\\": why,\\n                                \\"evidence\\": \\"PROVED_OPERATOR_CONTRACT\\"})\\n\\n    def subtree_dead(self, prefix, zs, r):\\n        if self.mode == \\"exact\\" or r == 0:\\n            return False\\n        for i, (z, y) in enumerate(zip(zs, self.ys)):\\n            if not D.contains(self.hole_summary(z, r), y):\\n                self._record(\\"subtree\\", prefix, r, i, \\"typed:dims\\" if self.mode == \\"typed\\" else \\"fwd\\")\\n                return True\\n        if self.mode == \\"fwdbwd\\":\\n            for i, (z, y) in enumerate(zip(zs, self.ys)):\\n                if np.array_equal(z, y) if z.shape == y.shape else False:\\n                    continue\\n                if D.meet(self.hole_summary(z, r - 1), self.pre(i)).bottom:\\n                    self._record(\\"subtree_bwd\\", prefix, r, i, \\"fwd-meet-bwd\\")\\n                    return True\\n        return False\\n\\n    def leaf_dead(self, prefix, zs, op, th):\\n        \\"\\"\\"Instance-level check before executing the last operator of a program (remaining depth 1).\\"\\"\\"\\n        if self.mode == \\"exact\\":\\n            return False\\n        for i, (z, y) in enumerate(zip(zs, self.ys)):\\n            self.budget.transfers += 1\\n            if not D.contains(self._proj(op.fwd(D.alpha(z), th)), y):\\n                self._record(\\"leaf\\", prefix + [(op, th)], 0, i, \\"instance contract\\")\\n                return True\\n        return False\\n\\n    # ---------------------------------------------------------------- search\\n    def run(self):\\n        zs = [x for x, _ in self.ctx.train]\\n        self._visit([], zs, list(self.queries), self.depth)\\n        status = \\"FIT\\" if self.fits else (\\"NO_FIT_EXHAUSTED\\" if self.complete else \\"BUDGET_EXHAUSTED\\")\\n        return {\\"status\\": status, \\"fits\\": self.fits, \\"execs\\": self.budget.execs, \\"transfers\\": self.budget.transfers,\\n                \\"cpu_s\\": round(self.budget.cpu_s, 4), \\"pruned\\": dict(self.n_pruned), \\"mode\\": self.mode,\\n                \\"depth\\": self.depth, \\"grammar\\": self.ghash, \\"n_instances\\": len(self.instances)}\\n\\n    def _visit(self, prefix, zs, qs, r):\\n        if self.budget.out():\\n            self.complete = False\\n            return\\n        if prefix and all(z.shape == y.shape and np.array_equal(z, y) for z, y in zip(zs, self.ys)):\\n            if len(self.fits) < self.max_fits:\\n                self.fits.append({\\"program\\": [(op.name, th) for op, th in prefix],\\n                                  \\"preds\\": [None if q is None else q.tolist() for q in qs]})\\n        if r == 0:\\n            return\\n        if prefix and self.subtree_dead(prefix, zs, r):\\n            return\\n        for op, th in self.instances:\\n            if self.budget.out():\\n                self.complete = False\\n                return\\n            if r == 1 and self.leaf_dead(prefix, zs, op, th):\\n                continue\\n            nz = []\\n            ok = True\\n            for z in zs:\\n                self.budget.execs += 1\\n                o = op.apply(z, th)\\n                if o is None:\\n                    ok = False\\n                    break\\n                nz.append(np.asarray(o, dtype=np.int64))\\n            if not ok:\\n                continue\\n            nq = []\\n            for q in qs:\\n                if q is None:\\n                    nq.append(None)\\n                    continue\\n                self.budget.execs += 1\\n                o = op.apply(q, th)\\n                nq.append(None if o is None else np.asarray(o, dtype=np.int64))\\n            if self.oe:\\n                key = tuple(z.shape + (z.tobytes(),) for z in nz) + tuple(\\n                    (None if q is None else q.shape + (q.tobytes(),)) for q in nq)\\n                if self.seen.get(key, -1) >= r - 1:\\n                    continue\\n                self.seen[key] = r - 1\\n            self._visit(prefix + [(op, th)], nz, nq, r - 1)\\n\\n\\ndef _same(a, b):\\n    if a.bottom != b.bottom:\\n        return False\\n    if a.bottom:\\n        return True\\n    if a.shapes != b.shapes or a.shape_lo != b.shape_lo or a.pal != b.pal or a.need != b.need:\\n        return False\\n    if (a.cells is None) != (b.cells is None):\\n        return False\\n    if a.cells is not None and not np.array_equal(a.cells, b.cells):\\n        return False\\n    return bool(np.array_equal(a.lo, b.lo) and np.array_equal(a.hi, b.hi))\\n\\n\\ndef stop_instance():\\n    return Identity(), ()\\n", "cast_v1/symmetry.py": "\\"\\"\\"M4, first item: general symmetry completion (a whole-grid program class with a declared contract).\\n\\nSemantics of SymComplete(c, crop), for an input grid g of shape (h, w) and a mask colour c (a task-level parameter;\\nnever assumed to be 0 and never a background guess):\\n  1. Known cells K = {p : g[p] != c}; masked cells M = {p : g[p] == c}. Undefined if M is empty.\\n  2. Candidate symmetries are lattice isometries with ANY centre or axis (no centroid, no cell-count cap):\\n     reflections about every row axis and column axis at half-integer steps, 180-degree rotations about every\\n     half-integer centre, 90-degree rotations about every centre that maps the lattice to itself, and reflections\\n     about every main and anti diagonal offset.\\n  3. A candidate sigma is VALID on g when (i) at least theta_inside * h * w cells p have sigma(p) inside the grid,\\n     (ii) at least theta_known * |K| known cells p have sigma(p) known, and (iii) g[p] == g[sigma(p)] for every\\n     known p with sigma(p) known (exact agreement; background included, so it adapts to any background).\\n  4. Orbits: the connected components of the graph on cells with an edge p -- sigma(p) for every valid sigma.\\n     If one orbit holds two different known colours the operator is UNDEFINED on g (returns None). Otherwise every\\n     masked cell takes its orbit\'s known colour; masked cells in orbits without a known cell keep colour c\\n     (undetermined; counted in the trace). The result is independent of the order of symmetries.\\n  5. crop=False returns the completed grid; crop=True returns the completed grid restricted to the box of M.\\nThresholds theta_inside and theta_known are frozen on source-selection data (CAST_select), never on evaluation\\noutcomes. Defaults 0.5 / 0.5 follow the synthesis review\'s probe.\\n\\nContract (PROVED_OPERATOR_CONTRACT; argument: only cells of M are written, each with a colour copied from a known\\ncell, i.e. from palette(g) minus c; tests/test_symmetry.py validates it exhaustively on tiny grids):\\n  crop=False: shape unchanged; footprint subset of M; written colours subset of palette(g) - {c};\\n              count_c does not increase; count_k (k != c) does not decrease; total cell count unchanged.\\n  crop=True:  shape == shape of box(M); every output cell outside M keeps g\'s colour; cells of M take a colour in\\n              palette(g) - {c} or keep c.\\nPredecessor relation (crop=False): w[p] in {y[p], c} for every p (and w[p] == c wherever y[p] == c).\\n\\"\\"\\"\\nimport numpy as np\\nfrom . import _nd\\n\\nfrom . import absdom as D\\nfrom . import grids as G\\nfrom .ops import Op\\n\\nTHETA_INSIDE = 0.5\\nTHETA_KNOWN = 0.5\\n\\n\\ndef _maps(h, w, min_inside):\\n    \\"\\"\\"Candidate symmetries as (name, (Y\', X\')) index arrays; analytic overlap pre-filter.\\"\\"\\"\\n    Y, X = np.mgrid[0:h, 0:w]\\n    out = []\\n    for a in range(0, 2 * w - 1):                         # column axis at a / 2\\n        if h * (w - abs(a - (w - 1))) >= min_inside:\\n            out.append((f\\"flip_lr@{a / 2}\\", Y, a - X))\\n    for a in range(0, 2 * h - 1):\\n        if w * (h - abs(a - (h - 1))) >= min_inside:\\n            out.append((f\\"flip_ud@{a / 2}\\", a - Y, X))\\n    for a in range(0, 2 * h - 1):\\n        for b in range(0, 2 * w - 1):\\n            if (h - abs(a - (h - 1))) * (w - abs(b - (w - 1))) >= min_inside:\\n                out.append((f\\"rot180@{a / 2},{b / 2}\\", a - Y, b - X))\\n                if (a - b) % 2 == 0:\\n                    d, s = (a - b) // 2, (a + b) // 2\\n                    out.append((f\\"rot90@{a / 2},{b / 2}\\", X + d, s - Y))\\n    for k in range(-(h - 1), w):                           # (y, x) -> (x - k, y + k): reflection about y = x - k\\n        out.append((f\\"transpose@{k}\\", X - k, Y + k))\\n    for s in range(0, h + w - 1):                          # (y, x) -> (s - x, s - y)\\n        out.append((f\\"antitranspose@{s}\\", s - X, s - Y))\\n    return out\\n\\n\\n_MAP_CACHE = {}\\n\\n\\ndef maps_for(h, w, theta_inside=THETA_INSIDE):\\n    key = (h, w, theta_inside)\\n    if key not in _MAP_CACHE:\\n        if len(_MAP_CACHE) > 16:\\n            _MAP_CACHE.clear()\\n        min_inside = theta_inside * h * w\\n        res = []\\n        for name, YY, XX in _maps(h, w, min_inside):\\n            inside = (YY >= 0) & (YY < h) & (XX >= 0) & (XX < w)\\n            n_in = int(inside.sum())\\n            if n_in < min_inside or n_in == 0:\\n                continue\\n            src = np.flatnonzero(inside).astype(np.int32)\\n            dst = (YY[inside] * w + XX[inside]).astype(np.int32)\\n            if np.array_equal(src, dst):                    # the identity is not a symmetry candidate\\n                continue\\n            res.append((name, src, dst))\\n        _MAP_CACHE[key] = res\\n    return _MAP_CACHE[key]\\n\\n\\ndef valid_symmetries(g, c, theta_inside=THETA_INSIDE, theta_known=THETA_KNOWN):\\n    g = np.asarray(g)\\n    h, w = g.shape\\n    flat = g.ravel()\\n    known = flat != c\\n    nk = int(known.sum())\\n    if nk == 0:\\n        return []\\n    out = []\\n    for name, src, dst in maps_for(h, w, theta_inside):\\n        both = known[src] & known[dst]\\n        nb = int(both.sum())\\n        if nb < theta_known * nk:\\n            continue\\n        if np.array_equal(flat[src[both]], flat[dst[both]]):\\n            out.append((name, src, dst))\\n    return out\\n\\n\\ndef complete(g, c, theta_inside=THETA_INSIDE, theta_known=THETA_KNOWN):\\n    \\"\\"\\"(completed grid or None, trace dict).\\"\\"\\"\\n    g = np.asarray(g, dtype=np.int64)\\n    h, w = g.shape\\n    flat = g.ravel()\\n    masked = flat == c\\n    trace = {\\"mask_colour\\": int(c), \\"masked\\": int(masked.sum())}\\n    if not masked.any():\\n        trace[\\"status\\"] = \\"UNDEFINED_NO_MASK\\"\\n        return None, trace\\n    syms = valid_symmetries(g, c, theta_inside, theta_known)\\n    trace[\\"symmetries\\"] = [s[0] for s in syms]\\n    if not syms:\\n        trace[\\"status\\"] = \\"UNDEFINED_NO_SYMMETRY\\"\\n        return None, trace\\n    n = h * w\\n    rows = np.concatenate([s[1] for s in syms])\\n    cols = np.concatenate([s[2] for s in syms])\\n    ncomp, lab = _nd.components_from_edges(n, rows, cols)\\n    known = ~masked\\n    kl, kc = lab[known], flat[known]\\n    colour_of = np.full(ncomp, -1, dtype=np.int64)\\n    pairs = np.unique(np.stack([kl, kc], 1), axis=0) if len(kl) else np.zeros((0, 2), dtype=np.int64)\\n    if len(pairs) and len(np.unique(pairs[:, 0])) != len(pairs):\\n        trace[\\"status\\"] = \\"UNDEFINED_CONFLICT\\"\\n        return None, trace\\n    colour_of[pairs[:, 0]] = pairs[:, 1]\\n    out = flat.copy()\\n    fill = colour_of[lab]\\n    sel = masked & (fill >= 0)\\n    out[sel] = fill[sel]\\n    trace[\\"undetermined\\"] = int((masked & (fill < 0)).sum())\\n    trace[\\"status\\"] = \\"COMPLETE\\" if trace[\\"undetermined\\"] == 0 else \\"PARTIAL\\"\\n    return out.reshape(h, w), trace\\n\\n\\nclass SymComplete(Op):\\n    name = \\"symcomplete\\"\\n    cost = 20\\n    scope = (\\"grids up to 30x30; mask colour in the demonstrations\' input palette; thresholds frozen on \\"\\n             \\"CAST_select\\")\\n\\n    def __init__(self, theta_inside=THETA_INSIDE, theta_known=THETA_KNOWN):\\n        self.theta_inside, self.theta_known = theta_inside, theta_known\\n        self.last_trace = None\\n\\n    @property\\n    def same_shape(self):\\n        return False\\n\\n    def domain(self, ctx):\\n        return [(c, crop) for c in ctx.in_pal for crop in (False, True)]\\n\\n    def apply(self, g, th):\\n        c, crop = th\\n        out, tr = complete(g, c, self.theta_inside, self.theta_known)\\n        self.last_trace = tr\\n        if out is None:\\n            return None\\n        if crop:\\n            box = G.bbox_of(np.asarray(g) == c)\\n            return G.crop(out, box)\\n        return out\\n\\n    # ---- contract\\n    def _cells_same(self, a, c):\\n        others = a.pal & ~D.bit(c)\\n        has_c = ((a.cells >> c) & 1).astype(bool)\\n        cells = a.cells.copy()\\n        cells[has_c] |= np.uint16(others)\\n        return cells\\n\\n    def fwd(self, a, th):\\n        c, crop = th\\n        if a.bottom or not (a.pal >> c & 1):\\n            return D.BOTTOM()\\n        lo, hi = a.lo.copy(), a.hi.copy()\\n        lo_c, hi_c = lo[c], hi[c]\\n        lo[c] = 0\\n        for k in range(10):\\n            if k != c and (a.pal >> k & 1):\\n                hi[k] = hi[k] + hi_c\\n        others = a.pal & ~D.bit(c)\\n        if not crop:\\n            if a.cells is None:\\n                return D.Abs(a.shapes, a.shape_lo, None, a.pal, 0, lo, hi)\\n            return D.Abs(a.shapes, a.shape_lo, self._cells_same(a, c), a.pal, 0, lo, hi)\\n        if D.is_concrete(a):\\n            g = D.concretize_one(a)\\n            box = G.bbox_of(g == c)\\n            if box is None:\\n                return D.BOTTOM()\\n            cells = G.crop(self._cells_same(a, c), box).astype(np.uint16)\\n            return D.Abs({cells.shape}, (1, 1), cells, a.pal)\\n        if a.shapes is None:\\n            return D.Abs(None, (1, 1), None, a.pal)\\n        shapes = {(hh, ww) for H, W in a.shapes for hh in range(1, H + 1) for ww in range(1, W + 1)}\\n        return D.Abs(shapes, (1, 1), None, a.pal | others)\\n\\n    def bwd(self, y, th):\\n        c, crop = th\\n        y = np.asarray(y)\\n        if crop:\\n            return D.Abs(None, y.shape, None, D.ALL, D.bit(c))\\n        cells = (D.BITS[y] | np.uint16(D.bit(c))).astype(np.uint16)\\n        cells[y == c] = D.bit(c)\\n        return D.Abs({y.shape}, (1, 1), cells, D.ALL, D.bit(c))\\n", "r29_cpu_package_20261101/LICENSE_STATUS.md": "# Private source/permission status\\n\\nThis review candidate contains newly authored wrapper code and exact snapshots of the independently authored CORA R25/R26/R27/CAGE R29 components listed in `VENDORED_PROVENANCE.json`. The original files remain untouched. No NVARC notebook, pretrained checkpoint, third-party source implementation, competition candidate or dataset is bundled. Standard-library APIs are used.\\n\\nNo new public license grant is applied here: the author must approve the release license and component notices before publication. This private inventory is not legal or competition eligibility clearance. A copied source comment acknowledging conceptual predecessors is attribution, not a claim to have copied their code or established rights over them. Existing notices and source comments are retained byte-for-byte in the vendor files.\\n\\nThis package does not inherit a model-license question because it contains no model; any later neural/C4 integration requires its own exact source/model/data permission inventory. No public push, release or submission occurred.\\n", "r29_cpu_package_20261101/MANIFEST.json": "{\\n  \\"schema\\": \\"CORA_R29_PORTABLE_CPU_PACKAGE_MANIFEST_V1\\",\\n  \\"status\\": \\"STATIC_REVIEW_CANDIDATE_TESTS_NOT_RUN\\",\\n  \\"project\\": \\"CORA\\",\\n  \\"subproject\\": \\"CAGE\\",\\n  \\"files\\": {\\n    \\"LICENSE_STATUS.md\\": \\"961a190f57ed497a78e721803016757e0c6ad4c94643304f7e4de74c390f485c\\",\\n    \\"README.md\\": \\"9c433733b3940cb438802117b0ad283cfd850cbfd3927a5fb83982f1d76359ea\\",\\n    \\"VENDORED_PROVENANCE.json\\": \\"32edf758c7e6032ac962fde6c49bbfc4d53042067ef3ec24105e03cdce728bc5\\",\\n    \\"__init__.py\\": \\"101ebb8b163749f38937ca955a046e30832941f4fddcbdf14677926f1f89d556\\",\\n    \\"api.py\\": \\"5006260b401e41099c463835635f8af49d859d76c337f81c4ed6c9a8d7162cbc\\",\\n    \\"cli.py\\": \\"52e9450ffe7e28c26ddbfb5a3bed202b4ac2502a73e97af41758180e44ad6d8d\\",\\n    \\"solver.py\\": \\"21101fb10a55fcafe950a84ad704701c93dce3fae8739d02ac0975d7dd096706\\",\\n    \\"test_package.py\\": \\"8f29a7472684c554baa0273f696e27992fb0c30a0ec188cfca594553233eea8f\\",\\n    \\"vendor/cage/__init__.py\\": \\"8b671d53028b09a9a552f1b361e1a68a1ef06df067c05c41c2dae7fd4ce6417f\\",\\n    \\"vendor/cage/dependencies.py\\": \\"fa95fe4b143c05363e48f2cf9f2c97a68eede57fff9ed1456985c9d6b424f4df\\",\\n    \\"vendor/cage/programs.py\\": \\"b6ccc7a65a19414a59bc26d101e7f15825c619d1389095bc8a1ae3febbf32a82\\",\\n    \\"vendor/cage/v2b/__init__.py\\": \\"e3c9233781d1c3b917a01e737a436be2d0b0961b19d6d34068e7d7a1c0821a8c\\",\\n    \\"vendor/cage/v2b/conditional_model.py\\": \\"39ce0652ce78519c0472b4627bc3e0640e2c1b9aae29f3656d9535f3df2db993\\",\\n    \\"vendor/cage/v2b/goal_constructor.py\\": \\"d067f9d3a5a696be9ec6829e5bdcc2947714bb8ab04ff88e495ef9b948f7092c\\",\\n    \\"vendor/experiments/independent_synthesis_20261029/round25/capability_growth/library.py\\": \\"974dab6c9e98df7f024081c7c4599e7b9242be39cc5f0763987fe5e60ea83dc6\\",\\n    \\"vendor/experiments/independent_synthesis_20261029/round25/diagnosis_constructor/engine.py\\": \\"009788d7071929240a5231032eaeb3e783b1313e92aea14a9a81043dccb56559\\",\\n    \\"vendor/experiments/independent_synthesis_20261029/round26/coverage_constructor/engine.py\\": \\"0e513986bd94565fa0c572f4dbd6dfd5a698a8fa87188f2f876e27f607919a82\\",\\n    \\"vendor/experiments/independent_synthesis_20261029/round27/learned_local_rules/model.py\\": \\"110bcb0c19801d22fc27de7f38b34d1c7e7aca65afdbcb7d0b10980b00a3fbd6\\",\\n    \\"worker.py\\": \\"703ab07bb55c424d7409cf7c18603ec437624fae1ac5cb650b0f5c87d1e6ac4d\\"\\n  },\\n  \\"vendored_sources_byte_exact\\": true,\\n  \\"test_execution\\": false,\\n  \\"prediction_execution\\": false,\\n  \\"source_or_evaluation_values_read\\": false,\\n  \\"Python_3_13_execution\\": \\"NOT_AVAILABLE_LOCALLY\\",\\n  \\"data_weights_or_NVARC_code\\": false,\\n  \\"public_release_or_submission\\": false\\n}\\n", "r29_cpu_package_20261101/README.md": "# CORA R29 portable CPU candidates \\u2014 review candidate\\n\\nThis self-contained private package adapts the frozen R29 forward, diagnosed-inverse and conditional-beam constructors to a stricter single-output interface. It contains ten byte-exact authored CORA source files, no pretrained model, ARC data, NVARC notebook/code, learned registry, C4 artifact or external service integration. This is CAGE work within CORA.\\n\\nThe wrapper is new. **The historical R29 3/32 result does not measure this package:** genuine leave-one-out and single-output priority change both acceptance and predictions. All-abstention is an admissible outcome; validation gates must not be weakened to obtain coverage. No prediction or scoring run has yet been performed by this package\'s author.\\n\\n```python\\nfrom cage.r29_cpu_package_20261101 import candidates, candidates_with_report\\nvalues = candidates(task, cpu_s=12.0, wall_s=20.0)\\nvalues, report = candidates_with_report(task, cpu_s=12.0, wall_s=20.0)\\n```\\n\\n`values` has one entry per test input, each an ARC grid or `None`. `task` contains `train` pairs with exactly `input,output`, and `test` records with exactly `input`. Optional top-level `id/task_id` metadata is ignored without reading its value; no identity reaches the constructors. Other fields, test outputs, invalid grids and invalid budgets fail closed. Grids are rectangular integer colors 0\\u20139, dimensions 1\\u201330, up to ten demonstrations and 100 queries. Booleans are not colors/budgets. Fewer than two demonstrations returns all `None` and `INSUFFICIENT_VALIDATION_EVIDENCE`. No inputs are modified.\\n\\nEach constructor must fit all demonstrations, independently replay them, and pass **every** leave-one-out fold. Every fold calls synthesis anew on all remaining pairs; no full model, fold model, cache or capability registry is shared across folds. Each fold replays its retained examples before predicting the excluded input. Imports/code may be cached within the isolated process; fitted state may not. Conditional trees require at least two retained demonstrations and therefore cannot qualify with only two total demonstrations. The full-model fit never receives test inputs. Only after full/LOO acceptance does its frozen program run on tests.\\n\\nFixed priority is forward, then diagnosed inverse, then conditional beam. Each test query receives the first valid program output from an accepted method. There is no score selection or answer access. Once all query slots are filled later methods are not run. R29\'s two-attempt complementarity is not preserved by a one-grid API; this package makes no claim that this priority is optimal.\\n\\nEvery full/fold fit has the original at-most-one-CPU-second ceiling shared with its own independent replay and subsequent queries, plus the original logical caps. Other folds\' CPU is excluded from that model\'s internal allowance but included in the whole call. Finite generation/semantic/predicate/cell exhaustion is ordinary method abstention (`UNKNOWN`, never impossibility). Actual CPU/wall timeout or an unexpected solver exception invalidates the entire task and returns all `None`. Ordinary query precondition/coverage failure abstains for that query. `KeyboardInterrupt/SystemExit` and external `TimeoutError` cancellation (distinct from the API\'s own deadline exception) are external lifecycle cancellation: the owned child is killed/waited and the original cancellation propagates unchanged.\\n\\nThe parent validates/copies the task, writes a private temporary request, and launches a fresh `python -I -S -B` worker. Thus off-main-thread calls still perform heavy work in a child **main thread**, where SIGPROF CPU and SIGALRM wall timers are installed before JSON/vendor imports. RLIMIT_CPU provides a coarser backup. Interpreter/minimal timer-bootstrap CPU is included in absolute process CPU, while the parent wall deadline includes process startup and all IPC. The parent uses per-PID `wait4` CPU accounting; its calling-thread CPU, serialization and response validation are added, with conservative finalization reserve. Any through-return overrun discards outputs. OS signal granularity and short cleanup tails are measured rather than presented as instruction-exact real-time bounds. No unrelated process or global signal handler is changed; every call uses its own child and temporary files.\\n\\nRuntime assumptions: Linux/POSIX `wait4`, SIGPROF/SIGALRM, RLIMIT_CPU and a supported CPython interpreter. The code targets Python 3.13 and uses only the standard library. **Local Python is 3.12.3; 3.13 compatibility has not been executed or certified.** A Kaggle Python 3.13 smoke test remains required. It does not require NumPy/SciPy, GPU, network, installation or a repository-specific absolute path. Copy this whole directory intact; its vendored `cage/experiments` layout resolves only within `vendor/`. Package hashes are checked inside every worker before solver imports. A mismatch yields all `None`.\\n\\nCLI from the CORA/CAGE checkout:\\n\\n```bash\\npython3 -B -m cage.r29_cpu_package_20261101.cli permitted_input_only_task.json --cpu-s 12 --wall-s 20 --output new_result.json\\n```\\n\\nPortable import outside the checkout: place the whole directory on a parent path and import `r29_cpu_package_20261101`; the child still resolves its private vendor tree. The CLI should then be invoked as `python -m r29_cpu_package_20261101.cli ...`. Outputs are created exclusively; existing files are not overwritten.\\n\\nTests are authored in `test_package.py`, including fresh-fold synthesis versus memorized full-fit rejection, no identity/answer access, invalid budgets/schema, real subprocess CPU/wall guards off the caller main thread, manifest mismatch, lifecycle cancellation cleanup, and repeated actual hflip synthesis. **They have not been run during preparation.** Root must review the frozen package and run them within the existing shared CPU admission protocol; do not bypass occupied CPU slots.\\n\\n`VENDORED_PROVENANCE.json` maps original relative source paths and exact hashes. `MANIFEST.json` identifies the reviewable package; its own hash is recorded outside itself by the reviewer. New acceptance/inference code is distinguished from the immutable original constructors. `LICENSE_STATUS.md` records unresolved release requirements. No public license, competition launch, submission, final selection, manuscript modification or publication is authorized by this package.\\n", "r29_cpu_package_20261101/VENDORED_PROVENANCE.json": "{\\n  \\"schema\\": \\"CORA_R29_PORTABLE_EXACT_VENDOR_V1\\",\\n  \\"files\\": [\\n    {\\n      \\"source\\": \\"cage/__init__.py\\",\\n      \\"vendored\\": \\"vendor/cage/__init__.py\\",\\n      \\"sha256\\": \\"8b671d53028b09a9a552f1b361e1a68a1ef06df067c05c41c2dae7fd4ce6417f\\",\\n      \\"byte_exact\\": true\\n    },\\n    {\\n      \\"source\\": \\"cage/dependencies.py\\",\\n      \\"vendored\\": \\"vendor/cage/dependencies.py\\",\\n      \\"sha256\\": \\"fa95fe4b143c05363e48f2cf9f2c97a68eede57fff9ed1456985c9d6b424f4df\\",\\n      \\"byte_exact\\": true\\n    },\\n    {\\n      \\"source\\": \\"cage/programs.py\\",\\n      \\"vendored\\": \\"vendor/cage/programs.py\\",\\n      \\"sha256\\": \\"b6ccc7a65a19414a59bc26d101e7f15825c619d1389095bc8a1ae3febbf32a82\\",\\n      \\"byte_exact\\": true\\n    },\\n    {\\n      \\"source\\": \\"cage/v2b/__init__.py\\",\\n      \\"vendored\\": \\"vendor/cage/v2b/__init__.py\\",\\n      \\"sha256\\": \\"e3c9233781d1c3b917a01e737a436be2d0b0961b19d6d34068e7d7a1c0821a8c\\",\\n      \\"byte_exact\\": true\\n    },\\n    {\\n      \\"source\\": \\"cage/v2b/goal_constructor.py\\",\\n      \\"vendored\\": \\"vendor/cage/v2b/goal_constructor.py\\",\\n      \\"sha256\\": \\"d067f9d3a5a696be9ec6829e5bdcc2947714bb8ab04ff88e495ef9b948f7092c\\",\\n      \\"byte_exact\\": true\\n    },\\n    {\\n      \\"source\\": \\"cage/v2b/conditional_model.py\\",\\n      \\"vendored\\": \\"vendor/cage/v2b/conditional_model.py\\",\\n      \\"sha256\\": \\"39ce0652ce78519c0472b4627bc3e0640e2c1b9aae29f3656d9535f3df2db993\\",\\n      \\"byte_exact\\": true\\n    },\\n    {\\n      \\"source\\": \\"experiments/independent_synthesis_20261029/round25/diagnosis_constructor/engine.py\\",\\n      \\"vendored\\": \\"vendor/experiments/independent_synthesis_20261029/round25/diagnosis_constructor/engine.py\\",\\n      \\"sha256\\": \\"009788d7071929240a5231032eaeb3e783b1313e92aea14a9a81043dccb56559\\",\\n      \\"byte_exact\\": true\\n    },\\n    {\\n      \\"source\\": \\"experiments/independent_synthesis_20261029/round25/capability_growth/library.py\\",\\n      \\"vendored\\": \\"vendor/experiments/independent_synthesis_20261029/round25/capability_growth/library.py\\",\\n      \\"sha256\\": \\"974dab6c9e98df7f024081c7c4599e7b9242be39cc5f0763987fe5e60ea83dc6\\",\\n      \\"byte_exact\\": true\\n    },\\n    {\\n      \\"source\\": \\"experiments/independent_synthesis_20261029/round26/coverage_constructor/engine.py\\",\\n      \\"vendored\\": \\"vendor/experiments/independent_synthesis_20261029/round26/coverage_constructor/engine.py\\",\\n      \\"sha256\\": \\"0e513986bd94565fa0c572f4dbd6dfd5a698a8fa87188f2f876e27f607919a82\\",\\n      \\"byte_exact\\": true\\n    },\\n    {\\n      \\"source\\": \\"experiments/independent_synthesis_20261029/round27/learned_local_rules/model.py\\",\\n      \\"vendored\\": \\"vendor/experiments/independent_synthesis_20261029/round27/learned_local_rules/model.py\\",\\n      \\"sha256\\": \\"110bcb0c19801d22fc27de7f38b34d1c7e7aca65afdbcb7d0b10980b00a3fbd6\\",\\n      \\"byte_exact\\": true\\n    }\\n  ],\\n  \\"weights_or_data_vendored\\": false,\\n  \\"upstream_NVARC_material\\": false,\\n  \\"history\\": \\"Exact source snapshots used in frozen R29 v2b source pilot; new acceptance wrapper changes eligibility and output policy, not original constructor code.\\"\\n}\\n", "r29_cpu_package_20261101/__init__.py": "\\"\\"\\"CORA R29 CPU portability package; no model, data or competition launch.\\"\\"\\"\\nfrom .api import candidates, candidates_with_report, last_report\\n__all__ = [\'candidates\', \'candidates_with_report\', \'last_report\']\\n", "r29_cpu_package_20261101/api.py": "\\"\\"\\"Portable, fail-closed R29 candidates API; heavy work is process isolated.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport math\\nimport os\\nfrom pathlib import Path\\nimport signal\\nimport subprocess\\nimport sys\\nimport tempfile\\nimport threading\\nimport time\\n\\nHERE = Path(__file__).resolve().parent\\n_LOCAL = threading.local()\\n\\nclass APITimeout(TimeoutError):\\n    \\"\\"\\"Owned per-call timeout; external cancellation retains its identity.\\"\\"\\"\\n    pass\\n\\n\\ndef _grid(value):\\n    if (type(value) is not list or not 1 <= len(value) <= 30 or type(value[0]) is not list\\n            or not 1 <= len(value[0]) <= 30 or any(type(row) is not list or len(row) != len(value[0])\\n            or any(type(v) is not int or not 0 <= v <= 9 for v in row) for row in value)):\\n        raise ValueError(\'GRID_SCHEMA\')\\n    return [row[:] for row in value]\\n\\n\\ndef _arity(task):\\n    if type(task) is dict and type(task.get(\'test\')) is list and len(task[\'test\']) <= 100:\\n        return len(task[\'test\'])\\n    return 0\\n\\n\\ndef _clean_task(task):\\n    # Optional identity metadata is never read, copied, hashed or passed to code.\\n    if type(task) is not dict or set(task) - {\'train\', \'test\', \'id\', \'task_id\'}:\\n        raise ValueError(\'TASK_FIELDS\')\\n    train, tests = task.get(\'train\'), task.get(\'test\')\\n    if type(train) is not list or not 0 <= len(train) <= 10 or type(tests) is not list or not 1 <= len(tests) <= 100:\\n        raise ValueError(\'TASK_SCHEMA\')\\n    out = {\'train\': [], \'test\': []}\\n    for pair in train:\\n        if type(pair) is not dict or set(pair) != {\'input\', \'output\'}:\\n            raise ValueError(\'TRAIN_FIELDS\')\\n        out[\'train\'].append({\'input\': _grid(pair[\'input\']), \'output\': _grid(pair[\'output\'])})\\n    for pair in tests:\\n        if type(pair) is not dict or set(pair) != {\'input\'}:\\n            raise ValueError(\'TEST_INPUT_ONLY_REQUIRED\')\\n        out[\'test\'].append({\'input\': _grid(pair[\'input\'])})\\n    return out\\n\\n\\ndef _wait(proc, deadline):\\n    \\"\\"\\"Wait only this PID, collecting exact child CPU even under concurrent calls.\\"\\"\\"\\n    while True:\\n        pid, status, usage = os.wait4(proc.pid, os.WNOHANG)\\n        if pid:\\n            proc.returncode = os.waitstatus_to_exitcode(status)\\n            return usage.ru_utime + usage.ru_stime\\n        remaining = deadline - time.monotonic()\\n        if remaining <= 0:\\n            raise APITimeout(\'CALL_WALL_CAP\')\\n        time.sleep(min(.005, remaining))\\n\\n\\ndef _stop(proc):\\n    # The isolated worker creates no grandchildren; signal only the owned PID.\\n    if proc is not None and proc.returncode is None:\\n        try:\\n            proc.kill()\\n        except ProcessLookupError:\\n            pass\\n        try:\\n            _, status, usage = os.wait4(proc.pid, 0)\\n            proc.returncode = os.waitstatus_to_exitcode(status)\\n            return usage.ru_utime + usage.ru_stime\\n        except ChildProcessError:\\n            pass\\n    return 0.0\\n\\n\\ndef last_report():\\n    \\"\\"\\"Per-calling-thread diagnostic copy; candidates() retains its list API.\\"\\"\\"\\n    return json.loads(json.dumps(getattr(_LOCAL, \'report\', {})))\\n\\n\\ndef candidates_with_report(task: dict, cpu_s: float, wall_s: float):\\n    started_wall, started_thread = time.monotonic(), time.thread_time()\\n    none = [None] * _arity(task)\\n    report = {\'schema\': \'CORA_R29_PORTABLE_API_V1\', \'status\': \'ABSTAIN\',\\n              \'reason\': \'NOT_STARTED\', \'outputs\': len(none), \'external_answers_received\': False,\\n              \'fresh_LOO_required\': True, \'selected_arm\': None}\\n    proc = None\\n    child_cpu = 0.0\\n    returned, caps_valid = none, False\\n    try:\\n        for value in (cpu_s, wall_s):\\n            if type(value) not in (int, float) or not math.isfinite(value) or value <= 0:\\n                raise ValueError(\'FINITE_POSITIVE_CAP_REQUIRED\')\\n        caps_valid = True\\n        deadline = started_wall + wall_s\\n        clean = _clean_task(task)\\n        none = [None] * len(clean[\'test\'])\\n        returned = none\\n        if len(clean[\'train\']) < 2:\\n            report.update(status=\'INSUFFICIENT_VALIDATION_EVIDENCE\', reason=\'FEWER_THAN_TWO_DEMONSTRATIONS\')\\n            return none, report\\n        with tempfile.TemporaryDirectory(prefix=\'cora_r29_\') as directory:\\n            request, response = Path(directory) / \'request.json\', Path(directory) / \'response.json\'\\n            # Charge this thread\'s validation/serialization/spawn work in addition\\n            # to the exact waited-child CPU. Reserve 10ms for caller finalization.\\n            remaining = cpu_s - (time.thread_time() - started_thread) - .010\\n            if remaining <= 0 or time.monotonic() >= deadline:\\n                raise APITimeout(\'CALL_SETUP_CAP\')\\n            payload = {\'task\': clean}\\n            with request.open(\'x\') as stream:\\n                json.dump(payload, stream, separators=(\',\', \':\'), allow_nan=False)\\n            os.chmod(request, 0o600)\\n            # -I ignores Python environment/user path; -S skips site imports.\\n            remaining = cpu_s - (time.thread_time() - started_thread) - .010\\n            if remaining <= 0:\\n                raise APITimeout(\'CALL_SETUP_CAP\')\\n            proc = subprocess.Popen([sys.executable, \'-I\', \'-S\', \'-B\', str(HERE / \'worker.py\'), str(request), str(response), str(remaining), str(deadline)],\\n                                    stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\\n                                    close_fds=True)\\n            child_cpu = _wait(proc, deadline)\\n            if proc.returncode != 0:\\n                raise RuntimeError(\'WORKER_FAILED_CLOSED\')\\n            if time.monotonic() >= deadline:\\n                raise APITimeout(\'CALL_WALL_CAP\')\\n            if response.stat().st_size > 1_000_000:\\n                raise ValueError(\'WORKER_RESPONSE_SIZE\')\\n            result = json.loads(response.read_text())\\n            if type(result) is not dict or set(result) != {\'predictions\', \'report\'}:\\n                raise ValueError(\'WORKER_RESPONSE_SCHEMA\')\\n            predictions = result[\'predictions\']\\n            if type(predictions) is not list or len(predictions) != len(none):\\n                raise ValueError(\'WORKER_OUTPUT_ARITY\')\\n            predictions = [None if value is None else _grid(value) for value in predictions]\\n            if type(result[\'report\']) is not dict:\\n                raise ValueError(\'WORKER_REPORT_SCHEMA\')\\n            report.update(result[\'report\'])\\n            total = child_cpu + time.thread_time() - started_thread\\n            if total >= cpu_s or time.monotonic() >= deadline:\\n                raise APITimeout(\'CALL_THROUGH_RETURN_CAP\')\\n            report.update(caller_thread_CPU_s=time.thread_time() - started_thread,\\n                          waited_child_CPU_s=child_cpu, combined_CPU_s=total,\\n                          elapsed_wall_s=time.monotonic() - started_wall,\\n                          child_waited=True, actual_child_exit_code=proc.returncode)\\n            returned = predictions\\n            return returned, report\\n    except Exception as error:\\n        if isinstance(error, TimeoutError) and not isinstance(error, APITimeout):\\n            raise\\n        report.update(status=\'ABSTAIN\', reason=type(error).__name__ + \':\' + str(error), selected_arm=None, selected_arms=[None]*len(none))\\n        return none, report\\n    finally:\\n        child_cpu = max(child_cpu, _stop(proc))\\n        total = child_cpu + time.thread_time() - started_thread\\n        elapsed = time.monotonic() - started_wall\\n        if caps_valid and (total >= cpu_s or elapsed >= wall_s):\\n            returned[:] = [None] * len(returned)\\n            report.update(status=\'ABSTAIN\', reason=\'FINAL_THROUGH_RETURN_CAP\', selected_arm=None, selected_arms=[None]*len(returned))\\n        report.update(caller_thread_CPU_s=time.thread_time() - started_thread,\\n                      waited_child_CPU_s=child_cpu, combined_CPU_s=total,\\n                      child_waited=proc is not None and proc.returncode is not None,\\n                      actual_child_exit_code=None if proc is None else proc.returncode,\\n                      elapsed_wall_s=elapsed)\\n        _LOCAL.report = dict(report)\\n\\n\\ndef candidates(task: dict, cpu_s: float, wall_s: float):\\n    return candidates_with_report(task, cpu_s, wall_s)[0]\\n", "r29_cpu_package_20261101/cli.py": "\\"\\"\\"Offline input-only CLI; no dataset paths, answer lookup or submission hooks.\\"\\"\\"\\nimport argparse\\nimport json\\nfrom pathlib import Path\\nfrom .api import candidates_with_report\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=\'CORA R29 full-fit plus genuine LOO CPU candidates\')\\n    parser.add_argument(\'task_json\', type=Path)\\n    parser.add_argument(\'--cpu-s\', type=float, required=True)\\n    parser.add_argument(\'--wall-s\', type=float, required=True)\\n    parser.add_argument(\'--output\', type=Path, required=True)\\n    args = parser.parse_args()\\n    task = json.loads(args.task_json.read_text())\\n    predictions, report = candidates_with_report(task, args.cpu_s, args.wall_s)\\n    with args.output.open(\'x\') as stream:\\n        json.dump({\'predictions\': predictions, \'report\': report}, stream, indent=2, allow_nan=False)\\n        stream.write(\'\\\\n\')\\n\\nif __name__ == \'__main__\':\\n    main()\\n", "r29_cpu_package_20261101/solver.py": "\\"\\"\\"R29 constructors with full-fit and fresh leave-one-out acceptance.\\n\\nNo filesystem dataset access, IDs, hidden outputs or query labels are accepted.\\nThe optional backend seam is solely for authored unit tests; worker uses Backend.\\n\\"\\"\\"\\nimport copy\\nimport hashlib\\nimport json\\nimport time\\n\\nARMS = (\'forward\', \'inverse_diagnosed\', \'conditional_beam\')\\nGOAL_CAPS = {\'CPU_s\': 1.0, \'generated_states\': 512, \'semantic_classes\': 128,\\n             \'primitive_executions\': 2048, \'cell_work\': 2000000,\\n             \'selected_programs\': 1, \'max_depth\': 4, \'library_seeds\': 16}\\nBEAM_CAPS = {\'CPU_s\': 1.0, \'max_depth\': 4, \'max_nodes\': 31, \'min_support\': 2,\\n             \'min_demos\': 2, \'max_samples\': 9000, \'predicate_work\': 2000000,\\n             \'feature_cell_work\': 2000000}\\n\\nclass ClockTimeout(BaseException):\\n    pass\\n\\n\\ndef digest(value):\\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(\',\', \':\'), allow_nan=False).encode()).hexdigest()\\n\\n\\nclass Backend:\\n    def __init__(self):\\n        # No data loading; isolated worker already pins exact package bytes.\\n        from cage.v2b import goal_constructor, conditional_model\\n        self.goal, self.beam = goal_constructor, conditional_model\\n\\n    def fit(self, arm, retained, check):\\n        started = time.process_time()\\n        def bounded():\\n            check()\\n            if time.process_time() - started >= 1.0:\\n                raise ClockTimeout(\'METHOD_FIT_REPLAY_QUERY_CPU_CAP\')\\n        if arm != \'conditional_beam\':\\n            fit = self.goal.fit(copy.deepcopy(retained), arm=arm, caps=GOAL_CAPS, check=bounded)\\n            accepted = fit[\'program\'] is not None and fit.get(\'retained_verified\') is True\\n            reason = fit[\'diagnosis\'][\'reason\']\\n            identity = fit[\'program\'][\'program_sha256\'] if accepted else None\\n        else:\\n            if len(retained) < 2 or any((len(p[\'input\']), len(p[\'input\'][0])) != (len(p[\'output\']), len(p[\'output\'][0])) for p in retained):\\n                return None, {\'reason\': \'CONDITIONAL_LANGUAGE_OR_SUPPORT_UNSUPPORTED\'}\\n            fit = self.beam.train(copy.deepcopy(retained), caps=BEAM_CAPS, check=bounded)\\n            accepted = fit[\'model\'] is not None and fit.get(\'status\') == \'VERIFIED_RETAINED_FIT\'\\n            reason = fit[\'reason\']\\n            identity = fit[\'model\'][\'model_sha256\'] if accepted else None\\n        bounded()\\n        if \'CPU\' in reason or \'OVERRUN\' in reason:\\n            raise ClockTimeout(\'CONSTRUCTOR_CPU_TIMEOUT\')\\n        receipt = {\'arm\': arm, \'reason\': reason, \'program_or_model_sha256\': identity,\\n                   \'retained_sha256\': digest(retained), \'retained_count\': len(retained),\\n                   \'cost\': copy.deepcopy(fit[\'cost\'])}\\n        if not accepted:\\n            return None, receipt\\n        return {\'arm\': arm, \'fit\': fit, \'CPU_s\': time.process_time()-started,\\n                \'cost\': copy.deepcopy(fit[\'cost\'])}, receipt\\n\\n    def predict(self, state, query, check):\\n        started, before = time.process_time(), state[\'CPU_s\']\\n        def bounded():\\n            check()\\n            if before >= 1.0 or before + (time.process_time()-started) >= 1.0:\\n                raise ClockTimeout(\'METHOD_FIT_REPLAY_QUERY_CPU_CAP\')\\n        bounded()\\n        if state[\'arm\'] != \'conditional_beam\':\\n            state[\'fit\'][\'cost\'][\'CPU_s\'] = before\\n            answer = self.goal.predict(state[\'fit\'], copy.deepcopy(query), check=bounded)\\n            reason, prediction = answer[\'reason\'], answer[\'prediction\']\\n            state[\'cost\'] = copy.deepcopy(answer[\'cost\'])\\n        else:\\n            cost = state[\'cost\']\\n            caps = dict(BEAM_CAPS, CPU_s=1.0-before-(time.process_time()-started),\\n                        predicate_work=BEAM_CAPS[\'predicate_work\']-cost[\'predicate_work\'],\\n                        feature_cell_work=BEAM_CAPS[\'feature_cell_work\']-cost[\'feature_cell_work\'])\\n            if caps[\'CPU_s\'] <= 0:\\n                raise ClockTimeout(\'CONDITIONAL_CUMULATIVE_CPU_CAP\')\\n            if min(caps[\'predicate_work\'], caps[\'feature_cell_work\']) <= 0:\\n                return None\\n            answer = self.beam.predict(state[\'fit\'][\'model\'], copy.deepcopy(query), caps=caps, check=bounded)\\n            reason, prediction = answer[\'reason\'], answer[\'prediction\']\\n            for key in (\'predicate_work\', \'feature_cell_work\'):\\n                cost[key] += answer[\'cost\'][key]\\n        bounded()\\n        state[\'CPU_s\'] = before + (time.process_time()-started)\\n        state[\'cost\'][\'CPU_s\'] = state[\'CPU_s\']\\n        if \'CPU\' in reason or \'OVERRUN\' in reason:\\n            raise ClockTimeout(\'QUERY_CPU_TIMEOUT\')\\n        return prediction\\n\\n\\ndef solve(task, check, backend=None):\\n    \\"\\"\\"Each accepted arm must pass full replay and *every* fresh LOO fold.\\"\\"\\"\\n    check()\\n    train, queries = task[\'train\'], task[\'test\']\\n    predictions = [None] * len(queries)\\n    report = {\'status\': \'ABSTAIN\', \'reason\': \'NO_FULL_AND_LOO_VERIFIED_METHOD\',\\n              \'selected_arm\': None, \'selected_arms\': [None] * len(queries),\\n              \'arms\': [], \'fit_invocations\': 0, \'LOO_fits_cache_reused\': False,\\n              \'test_answers_received\': False, \'query_values_used_in_fit\': False}\\n    if len(train) < 2:\\n        report.update(status=\'INSUFFICIENT_VALIDATION_EVIDENCE\', reason=\'FEWER_THAN_TWO_DEMONSTRATIONS\')\\n        return predictions, report\\n    backend = Backend() if backend is None else backend\\n    for arm in ARMS:\\n        check()\\n        arm_report = {\'arm\': arm, \'accepted\': False, \'full_replay_exact\': False, \'folds\': []}\\n        report[\'arms\'].append(arm_report)\\n        report[\'fit_invocations\'] += 1\\n        full, receipt = backend.fit(arm, copy.deepcopy(train), check)\\n        arm_report[\'full_fit\'] = receipt\\n        if full is None:\\n            continue\\n        full_exact = True\\n        for pair in train:\\n            check()\\n            if backend.predict(full, copy.deepcopy(pair[\'input\']), check) != pair[\'output\']:\\n                full_exact = False\\n                break\\n        arm_report[\'full_replay_exact\'] = full_exact\\n        if not full_exact:\\n            continue\\n        all_folds_exact = True\\n        for held_index, held in enumerate(train):\\n            check()\\n            retained = [copy.deepcopy(pair) for index, pair in enumerate(train) if index != held_index]\\n            # New synth invocation and new model every fold. No state/model is\\n            # passed from full-fit or another fold; imported code may be cached.\\n            report[\'fit_invocations\'] += 1\\n            fold, fold_receipt = backend.fit(arm, retained, check)\\n            row = {\'held_index\': held_index, \'fresh_fit_invocation\': report[\'fit_invocations\'],\\n                   \'fit\': fold_receipt, \'held_exact\': False}\\n            arm_report[\'folds\'].append(row)\\n            if fold is not None:\\n                # Verify retained inputs independently even though constructor\\n                # already exact-checks them; then execute the excluded input.\\n                retained_exact = all(backend.predict(fold, copy.deepcopy(p[\'input\']), check) == p[\'output\'] for p in retained)\\n                if retained_exact:\\n                    row[\'held_exact\'] = backend.predict(fold, copy.deepcopy(held[\'input\']), check) == held[\'output\']\\n            if not row[\'held_exact\']:\\n                all_folds_exact = False\\n                break\\n        arm_report[\'accepted\'] = all_folds_exact and len(arm_report[\'folds\']) == len(train)\\n        if not arm_report[\'accepted\']:\\n            continue\\n        for index, query in enumerate(queries):\\n            check()\\n            if predictions[index] is None:\\n                value = backend.predict(full, copy.deepcopy(query[\'input\']), check)\\n                if value is not None:\\n                    predictions[index] = copy.deepcopy(value)\\n                    report[\'selected_arms\'][index] = arm\\n        arm_report[\'full_cost_after_queries\'] = copy.deepcopy(full.get(\'cost\', {}))\\n        if all(value is not None for value in predictions):\\n            break\\n    check()\\n    if any(value is not None for value in predictions):\\n        report.update(status=\'ACCEPTED_FULL_AND_FRESH_LOO\', reason=\'EXECUTED_VERIFIED_PROGRAM\',\\n                      selected_arm=next(a for a in report[\'selected_arms\'] if a is not None))\\n    return predictions, report\\n", "r29_cpu_package_20261101/test_package.py": "\\"\\"\\"Authored package tests: run only under the root-reviewed CPU admission job.\\"\\"\\"\\nimport concurrent.futures\\nimport copy\\nimport hashlib\\nimport importlib.util\\nimport json\\nfrom pathlib import Path\\nimport shutil\\nimport tempfile\\nimport time\\nimport unittest\\nfrom unittest.mock import patch\\n\\nfrom . import api\\n\\nHERE = Path(__file__).resolve().parent\\n_spec = importlib.util.spec_from_file_location(\'_portable_solver_test_only\', HERE / \'solver.py\')\\nsolver = importlib.util.module_from_spec(_spec)\\n_spec.loader.exec_module(solver)\\n\\n\\ndef identity_task():\\n    return {\'train\': [{\'input\': [[1, 0]], \'output\': [[1, 0]]},\\n                      {\'input\': [[2, 0]], \'output\': [[2, 0]]},\\n                      {\'input\': [[3, 0]], \'output\': [[3, 0]]}],\\n            \'test\': [{\'input\': [[4, 0]]}]}\\n\\n\\nclass IdentityBackend:\\n    def __init__(self, memorize=False):\\n        self.fits, self.memorize = [], memorize\\n    def fit(self, arm, retained, check):\\n        self.fits.append(copy.deepcopy(retained))\\n        return {\'seen\': [p[\'input\'] for p in retained], \'cost\': {}, \'fit_number\': len(self.fits)}, {\'retained_count\': len(retained)}\\n    def predict(self, state, query, check):\\n        check()\\n        return copy.deepcopy(query) if not self.memorize or query in state[\'seen\'] else None\\n\\n\\nclass PackageTests(unittest.TestCase):\\n    def test_each_LOO_fold_is_resynthesized_fresh(self):\\n        backend = IdentityBackend()\\n        task = identity_task()\\n        predictions, report = solver.solve(task, lambda: None, backend)\\n        self.assertEqual(predictions, [[[4, 0]]])\\n        self.assertEqual(report[\'fit_invocations\'], 4)\\n        self.assertEqual(len(backend.fits), 4)\\n        self.assertEqual(backend.fits[0], task[\'train\'])\\n        for index, retained in enumerate(backend.fits[1:]):\\n            self.assertEqual(retained, [p for i, p in enumerate(task[\'train\']) if i != index])\\n            self.assertNotIn(task[\'train\'][index], retained)\\n        self.assertTrue(report[\'arms\'][0][\'accepted\'])\\n\\n    def test_memorized_full_fit_fails_genuine_LOO(self):\\n        backend = IdentityBackend(memorize=True)\\n        values, report = solver.solve(identity_task(), lambda: None, backend)\\n        self.assertEqual(values, [None])\\n        self.assertFalse(any(a[\'accepted\'] for a in report[\'arms\']))\\n        self.assertFalse(report[\'LOO_fits_cache_reused\'])\\n\\n    def test_fewer_than_two_demos(self):\\n        task = identity_task(); task[\'train\'] = task[\'train\'][:1]\\n        values, report = api.candidates_with_report(task, 1.0, 1.0)\\n        self.assertEqual(values, [None])\\n        self.assertEqual(report[\'status\'], \'INSUFFICIENT_VALIDATION_EVIDENCE\')\\n\\n    def test_caps_reject_nan_inf_bool_and_nonpositive(self):\\n        for value in (float(\'nan\'), float(\'inf\'), -1, 0, True, \'1\'):\\n            for cpu, wall in ((value, 1.0), (1.0, value)):\\n                values, report = api.candidates_with_report(identity_task(), cpu, wall)\\n                self.assertEqual(values, [None]); self.assertEqual(report[\'status\'], \'ABSTAIN\')\\n\\n    def test_metadata_is_ignored_and_answer_fields_rejected(self):\\n        class Unreadable:\\n            def __str__(self): raise AssertionError(\'identity value read\')\\n            def __repr__(self): raise AssertionError(\'identity value read\')\\n        task = identity_task(); task[\'id\'] = Unreadable(); task[\'task_id\'] = Unreadable()\\n        self.assertEqual(api._clean_task(task), identity_task())\\n        for changed in ({**identity_task(), \'answer\': Unreadable()},\\n                        {**identity_task(), \'test\': [{\'input\': [[1]], \'output\': Unreadable()}]}):\\n            self.assertEqual(api.candidates(changed, 1, 1), [None])\\n        bad = identity_task(); bad[\'train\'][0][\'input\'] = [[True]]\\n        self.assertEqual(api.candidates(bad, 1, 1), [None])\\n\\n    def _copy(self, directory, solver_source, reseal=True):\\n        target = Path(directory) / \'portable\'\\n        shutil.copytree(HERE, target, ignore=shutil.ignore_patterns(\'__pycache__\', \'runs\', \'reports\'))\\n        (target / \'solver.py\').write_text(solver_source)\\n        if reseal:\\n            manifest = json.loads((target / \'MANIFEST.json\').read_text())\\n            manifest[\'files\'][\'solver.py\'] = hashlib.sha256((target / \'solver.py\').read_bytes()).hexdigest()\\n            (target / \'MANIFEST.json\').write_text(json.dumps(manifest))\\n        return target\\n\\n    def _offthread(self, target, cpu, wall):\\n        with patch.object(api, \'HERE\', target), concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:\\n            return pool.submit(api.candidates_with_report, identity_task(), cpu, wall).result(timeout=wall + 3)\\n\\n    def test_running_CPU_cap_off_main_thread(self):\\n        with tempfile.TemporaryDirectory() as directory:\\n            target = self._copy(directory, \'def solve(task,check):\\\\n while True: pass\\\\n\')\\n            started = time.monotonic()\\n            values, report = self._offthread(target, .15, 2.0)\\n            self.assertEqual(values, [None]); self.assertEqual(report[\'status\'], \'ABSTAIN\')\\n            self.assertLess(time.monotonic()-started, 1.5)\\n            self.assertLess(report[\'waited_child_CPU_s\'], .30)\\n\\n    def test_running_wall_cap_off_main_thread(self):\\n        with tempfile.TemporaryDirectory() as directory:\\n            target = self._copy(directory, \'import time\\\\ndef solve(task,check):\\\\n time.sleep(60)\\\\n\')\\n            started = time.monotonic()\\n            values, report = self._offthread(target, 1.0, .15)\\n            self.assertEqual(values, [None]); self.assertEqual(report[\'status\'], \'ABSTAIN\')\\n            self.assertLess(time.monotonic()-started, 1.0)\\n\\n    def test_manifest_mismatch_is_fail_closed(self):\\n        with tempfile.TemporaryDirectory() as directory:\\n            target = self._copy(directory, \\"raise AssertionError(\'should never import\')\\\\n\\", reseal=False)\\n            values, report = self._offthread(target, 1.0, 2.0)\\n            self.assertEqual(values, [None]); self.assertEqual(report[\'status\'], \'ABSTAIN\')\\n\\n    def test_external_lifecycle_cancellation_propagates_after_cleanup(self):\\n        class ExternalTimeout(TimeoutError): pass\\n        real_stop = api._stop\\n        for cancellation in (KeyboardInterrupt(\'cancel\'), SystemExit(\'stop\'), ExternalTimeout(\'outer resource guard\')):\\n            with patch.object(api, \'_wait\', side_effect=cancellation), patch.object(api, \'_stop\', wraps=real_stop) as cleanup:\\n                with self.assertRaises(type(cancellation)) as caught:\\n                    api.candidates_with_report(identity_task(), 1.0, 2.0)\\n                self.assertIs(caught.exception, cancellation)\\n                proc = cleanup.call_args[0][0]\\n                self.assertIsNotNone(proc)\\n                self.assertIsNotNone(proc.returncode)\\n\\n    def test_actual_portable_hflip_is_deterministic(self):\\n        xs = [[[1,0,2],[0,3,0]], [[2,1,0],[3,0,1]], [[0,2,3],[1,0,2]]]\\n        task = {\'train\': [{\'input\': x, \'output\': [row[::-1] for row in x]} for x in xs],\\n                \'test\': [{\'input\': [[2,3,0],[0,0,1]]}]}\\n        one, r1 = api.candidates_with_report(task, 10.0, 20.0)\\n        two, r2 = api.candidates_with_report(task, 10.0, 20.0)\\n        self.assertEqual(one, [[[0,3,2],[1,0,0]]]); self.assertEqual(one, two)\\n        self.assertEqual(r1[\'status\'], \'ACCEPTED_FULL_AND_FRESH_LOO\')\\n        self.assertEqual(r1[\'selected_arms\'], r2[\'selected_arms\'])\\n        self.assertTrue(r1[\'arms\'][0][\'full_replay_exact\'])\\n        self.assertTrue(all(f[\'held_exact\'] for f in r1[\'arms\'][0][\'folds\']))\\n\\nif __name__ == \'__main__\':\\n    unittest.main()\\n", "r29_cpu_package_20261101/vendor/cage/__init__.py": "\\"\\"\\"CORA / CAGE: an isolated CORA grammar-evolution research subproject.\\"\\"\\"\\n__version__ = \'0.1.0\'\\n", "r29_cpu_package_20261101/vendor/cage/dependencies.py": "\\"\\"\\"Exact read-only CORA dependencies inside this isolated worktree.\\"\\"\\"\\nimport hashlib\\nimport importlib.util\\nfrom pathlib import Path\\n\\nROOT = Path(__file__).resolve().parents[1]\\nSPRINT = ROOT / \'experiments\' / \'independent_synthesis_20261029\'\\nPINS = {\\n    \'E25\': (\'round25/diagnosis_constructor/engine.py\', \'009788d7071929240a5231032eaeb3e783b1313e92aea14a9a81043dccb56559\'),\\n    \'G\': (\'round25/capability_growth/library.py\', \'974dab6c9e98df7f024081c7c4599e7b9242be39cc5f0763987fe5e60ea83dc6\'),\\n    \'E\': (\'round26/coverage_constructor/engine.py\', \'0e513986bd94565fa0c572f4dbd6dfd5a698a8fa87188f2f876e27f607919a82\'),\\n    \'M\': (\'round27/learned_local_rules/model.py\', \'110bcb0c19801d22fc27de7f38b34d1c7e7aca65afdbcb7d0b10980b00a3fbd6\'),\\n}\\n\\ndef _load(name):\\n    relative, sha = PINS[name]\\n    path = SPRINT / relative\\n    if hashlib.sha256(path.read_bytes()).hexdigest() != sha:\\n        raise RuntimeError(\'FROZEN_DEPENDENCY_CHANGED:\' + name)\\n    spec = importlib.util.spec_from_file_location(\'_cage_\' + name, path)\\n    module = importlib.util.module_from_spec(spec)\\n    spec.loader.exec_module(module)\\n    return module\\n\\nE25, G, E, M = (_load(name) for name in (\'E25\', \'G\', \'E\', \'M\'))\\n", "r29_cpu_package_20261101/vendor/cage/programs.py": "\\"\\"\\"Typed closed programs, deterministic JSON tokens, pure interpretation.\\"\\"\\"\\nimport copy\\nimport json\\nimport re\\nimport math\\nimport time\\nfrom .dependencies import E, M\\n\\nSCHEMA = \'CAGE_PROGRAM_V1\'\\n\\ndef wrap(body, kind=\'dsl\'):\\n    result = {\'schema\': SCHEMA, \'kind\': kind, \'body\': copy.deepcopy(body)}\\n    result[\'program_sha256\'] = E.digest(result)\\n    validate(result)\\n    return result\\n\\ndef validate(program):\\n    if type(program) is not dict or set(program) != {\'schema\', \'kind\', \'body\', \'program_sha256\'}:\\n        raise E.Failure(\'CAGE_PROGRAM_SCHEMA\')\\n    if program[\'schema\'] != SCHEMA or program[\'kind\'] not in (\'dsl\', \'conditional\'):\\n        raise E.Failure(\'CAGE_PROGRAM_KIND\')\\n    if E.digest({k: v for k, v in program.items() if k != \'program_sha256\'}) != program[\'program_sha256\']:\\n        raise E.Failure(\'CAGE_PROGRAM_HASH\')\\n    if program[\'kind\'] == \'dsl\':\\n        E.validate_ast(program[\'body\'])\\n    else:\\n        M.validate_model(program[\'body\'])\\n    return program\\n\\ndef execute(program, grid, check=lambda: None, meter=None, caps=None):\\n    started=time.process_time()\\n    supplied=dict(caps or {})\\n    if any(k not in (\'CPU_s\',\'predicate_work\',\'feature_cell_work\') for k in supplied):\\n        raise E.Failure(\'UNKNOWN_PROGRAM_EXECUTION_CAP\')\\n    if any(type(v) not in (int,float) or not math.isfinite(v) or v<=0 for v in supplied.values()):\\n        raise E.Failure(\'PROGRAM_EXECUTION_CAP_VALUE\')\\n    def bounded():\\n        check()\\n        if time.process_time()-started>=supplied.get(\'CPU_s\',2.0):\\n            raise E.Budget(\'PROGRAM_CPU_CAP_UNKNOWN\')\\n    bounded()\\n    validate(program)\\n    if program[\'kind\'] == \'dsl\':\\n        return E.execute(program[\'body\'], grid, check=bounded, meter=meter)\\n    Mlimits=dict(M.DEFAULT_CAPS,**supplied)\\n    Mlimits[\'CPU_s\']-=time.process_time()-started\\n    if Mlimits[\'CPU_s\']<=0:raise E.Budget(\'PROGRAM_CPU_CAP_UNKNOWN\')\\n    meter={} if meter is None else meter\\n    for key in (\'predicate_work\',\'cell_work\',\'feature_cell_work\'):meter.setdefault(key,0)\\n    if meter.get(\'cell_limit\') is not None:\\n        Mlimits[\'feature_cell_work\']=min(Mlimits[\'feature_cell_work\'],meter[\'cell_limit\']-meter[\'cell_work\'])\\n    if meter.get(\'predicate_limit\') is not None:\\n        Mlimits[\'predicate_work\']=min(Mlimits[\'predicate_work\'],meter[\'predicate_limit\']-meter[\'predicate_work\'])\\n    if Mlimits[\'feature_cell_work\']<=0 or Mlimits[\'predicate_work\']<=0:\\n        raise E.Budget(\'PROGRAM_CONDITIONAL_SHARED_WORK_CAP_UNKNOWN\')\\n    M._limits(Mlimits)\\n    # Preserve a conservative charge if caller cancellation interrupts M.predict.\\n    meter[\'cell_work\']+=Mlimits[\'feature_cell_work\']\\n    meter[\'feature_cell_work\']+=Mlimits[\'feature_cell_work\']\\n    meter[\'predicate_work\']+=Mlimits[\'predicate_work\']\\n    result = M.predict(program[\'body\'], grid, caps=Mlimits, check=bounded)\\n    meter[\'cell_work\']-=Mlimits[\'feature_cell_work\']-result[\'cost\'][\'feature_cell_work\']\\n    meter[\'feature_cell_work\']-=Mlimits[\'feature_cell_work\']-result[\'cost\'][\'feature_cell_work\']\\n    meter[\'predicate_work\']-=Mlimits[\'predicate_work\']-result[\'cost\'][\'predicate_work\']\\n    if result[\'prediction\'] is None:\\n        raise M.Abstention(result[\'reason\'])\\n    return result[\'prediction\']\\n\\ndef verify(program, train, check=lambda: None, meter=None, caps=None):\\n    E.valid_demos(train)\\n    started=time.process_time()\\n    limits=dict(caps or {})\\n    total=limits.get(\'CPU_s\',2.0)\\n    meter={} if meter is None else meter\\n    for pair in train:\\n        remaining=total-(time.process_time()-started)\\n        if remaining<=0:raise E.Budget(\'PROGRAM_VERIFICATION_TOTAL_CPU_CAP_UNKNOWN\')\\n        if execute(program,pair[\'input\'],check=check,meter=meter,\\n                   caps=dict(limits,CPU_s=remaining))!=pair[\'output\']:\\n            return False\\n    return True\\n\\ndef to_tokens(program):\\n    validate(program)\\n    # JSON lexical tokens preserve integers, parameter strings and typed syntax.\\n    return re.findall(r\'\\"(?:\\\\\\\\.|[^\\"\\\\\\\\])*\\"|-?[0-9]+|true|false|null|[{}\\\\[\\\\],:]\', E.canonical(program))\\n\\ndef from_tokens(tokens):\\n    if type(tokens) is not list or not tokens or len(tokens) > 100000 or any(type(t) is not str for t in tokens):\\n        raise E.Failure(\'CAGE_TOKEN_SCHEMA\')\\n    if sum(map(len, tokens)) > 2000000:\\n        raise E.Failure(\'CAGE_TOKEN_LENGTH\')\\n    program = json.loads(\'\'.join(tokens))\\n    validate(program)\\n    if to_tokens(program) != tokens:\\n        raise E.Failure(\'NONCANONICAL_OR_AMBIGUOUS_TOKENS\')\\n    return program\\n", "r29_cpu_package_20261101/vendor/cage/v2b/__init__.py": "\\"\\"\\"CORA CAGE v2b: preserved v2 failure follow-up with fixed-budget repairs.\\"\\"\\"\\n", "r29_cpu_package_20261101/vendor/cage/v2b/conditional_model.py": "\\"\\"\\"CORA/CAGE bounded conditional-program search, version 2b.\\n\\nThe feature/emission language and typed executable model are the frozen R27\\nlanguage. This constructor changes search: a bounded frontier of candidate\\nsplits receives two-level lookahead and unsuccessful branches can be revisited.\\nIt learns conditionals, not primitive semantics. No query participates in fit.\\n\\nCompiled masks/cache entries save physical work; all mask construction and all\\nunsuccessful split/leaf evaluations are charged conservatively in scalar\\npredicate-work units. Callers share one task allowance by subtracting each\\nreturned cost before calling predict, and supplying a whole-task check callback.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport copy\\nimport hashlib\\nimport importlib.util\\nfrom pathlib import Path\\nimport time\\n\\n_FROZEN = Path(__file__).resolve().parents[2] / \'experiments\' / \'independent_synthesis_20261029\' / \'round27\' / \'learned_local_rules\' / \'model.py\'\\n# Path.parents[2] is the repository for cage/v2/conditional_model.py.\\nFROZEN_SHA256 = \'110bcb0c19801d22fc27de7f38b34d1c7e7aca65afdbcb7d0b10980b00a3fbd6\'\\nif hashlib.sha256(_FROZEN.read_bytes()).hexdigest() != FROZEN_SHA256:\\n    raise RuntimeError(\'FROZEN_R27_MODEL_CHANGED\')\\n_spec = importlib.util.spec_from_file_location(\'_cora_cage_v2_frozen_model\', _FROZEN)\\nM = importlib.util.module_from_spec(_spec)\\n_spec.loader.exec_module(M)\\n\\nFEATURES, NUMERIC = M.FEATURES, M.NUMERIC\\nFailure, Abstention = M.Failure, M.Abstention\\ncanonical, digest, valid_grid, validate_model = M.canonical, M.digest, M.valid_grid, M.validate_model\\nDEFAULT_CAPS = dict(M.DEFAULT_CAPS, CPU_s=1.0, beam_width=8, candidate_width=16, lookahead=2)\\nCONSTRUCTOR = \'bounded_beam_lazy_lookahead_v2b\'\\n\\n\\ndef _limits(caps):\\n    result = dict(DEFAULT_CAPS)\\n    supplied = dict(caps or {})\\n    if any(key not in result for key in supplied):\\n        raise Failure(\'UNKNOWN_CAP\')\\n    result.update(supplied)\\n    core = M._limits({key: value for key, value in result.items() if key in M.DEFAULT_CAPS})\\n    if core[\'CPU_s\'] > 1.0 or core[\'predicate_work\'] > 2000000 or core[\'feature_cell_work\'] > 2000000:\\n        raise Failure(\'SHARED_TASK_ALLOWANCE_EXTENSION\')\\n    for key, maximum in ((\'beam_width\', 8), (\'candidate_width\', 16), (\'lookahead\', 2)):\\n        if type(result[key]) is not int or not 1 <= result[key] <= maximum:\\n            raise Failure(\'BOUNDED_SEARCH_CAP_VALUE\')\\n    core.update({key: result[key] for key in (\'beam_width\', \'candidate_width\', \'lookahead\')})\\n    return core\\n\\n\\nclass _Search:\\n    def __init__(self, examples, emissions, budget):\\n        self.examples, self.emissions, self.budget = examples, emissions, budget\\n        self.n = len(examples)\\n        self.full = (1 << self.n) - 1\\n        self.demo_masks = {}\\n        self.leaf_cache, self.split_cache, self.solve_cache = {}, {}, {}\\n        self.predicates, self.mismatches = [], []\\n        self.stats = {\'search_attempts\': 0, \'candidate_evaluations\': 0,\\n                      \'split_partitions_considered\': 0, \'leaf_evaluations\': 0,\\n                      \'lookahead_evaluations\': 0, \'unresolved_leaves\': 0}\\n        for i, (_, _, demo) in enumerate(examples):\\n            budget.guard()\\n            self.demo_masks[demo] = self.demo_masks.get(demo, 0) | (1 << i)\\n        for emission in emissions:\\n            budget.spend(\'predicate_work\', self.n)\\n            mismatch = 0\\n            for i, (feature, target, _) in enumerate(examples):\\n                if M._emit(emission, feature) != target:\\n                    mismatch |= 1 << i\\n            self.mismatches.append(mismatch)\\n        for feature_index, name in enumerate(FEATURES):\\n            budget.guard()\\n            values = sorted({sample[0][feature_index] for sample in examples})\\n            choices = [(\'eq\', value) for value in values]\\n            if name in NUMERIC:\\n                choices += [(\'le\', value) for value in values[:-1]]\\n            for operation, value in choices:\\n                budget.spend(\'predicate_work\', self.n)\\n                mask = 0\\n                for i, sample in enumerate(examples):\\n                    actual = sample[0][feature_index]\\n                    if actual == value if operation == \'eq\' else actual <= value:\\n                        mask |= 1 << i\\n                self.predicates.append((feature_index, name, operation, value, mask))\\n\\n    def support(self, mask):\\n        self.budget.guard()\\n        return (mask.bit_count() >= self.budget.limits[\'min_support\'] and\\n                sum(bool(mask & demo) for demo in self.demo_masks.values()) >= self.budget.limits[\'min_demos\'])\\n\\n    def info(self, mask):\\n        return {\'support\': mask.bit_count(),\\n                \'support_demos\': sum(bool(mask & demo) for demo in self.demo_masks.values())}\\n\\n    def leaf(self, mask):\\n        self.budget.guard()\\n        if mask not in self.leaf_cache:\\n            self.stats[\'leaf_evaluations\'] += 1\\n            self.budget.spend(\'predicate_work\', mask.bit_count() * len(self.emissions))\\n            ranked = [(mask.__and__(bad).bit_count(), 1 if spec[\'kind\'] == \'feature\' else 2,\\n                       i) for i, (bad, spec) in enumerate(zip(self.mismatches, self.emissions))]\\n            best = min(ranked)\\n            self.leaf_cache[mask] = (best[0], self.emissions[best[2]])\\n        return self.leaf_cache[mask]\\n\\n    def splits(self, mask):\\n        self.budget.guard()\\n        if mask in self.split_cache:\\n            return self.split_cache[mask]\\n        candidates, seen = [], set()\\n        for index, name, operation, value, global_yes in self.predicates:\\n            self.budget.spend(\'predicate_work\', mask.bit_count())\\n            self.stats[\'candidate_evaluations\'] += 1\\n            yes, no = mask & global_yes, mask & ~global_yes\\n            # A duplicate or complementary partition has the same tree-search\\n            # alternatives; its attempted routing is nevertheless charged above.\\n            partition = min(yes, no)\\n            if partition in seen or not self.support(yes) or not self.support(no):\\n                continue\\n            seen.add(partition)\\n            self.stats[\'split_partitions_considered\'] += 1\\n            error = self.leaf(yes)[0] + self.leaf(no)[0]\\n            values = sorted({self.examples[i][0][index] for i in range(self.n) if mask & (1 << i)})\\n            # The inference contract rejects unseen values in this node\'s context.\\n            predicate = {\'feature\': name, \'operation\': operation, \'value\': value,\\n                         \'observed_values\': values}\\n            key = (error, 0 if operation == \'eq\' else 1, index, value)\\n            candidates.append((key, predicate, yes, no))\\n            # Zero is a proved lower bound for classification error. Both\\n            # supported children already have exact executable emissions, so\\n            # evaluating other partitions cannot improve feasibility. The\\n            # attempts and leaf/cache construction above remain fully charged.\\n            if error == 0:\\n                self.split_cache[mask] = [candidates[-1]]\\n                return self.split_cache[mask]\\n        candidates.sort(key=lambda item: item[0])\\n        self.split_cache[mask] = candidates[:self.budget.limits[\'candidate_width\']]\\n        return self.split_cache[mask]\\n\\n    def lower_estimate(self, mask, depth):\\n        self.budget.guard()\\n        error = self.leaf(mask)[0]\\n        if not error or depth >= self.budget.limits[\'max_depth\']:\\n            return error\\n        options = self.splits(mask)\\n        return min(error, options[0][0][0]) if options else error\\n\\n    def solve(self, mask, depth=0):\\n        self.budget.guard()\\n        key = mask, depth\\n        if key in self.solve_cache:\\n            return copy.deepcopy(self.solve_cache[key])\\n        self.stats[\'search_attempts\'] += 1\\n        if not self.support(mask):\\n            return None\\n        error, emission = self.leaf(mask)\\n        if error == 0:\\n            node = dict(self.info(mask), kind=\'emit\', type=\'Color\', emission=copy.deepcopy(emission))\\n            self.solve_cache[key] = node\\n            return copy.deepcopy(node)\\n        if depth >= self.budget.limits[\'max_depth\']:\\n            self.stats[\'unresolved_leaves\'] += 1\\n            self.solve_cache[key] = None\\n            return None\\n        options = self.splits(mask)\\n        ranked = []\\n        for option in options[:self.budget.limits[\'beam_width\']]:\\n            self.budget.guard()\\n            ordinary, predicate, yes_mask, no_mask = option\\n            if self.budget.limits[\'lookahead\'] == 2:\\n                self.stats[\'lookahead_evaluations\'] += 1\\n                estimate = self.lower_estimate(yes_mask, depth + 1) + self.lower_estimate(no_mask, depth + 1)\\n            else:\\n                estimate = ordinary[0]\\n            ranked.append(((estimate, *ordinary), predicate, yes_mask, no_mask))\\n            # Evaluate the frontier lazily. A zero two-level estimate is an\\n            # exact lower bound, so try that constructive witness immediately\\n            # rather than compiling every unrelated numerical child partition.\\n            # Failed recursion is still charged and the next frontier entry is\\n            # explored; no allowance or feature/emission language is enlarged.\\n            if estimate == 0:\\n                yes = self.solve(yes_mask, depth + 1)\\n                no = self.solve(no_mask, depth + 1) if yes is not None else None\\n                if yes is not None and no is not None:\\n                    node = dict(self.info(mask), kind=\'if\', type=\'Color\', predicate=copy.deepcopy(predicate), yes=yes, no=no)\\n                    if _nodes(node) <= self.budget.limits[\'max_nodes\']:\\n                        self.solve_cache[key] = node\\n                        return copy.deepcopy(node)\\n        ranked.sort(key=lambda item: item[0])\\n        for _, predicate, yes_mask, no_mask in ranked[:self.budget.limits[\'beam_width\']]:\\n            self.budget.guard()\\n            yes = self.solve(yes_mask, depth + 1)\\n            if yes is None:\\n                continue\\n            no = self.solve(no_mask, depth + 1)\\n            if no is None:\\n                continue\\n            node = dict(self.info(mask), kind=\'if\', type=\'Color\', predicate=copy.deepcopy(predicate), yes=yes, no=no)\\n            if _nodes(node) <= self.budget.limits[\'max_nodes\']:\\n                self.solve_cache[key] = node\\n                return copy.deepcopy(node)\\n        self.stats[\'unresolved_leaves\'] += 1\\n        self.solve_cache[key] = None\\n        return None\\n\\n\\ndef _nodes(node):\\n    return 1 if node[\'kind\'] == \'emit\' else 1 + _nodes(node[\'yes\']) + _nodes(node[\'no\'])\\n\\n\\ndef train(retained, caps=None, check=lambda: None):\\n    limits = _limits(caps)\\n    budget = M._Budget(limits, check)\\n    if type(retained) is not list or not 2 <= len(retained) <= 10:\\n        raise Failure(\'AT_LEAST_TWO_RETAINED_DEMOS_REQUIRED\')\\n    examples, search, model = [], None, None\\n    reason = \'NO_EXACT_TREE_WITHIN_DECLARED_BEAM\'\\n    try:\\n        colors = set()\\n        for pair in retained:\\n            budget.guard()\\n            if type(pair) is not dict or set(pair) != {\'input\', \'output\'}:\\n                raise Failure(\'RETAINED_DEMO_FIELDS_ONLY\')\\n            valid_grid(pair[\'input\']); valid_grid(pair[\'output\'])\\n            if (len(pair[\'input\']), len(pair[\'input\'][0])) != (len(pair[\'output\']), len(pair[\'output\'][0])):\\n                raise Failure(\'SAME_SHAPED_PAIRS_REQUIRED\')\\n            features = M._features(pair[\'input\'], budget)\\n            targets = [value for row in pair[\'output\'] for value in row]\\n            if len(examples) + len(features) > limits[\'max_samples\']:\\n                raise M._Limit(\'SAMPLE_CAP_UNKNOWN\')\\n            identity = digest(pair)\\n            examples.extend((feature, target, identity) for feature, target in zip(features, targets))\\n            colors.update(targets)\\n        search = _Search(examples, M._emissions(colors), budget)\\n        tree = search.solve(search.full)\\n        if tree is not None:\\n            payload = {\'schema\': \'CORA_CONDITIONAL_COLOR_TREE_V1\', \'features\': list(FEATURES),\\n                       \'background\': 0, \'tree\': tree, \'training_demo_sha256\': digest(retained)}\\n            candidate = dict(payload, model_sha256=digest(payload))\\n            validate_model(candidate)\\n            for pair in retained:\\n                actual = M._apply_features(candidate, M._features(pair[\'input\'], budget), budget)\\n                if actual != [color for row in pair[\'output\'] for color in row]:\\n                    raise Failure(\'FRESH_REPLAY_DISAGREEMENT\')\\n            model, reason = candidate, \'EXACT_ALL_DEMO_CONDITIONAL_PROGRAM\'\\n    except M._Limit as error:\\n        if budget.external and budget.external[-1] is error:\\n            raise\\n        reason = str(error)\\n    elapsed = time.process_time() - budget.started\\n    if elapsed >= limits[\'CPU_s\']:\\n        model, reason = None, \'THROUGH_RETURN_CPU_CAP_UNKNOWN\'\\n    stats = search.stats if search else {}\\n    return {\'model\': model, \'status\': \'VERIFIED_RETAINED_FIT\' if model else \'UNKNOWN\',\\n            \'reason\': reason, \'samples\': len(examples),\\n            \'constructed_nodes\': _nodes(model[\'tree\']) if model else 0,\\n            \'unresolved_leaves\': stats.get(\'unresolved_leaves\', 0),\\n            \'cost\': dict(budget.used, CPU_s=elapsed), \'caps\': limits,\\n            \'query_or_task_id_received\': False, \'primitive_semantics_learned\': False,\\n            \'learned_capability\': \'conditional_Color_program_over_fixed_features\',\\n            \'constructor\': CONSTRUCTOR, **stats}\\n\\n\\ndef predict(model, grid, caps=None, check=lambda: None):\\n    # Frozen interpreter performs fresh features, hash/type checks, observed-value\\n    # checks, all-pixel execution, cancellation and through-return CPU checks.\\n    limits = _limits(caps)\\n    return M.predict(model, grid, {key: value for key, value in limits.items() if key in M.DEFAULT_CAPS}, check)\\n", "r29_cpu_package_20261101/vendor/cage/v2b/goal_constructor.py": "\\"\\"\\"CORA / CAGE v2: finite intermediate-goal construction in the frozen DSL.\\n\\nThe inverse views follow the tested R26 helper\'s all-demo D4 specification\\npropagation, but do not give every view a fresh budget.  Shape constraints apply\\nto a proposed resize\'s *intermediate* goal, rather than requiring its prefix to\\nalready have the final output shape.  This adds search coverage, not semantics.\\nNo query, task identifier, expected program, or teacher prediction enters fit.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport copy\\nimport math\\nimport time\\n\\nfrom ..dependencies import E, G\\nfrom .. import programs\\n\\nARMS = (\'forward\', \'inverse_fixed\', \'inverse_diagnosed\', \'inverse_library\')\\nSUFFIXES = (\'identity\', \'hflip\', \'vflip\', \'r90\', \'r180\', \'r270\', \'transpose\', \'antitranspose\')\\nINVERSE = {\'identity\': \'identity\', \'hflip\': \'hflip\', \'vflip\': \'vflip\',\\n           \'r90\': \'r270\', \'r180\': \'r180\', \'r270\': \'r90\',\\n           \'transpose\': \'transpose\', \'antitranspose\': \'antitranspose\'}\\nDEFAULT_CAPS = dict(E.DEFAULT_CAPS, CPU_s=1.0, generated_states=512,\\n                    semantic_classes=128, primitive_executions=2048,\\n                    cell_work=2000000, selected_programs=1, max_depth=4,\\n                    library_seeds=16)\\nLOGICAL = (\'generated_states\', \'semantic_classes\', \'primitive_executions\', \'cell_work\')\\n\\n\\ndef _caps(caps):\\n    return E._limits(dict(DEFAULT_CAPS, **dict(caps or {})))\\n\\n\\ndef _static_seeds(train, check):\\n    \\"\\"\\"Eight public mixed-family chains; output schema validated, values unused.\\n\\n    Frozen BASE.static_programs calls valid_demos, then only _families(inp(),\\n    None); it does not inspect an output to select a seed or its parameters.\\n    \\"\\"\\"\\n    roots, seen = [], set()\\n    for ast in E.static_programs(train, check=check, max_programs=32):\\n        check()\\n        key = E.digest(ast)\\n        if E.structural_cost(ast) == 1 and key not in seen:\\n            seen.add(key)\\n            roots.append(ast)\\n    seeds = []\\n    for index in range(min(8, len(roots))):\\n        check()\\n        inner = roots[index]\\n        # Cycle through distinct public primitive families before repeats.\\n        outer = next((root for root in roots[index+1:] + roots[:index+1]\\n                      if root[\'op\'] != inner[\'op\']), roots[(index+1) % len(roots)])\\n        joined = copy.deepcopy(outer)\\n        joined[\'args\'] = [copy.deepcopy(inner)]\\n        E.validate_ast(joined, DEFAULT_CAPS[\'max_depth\'])\\n        seeds.append(joined)\\n    return seeds\\n\\n\\ndef _prefixes(static, learned, check):\\n    \\"\\"\\"Interleaved existing root families and seeds; no final-shape filtering.\\"\\"\\"\\n    families = [(\'static\', iter(static)), (\'learned\', iter(learned))]\\n    families += E.BASE._families(E.inp(), None)\\n    # The extension primitives remain available in the common forward stage;\\n    # inverse geometry composes the original public geometry/object families.\\n    while families:\\n        next_round = []\\n        for name, iterator in families:\\n            check()\\n            try:\\n                ast = next(iterator)\\n            except StopIteration:\\n                continue\\n            next_round.append((name, iterator))\\n            yield name, ast\\n        families = next_round\\n\\n\\ndef _goal_order(outputs, goals, diagnosed, check):\\n    \\"\\"\\"Rank retained residual evidence; fixed computes and discards same scores.\\"\\"\\"\\n    scores = []\\n    for index, (suffix, goal) in enumerate(goals):\\n        check()\\n        shape_error, pixel_error = E.BASE._residual(outputs, goal)\\n        scores.append(((shape_error, pixel_error, index), suffix, goal))\\n    if diagnosed:\\n        scores.sort(key=lambda item: item[0])\\n    return scores\\n\\n\\ndef fit(train, *, arm=\'forward\', library=None, caps=None, check=lambda: None):\\n    \\"\\"\\"Return one closed, all-retained-reverified program and carried budgets.\\n\\n    All four arms have the same *total* ceilings. Inverse arms spend at most a\\n    quarter of their logical/CPU allowance on current forward search, then use\\n    remaining resources for backward goals. This is a mechanism comparison,\\n    not a guarantee of preserving forward coverage under a finite allowance.\\n    \\"\\"\\"\\n    started = time.process_time()\\n    limits = _caps(caps)\\n    E.valid_demos(train)\\n    if arm not in ARMS:\\n        raise E.Failure(\'GOAL_ABLATION_ARM\')\\n    library = G.empty_library() if library is None else library\\n    G.validate_library(library)\\n    cost = dict.fromkeys(LOGICAL, 0)\\n    meter = {\'primitive_executions\': 0, \'cell_work\': 0,\\n             \'primitive_limit\': limits[\'primitive_executions\'], \'cell_limit\': limits[\'cell_work\']}\\n    external, stages = [], []\\n    selected, reason, materialized = None, \'FINITE_CONSTRUCTION_EXHAUSTED_UNKNOWN\', {\'expanded\': []}\\n    reserved_child = False\\n    seen, semantics = set(), set()\\n\\n    def bounded():\\n        try:\\n            check()\\n        except BaseException as error:\\n            external.append(error)\\n            raise\\n        if time.process_time()-started >= limits[\'CPU_s\']:\\n            raise E.Budget(\'TOTAL_GOAL_FIT_CPU_CAP_UNKNOWN\')\\n\\n    def fresh(ast):\\n        bounded()\\n        if E.verify(ast, train, check=bounded, meter=meter):\\n            return programs.wrap(ast)\\n        return None\\n\\n    def state(ast):\\n        bounded()\\n        if E.structural_cost(ast) > limits[\'max_depth\']:\\n            return False\\n        key = E.digest(ast)\\n        if key in seen:\\n            return False\\n        if cost[\'generated_states\'] >= limits[\'generated_states\']:\\n            raise E.Budget(\'TOTAL_GOAL_GENERATION_CAP_UNKNOWN\')\\n        seen.add(key)\\n        cost[\'generated_states\'] += 1\\n        return True\\n\\n    def semantic(outputs):\\n        signature = E.canonical(outputs)\\n        if signature in semantics:\\n            return False\\n        if cost[\'semantic_classes\'] >= limits[\'semantic_classes\']:\\n            raise E.Budget(\'TOTAL_GOAL_SEMANTIC_CAP_UNKNOWN\')\\n        semantics.add(signature)\\n        cost[\'semantic_classes\'] += 1\\n        return True\\n\\n    try:\\n        bounded()\\n        static = [ast for ast in _static_seeds(train, bounded)\\n                  if E.structural_cost(ast) <= limits[\'max_depth\']]\\n        # Same library decoding/materialization overhead in every arm. Its\\n        # expansions are admitted only in the fourth, explicitly learned arm.\\n        materialized = G.materialize(library, train=train, constructor=E,\\n                                     check=bounded, max_variants=min(8, limits[\'library_seeds\']))\\n        learned = ([ast for ast in materialized[\'expanded\']\\n                    if E.structural_cost(ast) <= limits[\'max_depth\']]\\n                   if arm == \'inverse_library\' else [])\\n        seeds = static + learned\\n        fraction = 1.0 if arm == \'forward\' else 0.25\\n        child = dict(limits, CPU_s=min(limits[\'CPU_s\']*fraction,\\n                                      limits[\'CPU_s\']-(time.process_time()-started)))\\n        for key in LOGICAL:\\n            child[key] = max(1, int(limits[key]*fraction))\\n            cost[key] += child[key]\\n        reserved_child = True\\n        proposal = E.propose(train, library={\'expanded\': seeds}, caps=child, check=bounded)\\n        diagnostic = proposal[\'diagnosis\']\\n        for key in (\'generated_states\', \'semantic_classes\'):\\n            cost[key] -= child[key]-diagnostic[\'counts\'].get(key, 0)\\n        for key in (\'primitive_executions\', \'cell_work\'):\\n            cost[key] -= child[key]-diagnostic[\'meter\'][key]\\n            meter[key] = cost[key]\\n        reserved_child = False\\n        stages.append({\'kind\': \'forward\', \'diagnosis\': diagnostic,\\n                       \'proposal_sha256\': proposal[\'proposal_sha256\']})\\n        if proposal[\'programs\']:\\n            selected = fresh(proposal[\'programs\'][0])\\n            if selected:\\n                reason = \'EXACT_RETAINED_WITNESS_SEARCH_INCOMPLETE\'\\n        if selected is None and arm != \'forward\':\\n            goals = []\\n            for suffix in SUFFIXES:\\n                bounded()\\n                if suffix == \'identity\':\\n                    transformed = copy.deepcopy(train)\\n                else:\\n                    inverse = E.node(\'d4\', E.inp(), mode=INVERSE[suffix])\\n                    transformed = [{\'input\': copy.deepcopy(pair[\'input\']),\\n                                    \'output\': E.execute(inverse, pair[\'output\'], check=bounded, meter=meter)}\\n                                   for pair in train]\\n                goals.append((suffix, transformed))\\n            inverse_diagnosis = {\'kind\': \'intermediate_goal_search\', \'prefixes\': 0,\\n                                 \'failed_prefixes\': 0, \'resize_candidates\': 0,\\n                                 \'suffix_order_uses_retained_residuals\': arm != \'inverse_fixed\',\\n                                 \'finite_goal_views\': list(SUFFIXES), \'selected_suffix\': None}\\n            stages.append(inverse_diagnosis)\\n            for family, prefix in _prefixes(static, learned, bounded):\\n                if not state(prefix):\\n                    continue\\n                inverse_diagnosis[\'prefixes\'] += 1\\n                try:\\n                    outputs = [E.execute(prefix, pair[\'input\'], check=bounded, meter=meter) for pair in train]\\n                except E.Failure as error:\\n                    if external and external[-1] is error:\\n                        raise\\n                    inverse_diagnosis[\'failed_prefixes\'] += 1\\n                    continue\\n                if not semantic(outputs):\\n                    continue\\n                for _, suffix, goal in _goal_order(outputs, goals, arm != \'inverse_fixed\', bounded):\\n                    bounded()\\n                    # Bare prefixes and shape-derived resizes are both checked.\\n                    # Only inferred *integer* factors in the unchanged DSL are\\n                    # admitted; no scaling assumption supplies expected pixels.\\n                    repairs = [None] + list(E.BASE._infer_resizes(outputs, goal))\\n                    for params in repairs:\\n                        bounded()\\n                        ast = prefix if params is None else E.node(\'resize\', prefix, **params)\\n                        if suffix != \'identity\':\\n                            ast = E.node(\'d4\', ast, mode=suffix)\\n                        if not state(ast):\\n                            # The bare identity prefix was counted above, but\\n                            # still needs its retained-goal comparison once.\\n                            if not (params is None and suffix == \'identity\'):\\n                                continue\\n                        if params is None:\\n                            trial = outputs\\n                        else:\\n                            inverse_diagnosis[\'resize_candidates\'] += 1\\n                            try:\\n                                resize = E.node(\'resize\', E.inp(), **params)\\n                                trial = [E.execute(resize, grid, check=bounded, meter=meter) for grid in outputs]\\n                            except E.Failure as error:\\n                                if external and external[-1] is error:\\n                                    raise\\n                                continue\\n                        # Count and charge every attempted suffix, including\\n                        # failures. Semantic signatures refer to actual final\\n                        # outputs, not differently oriented inverse goals.\\n                        if suffix == \'identity\':\\n                            final_outputs = trial\\n                        else:\\n                            suffix_ast = E.node(\'d4\', E.inp(), mode=suffix)\\n                            final_outputs = [E.execute(suffix_ast, grid, check=bounded, meter=meter)\\n                                             for grid in trial]\\n                        exact = all(grid == pair[\'output\'] for grid, pair in zip(final_outputs, train))\\n                        if not exact:\\n                            semantic(final_outputs)\\n                            continue\\n                        selected = fresh(ast)\\n                        if selected:\\n                            inverse_diagnosis[\'selected_suffix\'] = suffix\\n                            inverse_diagnosis[\'selected_prefix_family\'] = family\\n                            reason = \'EXACT_RETAINED_INTERMEDIATE_GOAL_WITNESS_SEARCH_INCOMPLETE\'\\n                            break\\n                    if selected:\\n                        break\\n                if selected:\\n                    break\\n    except E.Budget as error:\\n        if external and external[-1] is error:\\n            raise\\n        selected, reason = None, str(error)\\n    finally:\\n        if not reserved_child:\\n            cost[\'primitive_executions\'] = meter[\'primitive_executions\']\\n            cost[\'cell_work\'] = meter[\'cell_work\']\\n    attribution = (G.attribute_program(selected[\'body\'], materialized)\\n                   if selected and arm == \'inverse_library\' else None)\\n    cost[\'CPU_s\'] = time.process_time()-started\\n    if cost[\'CPU_s\'] >= limits[\'CPU_s\'] or any(cost[k] > limits[k] for k in LOGICAL):\\n        selected, reason = None, \'TOTAL_GOAL_THROUGH_RETURN_OVERRUN_DISCARDED\'\\n    return {\'schema\': \'CORA_CAGE_GOAL_V2\', \'program\': selected, \'retained_verified\': selected is not None,\\n            \'arm\': arm, \'cost\': cost, \'caps\': limits,\\n            \'diagnosis\': {\'reason\': reason, \'stages\': stages,\\n                          \'library_sha256\': library[\'library_sha256\'],\\n                          \'library_attribution\': attribution if selected else None,\\n                          \'logical_cost_status\': (\'CONSERVATIVE_CHILD_RESERVATION\' if reserved_child else \'RETURNED_METERS\'),\\n                          \'query_or_teacher_received\': False, \'inexpressibility_proven\': False,\\n                          \'new_primitive_semantics\': False,\\n                          \'work_metric\': \'frozen DSL precharged primitive/cell proxy; comparisons and AST preparation charged to CPU\',\\n                          \'inverse_scope\': \'eight exact D4 suffix goals and shape-derived resize over finite public prefixes\'}}\\n\\n\\ndef predict(result, grid, *, check=lambda: None):\\n    \\"\\"\\"Execute one query, mutating cumulative cost for subsequent query calls.\\n\\n    Inter-task wall/CPU elapsed since fit is deliberately not charged. Only this\\n    task\'s fit and prediction calls consume its predeclared total CPU allowance.\\n    No expected query output or answer is accepted.\\n    \\"\\"\\"\\n    started = time.process_time()\\n    if type(result) is not dict or result.get(\'schema\') != \'CORA_CAGE_GOAL_V2\':\\n        raise E.Failure(\'GOAL_RESULT_SCHEMA\')\\n    limits = _caps(result[\'caps\'])\\n    E.valid_grid(grid)\\n    cost = result[\'cost\']\\n    if type(cost) is not dict or set(cost) != set(LOGICAL) | {\'CPU_s\'}:\\n        raise E.Failure(\'GOAL_RESULT_COST\')\\n    if any(type(cost[key]) is not int or cost[key] < 0 or cost[key] > limits[key] for key in LOGICAL):\\n        raise E.Failure(\'GOAL_RESULT_LOGICAL_COST\')\\n    if (type(cost[\'CPU_s\']) not in (int, float) or not math.isfinite(cost[\'CPU_s\'])\\n            or not 0 <= cost[\'CPU_s\']):\\n        raise E.Failure(\'GOAL_RESULT_CPU_COST\')\\n    before = cost[\'CPU_s\']\\n    meter = {\'primitive_executions\': cost[\'primitive_executions\'], \'cell_work\': cost[\'cell_work\'],\\n             \'primitive_limit\': limits[\'primitive_executions\'], \'cell_limit\': limits[\'cell_work\']}\\n    external = []\\n    prediction, reason = None, \'NO_EXACT_RETAINED_PROGRAM_UNKNOWN\'\\n\\n    def bounded():\\n        try:\\n            check()\\n        except BaseException as error:\\n            external.append(error)\\n            raise\\n        # Check the carried cost separately. Left-associated (before + now) -\\n        # started can round below before when a coarse CPU clock has not ticked.\\n        # Already exhausted fit allowances must never permit a query call.\\n        if before >= limits[\'CPU_s\'] or before+(time.process_time()-started) >= limits[\'CPU_s\']:\\n            raise E.Budget(\'TOTAL_GOAL_FIT_PLUS_QUERY_CPU_CAP_UNKNOWN\')\\n\\n    try:\\n        bounded()\\n        if result[\'program\'] is not None:\\n            programs.validate(result[\'program\'])\\n            if result[\'program\'][\'kind\'] != \'dsl\' or not result.get(\'retained_verified\'):\\n                raise E.Failure(\'GOAL_CLOSED_DSL_REQUIRED\')\\n            prediction = E.execute(result[\'program\'][\'body\'], grid, check=bounded, meter=meter)\\n            reason = \'CLOSED_PROGRAM_EXECUTED_QUERY_CORRECTNESS_UNKNOWN\'\\n        bounded()\\n    except (E.Budget, E.Failure) as error:\\n        if external and external[-1] is error:\\n            raise\\n        prediction, reason = None, str(error)\\n    finally:\\n        cost[\'primitive_executions\'] = meter[\'primitive_executions\']\\n        cost[\'cell_work\'] = meter[\'cell_work\']\\n        cost[\'CPU_s\'] = before+(time.process_time()-started)\\n    response = {\'prediction\': prediction, \'status\': \'EXECUTED\' if prediction is not None else \'ABSTAIN\',\\n                \'reason\': reason, \'cost\': copy.deepcopy(cost), \'query_correctness_known\': False}\\n    # Include response construction/cost copying before the final snapshot.\\n    # A supervisor must still charge the small Python return/assignment tail.\\n    cost[\'CPU_s\'] = before+(time.process_time()-started)\\n    response[\'cost\'][\'CPU_s\'] = cost[\'CPU_s\']\\n    if cost[\'CPU_s\'] >= limits[\'CPU_s\'] or any(cost[k] > limits[k] for k in LOGICAL):\\n        response.update(prediction=None, status=\'ABSTAIN\',\\n                        reason=\'TOTAL_GOAL_FIT_PLUS_QUERY_OVERRUN_DISCARDED\')\\n    return response\\n", "r29_cpu_package_20261101/vendor/experiments/independent_synthesis_20261029/round25/capability_growth/library.py": "\\"\\"\\"Train-only executable grammar compression for CORA R25.\\n\\nThis module accepts retained demonstrations and already constructed, closed ASTs.\\nIt cannot open a dataset or receive a query/answer.  A repeated, verified subtree\\nis admitted only when its typed template compresses at least two distinct tasks.\\nEvery learned rule lowers to the unchanged constructor interpreter primitives.\\nRules improve the available search representation, not the primitive semantics.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport copy\\nimport hashlib\\nimport itertools\\nimport json\\nimport math\\nimport time\\nfrom collections import Counter, defaultdict\\n\\n\\nSCHEMA = \\"CORA_R25_GROWTH_LIBRARY_V1\\"\\nDEFAULT_CAPS = {\\n    \\"max_records\\": 64,\\n    \\"max_programs_per_record\\": 2,\\n    \\"max_ast_nodes\\": 16,\\n    \\"max_subtrees\\": 2048,\\n    \\"max_rules\\": 8,\\n    \\"min_support_tasks\\": 2,\\n    \\"min_body_operators\\": 2,\\n    \\"min_MDL_gain\\": 1,\\n    \\"max_replays\\": 4096,\\n    \\"CPU_s\\": 8.0,\\n}\\nFORBIDDEN_AST_OPS = {\\"hole\\", \\"call\\", \\"eval\\", \\"literal\\", \\"var\\", \\"query\\"}\\n\\n\\ndef canonical(value):\\n    return json.dumps(value, sort_keys=True, separators=(\\",\\", \\":\\"), allow_nan=False)\\n\\n\\ndef digest(value):\\n    return hashlib.sha256(canonical(value).encode()).hexdigest()\\n\\n\\nclass GrowthBudget(TimeoutError):\\n    pass\\n\\n\\ndef _walk(ast, path=()):\\n    yield path, ast\\n    for index, child in enumerate(ast[\\"args\\"]):\\n        yield from _walk(child, path + (index,))\\n\\n\\ndef _operator_count(ast):\\n    return sum(node[\\"op\\"] != \\"input\\" for _, node in _walk(ast))\\n\\n\\ndef _replace(ast, path, replacement):\\n    if not path:\\n        return copy.deepcopy(replacement)\\n    result = copy.deepcopy(ast)\\n    parent = result\\n    for index in path[:-1]:\\n        parent = parent[\\"args\\"][index]\\n    parent[\\"args\\"][path[-1]] = copy.deepcopy(replacement)\\n    return result\\n\\n\\ndef _value_kind(value):\\n    # bool must precede int: Python considers bool an int subclass.\\n    if type(value) is bool:\\n        return \\"Bool\\"\\n    if type(value) is int:\\n        return \\"Int\\"\\n    if type(value) is str:\\n        return \\"Enum\\"\\n    if type(value) is float and math.isfinite(value):\\n        return \\"Real\\"\\n    if value is None:\\n        return \\"Null\\"\\n    if type(value) is list:\\n        return [\\"List\\", [_value_kind(item) for item in value]]\\n    if type(value) is dict and all(type(k) is str for k in value):\\n        # Mapping keys such as palette colors are values, not language syntax.\\n        return [\\"Map\\", sorted(set(canonical(_value_kind(v)) for v in value.values()))]\\n    raise ValueError(\\"PARAMETER_VALUE_OUTSIDE_FINITE_JSON\\")\\n\\n\\ndef _valid_train(train):\\n    if (type(train) is not list or not 1 <= len(train) <= 10\\n            or any(type(pair) is not dict or set(pair) != {\\"input\\", \\"output\\"}\\n                   for pair in train)):\\n        raise ValueError(\\"RETAINED_DEMONSTRATION_SCHEMA\\")\\n    for pair in train:\\n        for key in (\\"input\\", \\"output\\"):\\n            grid = pair[key]\\n            if (type(grid) is not list or not 1 <= len(grid) <= 30\\n                    or type(grid[0]) is not list or not 1 <= len(grid[0]) <= 30\\n                    or any(type(row) is not list or len(row) != len(grid[0])\\n                           or any(type(value) is not int or not 0 <= value <= 9 for value in row)\\n                           for row in grid)):\\n                raise ValueError(\\"RETAINED_GRID_SCHEMA\\")\\n    return train\\n\\n\\ndef _closed_ast(ast, max_nodes):\\n    if type(ast) is not dict or set(ast) != {\\"op\\", \\"type\\", \\"args\\", \\"params\\"}:\\n        raise ValueError(\\"CLOSED_AST_SCHEMA\\")\\n    nodes = list(_walk(ast))\\n    if len(nodes) > max_nodes:\\n        raise ValueError(\\"AST_NODE_CAP\\")\\n    for _, node in nodes:\\n        if (set(node) != {\\"op\\", \\"type\\", \\"args\\", \\"params\\"}\\n                or type(node[\\"op\\"]) is not str\\n                or node[\\"op\\"] in FORBIDDEN_AST_OPS\\n                or node[\\"type\\"] != \\"Grid\\"\\n                or type(node[\\"args\\"]) is not list\\n                or type(node[\\"params\\"]) is not dict\\n                or any(type(key) is not str for key in node[\\"params\\"])):\\n            raise ValueError(\\"CLOSED_TYPED_AST_REQUIRED\\")\\n        if node[\\"op\\"] == \\"input\\" and (node[\\"args\\"] or node[\\"params\\"]):\\n            raise ValueError(\\"INPUT_LEAF_REQUIRED\\")\\n        for value in node[\\"params\\"].values():\\n            _value_kind(value)\\n    if sum(node[\\"op\\"] == \\"input\\" for _, node in nodes) != 1:\\n        # R25 constructor uses closed, unary Grid transformations.  Restricting\\n        # to one Grid hole avoids inventing unsupported multi-input semantics.\\n        raise ValueError(\\"SINGLE_GRID_INPUT_REQUIRED\\")\\n    return ast\\n\\n\\ndef _shape(ast):\\n    return {\\"op\\": ast[\\"op\\"], \\"type\\": ast[\\"type\\"],\\n            \\"params\\": {key: _value_kind(value)\\n                       for key, value in sorted(ast[\\"params\\"].items())},\\n            \\"args\\": [_shape(child) for child in ast[\\"args\\"]]}\\n\\n\\ndef _parameter_locations(ast):\\n    for path, node in _walk(ast):\\n        for key in sorted(node[\\"params\\"]):\\n            yield path, key, node[\\"params\\"][key]\\n\\n\\ndef _at(ast, path):\\n    for index in path:\\n        ast = ast[\\"args\\"][index]\\n    return ast\\n\\n\\ndef _template(members):\\n    \\"\\"\\"Least general structural anti-unifier, generalizing typed params only.\\n\\n    Operators, Grid edges and parameter names are never generalized.  A slot is\\n    introduced only where its observed values differ, and equal value vectors\\n    share a slot.  Thus repeated parameter equality is preserved by construction.\\n    \\"\\"\\"\\n    first = members[0][\\"subtree\\"]\\n    body = copy.deepcopy(first)\\n    specifications = []\\n    shared_slots = {}\\n    for path, key, original in _parameter_locations(first):\\n        values = [_at(member[\\"subtree\\"], path)[\\"params\\"][key] for member in members]\\n        if all(value == original for value in values):\\n            continue\\n        kinds = [_value_kind(value) for value in values]\\n        if any(kind != kinds[0] for kind in kinds):\\n            raise ValueError(\\"ANTI_UNIFICATION_TYPE_CHANGE\\")\\n        signature = canonical({\\"kind\\": kinds[0], \\"values\\": values})\\n        if signature not in shared_slots:\\n            name = \\"p\\" + str(len(specifications))\\n            shared_slots[signature] = name\\n            unique = {canonical(value): value for value in values}\\n            specifications.append({\\"name\\": name, \\"kind\\": kinds[0],\\n                                   \\"observed_values\\": [unique[k] for k in sorted(unique)],\\n                                   \\"locations\\": []})\\n        name = shared_slots[signature]\\n        specification = next(p for p in specifications if p[\\"name\\"] == name)\\n        specification[\\"locations\\"].append({\\"path\\": list(path), \\"key\\": key})\\n        _at(body, path)[\\"params\\"][key] = {\\"$param\\": name}\\n    bindings = []\\n    for member in members:\\n        binding = {}\\n        for specification in specifications:\\n            location = specification[\\"locations\\"][0]\\n            binding[specification[\\"name\\"]] = copy.deepcopy(\\n                _at(member[\\"subtree\\"], location[\\"path\\"])[\\"params\\"][location[\\"key\\"]])\\n        bindings.append(binding)\\n    return body, specifications, bindings\\n\\n\\ndef _instantiate(body, specs, binding):\\n    if set(binding) != {spec[\\"name\\"] for spec in specs}:\\n        raise ValueError(\\"RULE_PARAMETER_BINDING_SCHEMA\\")\\n    result = copy.deepcopy(body)\\n    for spec in specs:\\n        value = binding[spec[\\"name\\"]]\\n        if _value_kind(value) != spec[\\"kind\\"]:\\n            raise ValueError(\\"RULE_PARAMETER_TYPE_CHANGE\\")\\n        for location in spec[\\"locations\\"]:\\n            target = _at(result, location[\\"path\\"])[\\"params\\"]\\n            if target[location[\\"key\\"]] != {\\"$param\\": spec[\\"name\\"]}:\\n                raise ValueError(\\"RULE_PARAMETER_LOCATION_CHANGED\\")\\n            target[location[\\"key\\"]] = copy.deepcopy(value)\\n    return result\\n\\n\\ndef _library_payload(library):\\n    result = copy.deepcopy(library)\\n    result.pop(\\"library_sha256\\", None)\\n    return result\\n\\n\\ndef _overlaps(a, b):\\n    if a[\\"retained_sha256\\"] != b[\\"retained_sha256\\"]:\\n        return False\\n    # Distinct fitted aliases from one task do not constitute independent\\n    # compression evidence; charge at most one parent representation per task.\\n    if a[\\"verified_parent_program_sha256\\"] != b[\\"verified_parent_program_sha256\\"]:\\n        return True\\n    x, y = a[\\"subtree_path\\"], b[\\"subtree_path\\"]\\n    return x[:len(y)] == y or y[:len(x)] == x\\n\\n\\ndef empty_library():\\n    result = {\\"schema\\": SCHEMA, \\"rules\\": {}, \\"primitive_semantics_learned\\": False,\\n              \\"query_values_received\\": False, \\"held_answers_received\\": False}\\n    result[\\"library_sha256\\"] = digest(_library_payload(result))\\n    return result\\n\\n\\ndef validate_library(library):\\n    if (type(library) is not dict or library.get(\\"schema\\") != SCHEMA\\n            or type(library.get(\\"rules\\")) is not dict\\n            or library.get(\\"library_sha256\\") != digest(_library_payload(library))):\\n        raise ValueError(\\"FROZEN_LIBRARY_CHANGED\\")\\n    for rid, rule in library[\\"rules\\"].items():\\n        payload = copy.deepcopy(rule)\\n        rule_hash = payload.pop(\\"rule_sha256\\", None)\\n        if digest(payload) != rule_hash or rid != \\"r_\\" + rule_hash[:20]:\\n            raise ValueError(\\"FROZEN_RULE_CHANGED\\")\\n        if (rule[\\"MDL\\"][\\"total_gain\\"] <= 0 or rule[\\"support_tasks\\"] < 2\\n                or rule[\\"body_operators\\"] < 2):\\n            raise ValueError(\\"NON_GROWTH_RULE_ADMISSION\\")\\n    return library\\n\\n\\ndef _diagnosis(counts, status):\\n    if counts[\\"exact_verified_programs\\"] == 0:\\n        return {\\"bottleneck\\": \\"NO_EXECUTABLE_DISCOVERY_CORPUS\\",\\n                \\"next_train_only_action\\": \\"construct_and_exact_verify_demonstration_programs_before_induction\\"}\\n    if counts[\\"subtrees_collected\\"] == 0:\\n        return {\\"bottleneck\\": \\"SOLVES_HAVE_NO_MULTI_OPERATOR_REUSABLE_STRUCTURE\\",\\n                \\"next_train_only_action\\": \\"use_retained_failure_residuals_to_construct_compositions\\"}\\n    if counts[\\"admitted_rules\\"] == 0:\\n        return {\\"bottleneck\\": \\"NO_REPEATED_COMPRESSIVE_STRUCTURE\\",\\n                \\"next_train_only_action\\": \\"increase_diverse_solved_discovery_support_without_relaxing_exactness\\"}\\n    return {\\"bottleneck\\": \\"TRANSFER_UTILITY_UNMEASURED\\",\\n            \\"next_train_only_action\\": \\"freeze_library_and_compare_fresh_composition_transfer_under_matched_budgets\\",\\n            \\"induction_interrupted\\": status != \\"COMPLETE\\"}\\n\\n\\ndef induce(discovery_records, constructor, caps=None, check=lambda: None):\\n    \\"\\"\\"Learn from solved retained demonstrations only; return immutable library.\\n\\n    ``discovery_records`` has exactly train, program/programs, optional\\n    opaque_index.  Identical train contents count as one task even under distinct\\n    indices.  Parent-program replay establishes semantic support for subtrees.\\n    No query or target is accepted, copied into a library, or used for gating.\\n    \\"\\"\\"\\n    started = time.process_time()\\n    limits = dict(DEFAULT_CAPS)\\n    if caps:\\n        if not set(caps) <= set(limits):\\n            raise ValueError(\\"UNKNOWN_GROWTH_CAP\\")\\n        limits.update(caps)\\n    for key, value in limits.items():\\n        if type(value) not in (int, float) or not math.isfinite(value) or value <= 0:\\n            raise ValueError(\\"POSITIVE_FINITE_GROWTH_CAP\\")\\n        if key != \\"CPU_s\\" and type(value) is not int:\\n            raise ValueError(\\"INTEGER_GROWTH_CAP\\")\\n    if type(discovery_records) is not list or len(discovery_records) > limits[\\"max_records\\"]:\\n        raise ValueError(\\"DISCOVERY_RECORD_CAP\\")\\n    counters = {\\"input_records\\": len(discovery_records), \\"duplicate_task_records\\": 0,\\n                \\"programs_examined\\": 0, \\"exact_verified_programs\\": 0,\\n                \\"rejected_programs\\": 0, \\"subtrees_collected\\": 0,\\n                \\"structural_groups\\": 0, \\"support_rejections\\": 0,\\n                \\"MDL_rejections\\": 0, \\"context_replay_failures\\": 0,\\n                \\"retained_replays\\": 0, \\"admitted_rules\\": 0}\\n    rejection_reasons = defaultdict(int)\\n\\n    def checked():\\n        check()\\n        if time.process_time() - started >= limits[\\"CPU_s\\"]:\\n            raise GrowthBudget(\\"GROWTH_CPU_CAP_UNKNOWN\\")\\n        if counters[\\"retained_replays\\"] >= limits[\\"max_replays\\"]:\\n            raise GrowthBudget(\\"GROWTH_REPLAY_CAP_UNKNOWN\\")\\n\\n    def exact(program, train):\\n        for pair in train:\\n            checked()\\n            counters[\\"retained_replays\\"] += 1\\n            value = constructor.execute(program, pair[\\"input\\"], library={}, check=checked)\\n            if value != pair[\\"output\\"]:\\n                return False\\n        return True\\n\\n    groups = defaultdict(list)\\n    source_provenance = []\\n    seen_tasks = set()\\n    status = \\"COMPLETE\\"\\n    admitted = []\\n    try:\\n        for record in discovery_records:\\n            checked()\\n            if (type(record) is not dict\\n                    or not set(record) <= {\\"train\\", \\"program\\", \\"programs\\", \\"opaque_index\\"}\\n                    or \\"train\\" not in record\\n                    or (\\"program\\" in record) == (\\"programs\\" in record)):\\n                raise ValueError(\\"TRAIN_ONLY_DISCOVERY_RECORD_REQUIRED\\")\\n            train = record[\\"train\\"]\\n            _valid_train(train)\\n            task_hash = digest(train)\\n            if task_hash in seen_tasks:\\n                counters[\\"duplicate_task_records\\"] += 1\\n                continue\\n            seen_tasks.add(task_hash)\\n            programs = record.get(\\"programs\\", [record.get(\\"program\\")])\\n            if type(programs) is not list or len(programs) > limits[\\"max_programs_per_record\\"]:\\n                raise ValueError(\\"PROGRAMS_PER_DISCOVERY_RECORD_CAP\\")\\n            task_verified_hashes = []\\n            for program in programs:\\n                checked()\\n                counters[\\"programs_examined\\"] += 1\\n                try:\\n                    _closed_ast(program, limits[\\"max_ast_nodes\\"])\\n                    constructor.validate_ast(program)\\n                    verified = exact(program, train)\\n                except GrowthBudget:\\n                    raise\\n                except (ValueError, TypeError, KeyError, IndexError) as error:\\n                    counters[\\"rejected_programs\\"] += 1\\n                    rejection_reasons[type(error).__name__ + \\":\\" + str(error)[:96]] += 1\\n                    continue\\n                if not verified:\\n                    counters[\\"rejected_programs\\"] += 1\\n                    rejection_reasons[\\"EXACT_RETAINED_REPLAY_FAILED\\"] += 1\\n                    continue\\n                counters[\\"exact_verified_programs\\"] += 1\\n                program_hash = digest(program)\\n                task_verified_hashes.append(program_hash)\\n                local_seen = set()\\n                for path, subtree in _walk(program):\\n                    checked()\\n                    if _operator_count(subtree) < limits[\\"min_body_operators\\"]:\\n                        continue\\n                    subtree_hash = digest(subtree)\\n                    if subtree_hash in local_seen:\\n                        continue\\n                    local_seen.add(subtree_hash)\\n                    if counters[\\"subtrees_collected\\"] >= limits[\\"max_subtrees\\"]:\\n                        raise GrowthBudget(\\"GROWTH_SUBTREE_CAP_UNKNOWN\\")\\n                    groups[canonical(_shape(subtree))].append({\\n                        \\"task_hash\\": task_hash, \\"train\\": train,\\n                        \\"program\\": program, \\"program_hash\\": program_hash,\\n                        \\"path\\": path, \\"subtree\\": subtree})\\n                    counters[\\"subtrees_collected\\"] += 1\\n            source_provenance.append({\\"retained_sha256\\": task_hash,\\n                                      \\"verified_program_sha256\\": sorted(task_verified_hashes)})\\n        counters[\\"structural_groups\\"] = len(groups)\\n        for _, raw_members in sorted(groups.items()):\\n            checked()\\n            # A source task contributes once per structural template.  This\\n            # conservative choice prevents one search\'s aliases inflating gain.\\n            task_members = {}\\n            for member in sorted(raw_members, key=lambda m: (m[\\"task_hash\\"], m[\\"program_hash\\"], m[\\"path\\"])):\\n                task_members.setdefault(member[\\"task_hash\\"], member)\\n            members = list(task_members.values())\\n            if len(members) < limits[\\"min_support_tasks\\"]:\\n                counters[\\"support_rejections\\"] += 1\\n                continue\\n            body, specifications, bindings = _template(members)\\n            body_operators = _operator_count(body)\\n            # Declared token-level MDL model: one symbol per primitive, one per\\n            # macro call, one per parameter binding, plus a definition cost of\\n            # body primitive symbols and parameter declarations.  Shared input\\n            # and unchanged literal constants cancel on both sides.  This is\\n            # explicit grammar compression, not a probability/accuracy metric.\\n            without = body_operators * len(members)\\n            definition = body_operators + len(specifications)\\n            calls = len(members) * (1 + len(specifications))\\n            gain = without - definition - calls\\n            if gain < limits[\\"min_MDL_gain\\"]:\\n                counters[\\"MDL_rejections\\"] += 1\\n                continue\\n            fresh_context_verified = True\\n            lineage = []\\n            for member, binding in zip(members, bindings):\\n                checked()\\n                replacement = _instantiate(body, specifications, binding)\\n                if replacement != member[\\"subtree\\"]:\\n                    raise ValueError(\\"ANTI_UNIFICATION_ROUNDTRIP_FAILED\\")\\n                lowered = _replace(member[\\"program\\"], member[\\"path\\"], replacement)\\n                if not exact(lowered, member[\\"train\\"]):\\n                    fresh_context_verified = False\\n                    counters[\\"context_replay_failures\\"] += 1\\n                    break\\n                lineage.append({\\"retained_sha256\\": member[\\"task_hash\\"],\\n                                \\"verified_parent_program_sha256\\": member[\\"program_hash\\"],\\n                                \\"subtree_path\\": list(member[\\"path\\"]),\\n                                \\"subtree_sha256\\": digest(member[\\"subtree\\"]),\\n                                \\"binding_sha256\\": digest(binding),\\n                                \\"fresh_context_exact\\": True})\\n            if not fresh_context_verified:\\n                continue\\n            unique_bindings = {canonical(binding): binding for binding in bindings}\\n            rule = {\\"version\\": 1, \\"body\\": body, \\"parameters\\": specifications,\\n                    \\"observed_bindings\\": [unique_bindings[k] for k in sorted(unique_bindings)],\\n                    \\"support_tasks\\": len(members), \\"body_operators\\": body_operators,\\n                    \\"MDL\\": {\\"model\\": \\"primitive_call_parameter_symbol_v1\\",\\n                            \\"without_rule\\": without, \\"definition\\": definition,\\n                            \\"calls_and_bindings\\": calls, \\"total_gain\\": gain},\\n                    \\"lineage\\": lineage,\\n                    \\"semantic_origin\\": \\"train_only_exact_verified_discovery_subtree\\",\\n                    \\"new_primitive_semantics\\": False}\\n            rule[\\"rule_sha256\\"] = digest(rule)\\n            admitted.append(rule)\\n    except GrowthBudget as error:\\n        status = str(error)\\n    # Partial induction can retain only fully replayed, individually sealed\\n    # rules.  A budget interruption can never admit the in-progress candidate.\\n    selected, charged = [], []\\n    for rule in sorted(admitted, key=lambda r: (-r[\\"MDL\\"][\\"total_gain\\"],\\n                       -r[\\"support_tasks\\"], r[\\"rule_sha256\\"])):\\n        # Do not add gains from overlapping occurrences.  Conservative reject\\n        # rather than refit a different template after observing the selection.\\n        if any(_overlaps(lineage, earlier) for lineage in rule[\\"lineage\\"] for earlier in charged):\\n            rejection_reasons[\\"OVERLAPPING_COMPRESSION_EVIDENCE\\"] += 1\\n            continue\\n        selected.append(rule)\\n        charged.extend(rule[\\"lineage\\"])\\n        if len(selected) >= limits[\\"max_rules\\"]:\\n            break\\n    library = empty_library()\\n    library[\\"rules\\"] = {\\"r_\\" + rule[\\"rule_sha256\\"][:20]: rule for rule in selected}\\n    library[\\"source_provenance_sha256\\"] = digest(sorted(source_provenance,\\n                                                     key=lambda p: p[\\"retained_sha256\\"]))\\n    library[\\"library_sha256\\"] = digest(_library_payload(library))\\n    counters[\\"admitted_rules\\"] = len(selected)\\n    receipt = {\\"schema\\": \\"CORA_R25_GROWTH_RECEIPT_V1\\", \\"status\\": status,\\n               \\"UNKNOWN\\": status != \\"COMPLETE\\", \\"caps\\": limits,\\n               \\"counts\\": counters, \\"rejection_reasons\\": dict(rejection_reasons),\\n               \\"source_provenance\\": sorted(source_provenance, key=lambda p: p[\\"retained_sha256\\"]),\\n               \\"library_sha256\\": library[\\"library_sha256\\"],\\n               \\"actual_CPU_s\\": time.process_time() - started,\\n               \\"query_values_received\\": False, \\"held_answers_received\\": False,\\n               \\"description_length_is_declared_symbol_model\\": True,\\n               \\"selected_total_disjoint_MDL_gain\\": sum(rule[\\"MDL\\"][\\"total_gain\\"] for rule in selected),\\n               \\"overlapping_occurrences_not_double_counted\\": True,\\n               \\"same_interpreter_replay_is_not_independent_semantic_oracle\\": True}\\n    receipt[\\"diagnosis\\"] = _diagnosis(counters, status)\\n    receipt[\\"receipt_sha256\\"] = digest(receipt)\\n    return {\\"library\\": library, \\"receipt\\": receipt}\\n\\n\\ndef _external_domain(domains, op, key):\\n    if not isinstance(domains, dict):\\n        return []\\n    selected = domains.get(op, {})\\n    if not isinstance(selected, dict):\\n        return []\\n    values = selected.get(key, [])\\n    if isinstance(values, range):\\n        values = list(values)\\n    return list(values) if isinstance(values, (tuple, list)) else []\\n\\n\\ndef materialize(library, train=None, constructor=None, check=lambda: None,\\n                max_variants=64, expand_parameter_domains=True):\\n    \\"\\"\\"Lower learned templates to closed AST seeds, without receiving a query.\\n\\n    Observed complete parameter bindings come first.  Then one-parameter\\n    changes use the constructor\'s *same public static domains*.  The unchanged\\n    constructor must exact-verify every materialized seed on retained examples.\\n    No successful transfer or failed query is fed back to this library.\\n    \\"\\"\\"\\n    validate_library(library)\\n    if train is not None:\\n        _valid_train(train)\\n    if type(max_variants) is not int or max_variants < 0:\\n        raise ValueError(\\"MATERIALIZATION_VARIANT_CAP\\")\\n    domains = getattr(constructor, \\"OP_PARAMETER_DOMAINS\\", {}) if constructor else {}\\n    output, traces, seen = [], [], set()\\n    invalid_variants = 0\\n    candidates = {}\\n    for rid, rule in sorted(library[\\"rules\\"].items()):\\n        check()\\n        bindings = [copy.deepcopy(binding) for binding in rule[\\"observed_bindings\\"]]\\n        if expand_parameter_domains and rule[\\"parameters\\"]:\\n            for binding in rule[\\"observed_bindings\\"]:\\n                for spec in rule[\\"parameters\\"]:\\n                    check()\\n                    values = list(spec[\\"observed_values\\"])\\n                    location = spec[\\"locations\\"][0]\\n                    owner = _at(rule[\\"body\\"], location[\\"path\\"])\\n                    values += _external_domain(domains, owner[\\"op\\"], location[\\"key\\"])\\n                    for value in values:\\n                        if _value_kind(value) != spec[\\"kind\\"]:\\n                            continue\\n                        proposed = copy.deepcopy(binding)\\n                        proposed[spec[\\"name\\"]] = copy.deepcopy(value)\\n                        bindings.append(proposed)\\n        unique = {canonical(binding): binding for binding in bindings}\\n        observed_keys = [canonical(binding) for binding in rule[\\"observed_bindings\\"]]\\n        ordered = observed_keys + [key for key in sorted(unique) if key not in observed_keys]\\n        candidates[rid] = [(unique[key], key in observed_keys) for key in ordered]\\n    # Round robin over rules prevents the first rule\'s parameter domain from\\n    # consuming the entire shared variant cap.\\n    depth = 0\\n    while len(output) < max_variants:\\n        progressed = False\\n        for rid, choices in sorted(candidates.items()):\\n            check()\\n            if depth >= len(choices) or len(output) >= max_variants:\\n                continue\\n            progressed = True\\n            binding, observed = choices[depth]\\n            rule = library[\\"rules\\"][rid]\\n            ast = _instantiate(rule[\\"body\\"], rule[\\"parameters\\"], binding)\\n            _closed_ast(ast, 64)\\n            if constructor:\\n                try:\\n                    constructor.validate_ast(ast)\\n                except (ValueError, TypeError, KeyError):\\n                    # Equality-coupled slots can inhabit different enum domains.\\n                    # Their candidate values must satisfy the complete unchanged\\n                    # typed interpreter signature at every occurrence.\\n                    invalid_variants += 1\\n                    continue\\n            key = digest(ast)\\n            if key in seen:\\n                continue\\n            seen.add(key)\\n            output.append(ast)\\n            traces.append({\\"AST_sha256\\": key, \\"rule_id\\": rid,\\n                           \\"binding_sha256\\": digest(binding),\\n                           \\"observed_binding\\": observed,\\n                           \\"retained_verification_still_required\\": True})\\n        if not progressed:\\n            break\\n        depth += 1\\n    return {\\"expanded\\": output, \\"library_sha256\\": library[\\"library_sha256\\"],\\n            \\"materialization_receipts\\": traces, \\"variant_cap\\": max_variants,\\n            \\"invalid_domain_variants_skipped\\": invalid_variants,\\n            \\"query_values_received\\": False,\\n            \\"retained_demos_sha256\\": digest(train) if train is not None else None}\\n\\n\\ndef controls(library, discovery_records=None, check=lambda: None):\\n    \\"\\"\\"Return explicit no-growth and operation-order shuffle controls.\\n\\n    Reverse each unary rule\'s operation stack, carrying its parameters with its\\n    operator.  This preserves typing and parameter domains and often changes\\n    semantics.  It can leave commuting operations semantically equivalent, so\\n    syntactic changes alone are never claimed as a proven semantic derangement.\\n    A strong static control is constructed separately by static_matched().\\n    \\"\\"\\"\\n    validate_library(library)\\n    changed = 0\\n    shuffled = empty_library()\\n    new_rules = {}\\n    for _, original in sorted(library[\\"rules\\"].items()):\\n        check()\\n        rule = copy.deepcopy(original)\\n        stack, cursor = [], rule[\\"body\\"]\\n        while cursor[\\"op\\"] != \\"input\\":\\n            if len(cursor[\\"args\\"]) != 1:\\n                raise ValueError(\\"SHUFFLE_REQUIRES_UNARY_TYPED_RULE\\")\\n            item = copy.deepcopy(cursor)\\n            item[\\"args\\"] = []\\n            stack.append(item)\\n            cursor = cursor[\\"args\\"][0]\\n        reversed_body = copy.deepcopy(cursor)\\n        for item in stack:\\n            item[\\"args\\"] = [reversed_body]\\n            reversed_body = item\\n        if reversed_body != rule[\\"body\\"]:\\n            changed += 1\\n        rule[\\"body\\"] = reversed_body\\n        for specification in rule[\\"parameters\\"]:\\n            specification[\\"locations\\"] = []\\n        by_name = {p[\\"name\\"]: p for p in rule[\\"parameters\\"]}\\n        for path, node in _walk(reversed_body):\\n            for key, value in node[\\"params\\"].items():\\n                if type(value) is dict and set(value) == {\\"$param\\"}:\\n                    by_name[value[\\"$param\\"]][\\"locations\\"].append({\\"path\\": list(path), \\"key\\": key})\\n        rule.pop(\\"rule_sha256\\", None)\\n        # Controls never claim new admission proof: lineage belongs to the\\n        # authentic source rule, and the altered rule is only a search seed.\\n        rule[\\"control\\"] = {\\"kind\\": \\"unary_operation_order_reverse\\",\\n                           \\"original_rule_sha256\\": original[\\"rule_sha256\\"],\\n                           \\"independently_admitted\\": False}\\n        rule[\\"rule_sha256\\"] = digest(rule)\\n        new_rules[\\"r_\\" + rule[\\"rule_sha256\\"][:20]] = rule\\n    shuffled[\\"rules\\"] = new_rules\\n    shuffled[\\"control\\"] = {\\"kind\\": \\"unary_operation_order_reverse\\",\\n                           \\"semantic_change_proven\\": False,\\n                           \\"rules_with_changed_operation_order\\": changed,\\n                           \\"unchanged_control_is_not_causal_evidence\\": changed == 0}\\n    shuffled[\\"library_sha256\\"] = digest(_library_payload(shuffled))\\n    return {\\"unlearned\\": empty_library(), \\"blind\\": empty_library(),\\n            \\"shuffled\\": shuffled,\\n            \\"receipt\\": {\\"kind\\": \\"explicit_controls\\", \\"rules_changed\\": changed,\\n                        \\"blind_means_no_growth_not_blind_discovery\\": True,\\n                        \\"static_matched_requires_constructor_static_programs\\": True}}\\n\\n\\ndef static_matched(constructor, train, learned_materialization, caps=None,\\n                   check=lambda: None):\\n    \\"\\"\\"Strong deterministic static seeds under matched slot and AST-cost caps.\\n\\n    This path does not receive discovery records or library rule bodies.  Only\\n    the learned arm\'s number of seed slots and total expanded operator budget\\n    are public matching metadata.  static_programs must use retained demos and\\n    the unchanged shared primitive families, with all cost charged by runner.\\n    \\"\\"\\"\\n    _valid_train(train)\\n    target_costs = [_operator_count(ast) for ast in learned_materialization.get(\\"expanded\\", [])]\\n    target_count, target_cost = len(target_costs), sum(target_costs)\\n    target_histogram = dict(sorted(Counter(target_costs).items()))\\n    if not hasattr(constructor, \\"static_programs\\"):\\n        return {\\"expanded\\": [], \\"control\\": \\"UNAVAILABLE_NO_STATIC_API\\",\\n                \\"target_slots\\": target_count, \\"target_operator_cost\\": target_cost,\\n                \\"matched\\": False}\\n    # Roots come exclusively from the constructor\'s public, output-independent\\n    # static grammar.  Build all needed cost layers, rather than matching a\\n    # multi-operation learned seed with a cheaper one-operation baseline.\\n    programs = constructor.static_programs(train, caps=caps, check=check, max_programs=32)\\n    root_by_family = defaultdict(list)\\n    root_seen = set()\\n    for program in programs:\\n        _closed_ast(program, 64)\\n        constructor.validate_ast(program)\\n        if _operator_count(program) != 1 or len(program[\\"args\\"]) != 1:\\n            continue\\n        key = digest(program)\\n        if key not in root_seen:\\n            root_seen.add(key)\\n            root_by_family[program[\\"op\\"]].append(copy.deepcopy(program))\\n    roots = []\\n    # Preserve the public family\'s original appearance order, with one parameter\\n    # setting from each family before taking its next parameter setting.\\n    for depth in range(max((len(group) for group in root_by_family.values()), default=0)):\\n        for group in root_by_family.values():\\n            if depth < len(group):\\n                roots.append(group[depth])\\n    roots = roots[:32]\\n    pool, pool_seen = defaultdict(list), set()\\n    candidates_examined = 0\\n    candidate_cap = 8192\\n\\n    def add_sequence(indices):\\n        nonlocal candidates_examined\\n        check()\\n        candidates_examined += 1\\n        ast = {\\"op\\": \\"input\\", \\"type\\": \\"Grid\\", \\"args\\": [], \\"params\\": {}}\\n        for index in reversed(indices):\\n            root = copy.deepcopy(roots[index])\\n            root[\\"args\\"] = [ast]\\n            ast = root\\n        key = digest(ast)\\n        if key in pool_seen:\\n            return\\n        pool_seen.add(key)\\n        constructor.validate_ast(ast)\\n        pool[len(indices)].append(ast)\\n\\n    for amount, required in target_histogram.items():\\n        if amount == 0:\\n            pool[0] = [{\\"op\\": \\"input\\", \\"type\\": \\"Grid\\", \\"args\\": [], \\"params\\": {}}]\\n            continue\\n        if amount > 12 or not roots:\\n            continue\\n        n = len(roots)\\n        # Cyclic mixed-family chains are deliberately visited before a plain\\n        # Cartesian enumeration, which would spend its prefix repeating a single\\n        # outer family.  These chains are deterministic, with no discovered rule\\n        # body, retained-output comparison, score, query, or task ID involved.\\n        for stride in range(1, n + 1):\\n            for start in range(n):\\n                if len(pool[amount]) >= required or candidates_examined >= candidate_cap:\\n                    break\\n                add_sequence(tuple((start + stride * position) % n for position in range(amount)))\\n            if len(pool[amount]) >= required or candidates_examined >= candidate_cap:\\n                break\\n        if len(pool[amount]) < required and candidates_examined < candidate_cap:\\n            for indices in itertools.product(range(n), repeat=amount):\\n                if len(pool[amount]) >= required or candidates_examined >= candidate_cap:\\n                    break\\n                add_sequence(indices)\\n    result, offsets = [], Counter()\\n    for amount in target_costs:\\n        if offsets[amount] < len(pool[amount]):\\n            result.append(pool[amount][offsets[amount]])\\n            offsets[amount] += 1\\n    cost = sum(_operator_count(ast) for ast in result)\\n    actual_histogram = dict(sorted(Counter(_operator_count(ast) for ast in result).items()))\\n    def literal_bytes(asts):\\n        return sum(len(canonical(node[\\"params\\"]).encode()) for ast in asts for _, node in _walk(ast))\\n    return {\\"expanded\\": result, \\"control\\": \\"strong_static_matched_seeds\\",\\n            \\"target_slots\\": target_count, \\"target_operator_cost\\": target_cost,\\n            \\"actual_slots\\": len(result), \\"actual_operator_cost\\": cost,\\n            \\"target_primitive_cost_histogram\\": target_histogram,\\n            \\"actual_primitive_cost_histogram\\": actual_histogram,\\n            \\"matched\\": len(result) == target_count and actual_histogram == target_histogram,\\n            \\"public_roots\\": len(roots), \\"static_candidates_examined\\": candidates_examined,\\n            \\"static_candidate_cap\\": candidate_cap,\\n            \\"learned_parameter_literal_bytes\\": literal_bytes(learned_materialization.get(\\"expanded\\", [])),\\n            \\"static_parameter_literal_bytes\\": literal_bytes(result),\\n            \\"parameter_literal_costs_matched\\": False,\\n            \\"uses_discovery_programs_for_seed_construction\\": False,\\n            \\"uses_retained_outputs_for_seed_construction\\": False,\\n            \\"query_values_received\\": False,\\n            \\"causal_gain_claim_allowed_only_if_matched_and_scored\\": True}\\n\\n\\ndef attribute_program(program, materialization):\\n    \\"\\"\\"Explain exact learned-subtree reuse without observing any answer.\\n\\n    This is structural attribution only.  A separate, frozen-prediction scoring\\n    stage must establish new solves and a matched unlearned comparison must\\n    establish any increment attributable to growth.\\n    \\"\\"\\"\\n    _closed_ast(program, 64)\\n    by_hash = {row[\\"AST_sha256\\"]: row for row in materialization.get(\\"materialization_receipts\\", [])}\\n    matches = []\\n    for path, subtree in _walk(program):\\n        key = digest(subtree)\\n        if key in by_hash:\\n            matches.append({\\"path\\": list(path), \\"AST_sha256\\": key,\\n                            \\"rule_id\\": by_hash[key][\\"rule_id\\"],\\n                            \\"operators_outside_match\\": _operator_count(program) - _operator_count(subtree),\\n                            \\"binding_observed_in_discovery\\": by_hash[key][\\"observed_binding\\"]})\\n    return {\\"learned_expansion_used\\": bool(matches),\\n            \\"proper_composition_of_learned_expansion\\": any(row[\\"operators_outside_match\\"] > 0 for row in matches),\\n            \\"matches\\": matches, \\"structural_attribution_is_not_accuracy_gain\\": True,\\n            \\"query_values_received\\": False}\\n", "r29_cpu_package_20261101/vendor/experiments/independent_synthesis_20261029/round25/diagnosis_constructor/engine.py": "\\"\\"\\"CORA R25: finite demo-constrained typed program construction.\\n\\nNo task IDs, query grids, neural predictions, file I/O or held-out answers enter\\npropose(). Closed programs are independently replayed on every retained demo.\\nSearch failure returns UNKNOWN, never an assertion that no program exists.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport collections\\nimport copy\\nimport hashlib\\nimport heapq\\nimport itertools\\nimport json\\nimport math\\nimport time\\n\\nSCHEMA = \'CORA_R25_PROPOSAL_V1\'\\nACTIONS = (\'rebind\', \'change_representation\', \'revise_fragment\', \'compose\', \'library\', \'abstain\')\\nBACKGROUNDS = (\'zero\', \'mode\', \'border\')\\nD4 = (\'identity\', \'hflip\', \'vflip\', \'r90\', \'r180\', \'r270\', \'transpose\', \'antitranspose\')\\nOP_PARAMETER_DOMAINS = {\\n    \'input\': {},\\n    \'d4\': {\'mode\': D4},\\n    \'crop\': {\'background\': BACKGROUNDS},\\n    \'object\': {\'background\': BACKGROUNDS, \'connectivity\': (4, 8),\\n               \'color_mode\': (\'mono\', \'foreground\'),\\n               \'selector\': (\'largest\', \'smallest\', \'top\', \'bottom\', \'left\', \'right\'),\\n               \'output\': (\'crop\', \'canvas\')},\\n    \'recolor\': {\'mapping\': None},\\n    \'resize\': {\'mode\': (\'scale\', \'tile\', \'reduce\', \'untile\'),\\n               \'row_factor\': tuple(range(1, 6)), \'col_factor\': tuple(range(1, 6))},\\n    \'shift\': {\'background\': BACKGROUNDS, \'dr\': tuple(range(-29, 30)), \'dc\': tuple(range(-29, 30))},\\n    \'pack\': {\'background\': BACKGROUNDS, \'direction\': (\'left\', \'right\', \'up\', \'down\')},\\n    \'fill_bbox\': {\'background\': BACKGROUNDS, \'color\': (\'foreground_mode\',)},\\n    \'complete_symmetry\': {\'background\': BACKGROUNDS, \'axis\': (\'horizontal\', \'vertical\', \'both\')},\\n    \'fill_holes\': {\'background\': BACKGROUNDS, \'color\': (\'foreground_mode\',)},\\n}\\nDEFAULT_CAPS = {\'CPU_s\': 0.5, \'generated_states\': 512, \'semantic_classes\': 128,\\n                \'primitive_executions\': 10000, \'cell_work\': 2000000,\\n                \'selected_programs\': 4, \'max_depth\': 4, \'library_seeds\': 16}\\nEVIDENCE_FEATURES = (\'shape_error\', \'pixel_error\', \'palette_error\', \'ink_error\',\\n                     \'bbox_error\', \'component_error\', \'retained_error\', \'type_error\',\\n                     \'ambiguity\', \'search_usage\')\\n\\n\\nclass Failure(ValueError):\\n    \\"\\"\\"An executable primitive\'s explicit precondition failed.\\"\\"\\"\\n\\n\\nclass Budget(TimeoutError):\\n    pass\\n\\n\\ndef canonical(value):\\n    return json.dumps(value, sort_keys=True, separators=(\',\', \':\'), allow_nan=False)\\n\\n\\ndef digest(value):\\n    return hashlib.sha256(canonical(value).encode()).hexdigest()\\n\\n\\ndef valid_grid(grid):\\n    if type(grid) is not list or not 1 <= len(grid) <= 30:\\n        raise Failure(\'GRID_HEIGHT\')\\n    if type(grid[0]) is not list or not 1 <= len(grid[0]) <= 30:\\n        raise Failure(\'GRID_WIDTH\')\\n    width = len(grid[0])\\n    if any(type(row) is not list or len(row) != width or\\n           any(type(v) is not int or not 0 <= v <= 9 for v in row) for row in grid):\\n        raise Failure(\'GRID_CELLS\')\\n    return grid\\n\\n\\ndef valid_demos(train):\\n    if type(train) is not list or not 1 <= len(train) <= 10:\\n        raise Failure(\'TRAIN_SCHEMA\')\\n    for pair in train:\\n        if type(pair) is not dict or set(pair) != {\'input\', \'output\'}:\\n            raise Failure(\'TRAIN_FIELDS\')\\n        valid_grid(pair[\'input\'])\\n        valid_grid(pair[\'output\'])\\n\\n\\ndef inp():\\n    return {\'op\': \'input\', \'type\': \'Grid\', \'args\': [], \'params\': {}}\\n\\n\\ninput_ast = inp\\n\\n\\ndef node(op, child=None, **params):\\n    return {\'op\': op, \'type\': \'Grid\', \'args\': [] if child is None else [copy.deepcopy(child)],\\n            \'params\': copy.deepcopy(params)}\\n\\n\\ndef structural_cost(ast):\\n    \\"\\"\\"Explicit primitive count; every library seed is already expanded.\\"\\"\\"\\n    return (ast[\'op\'] != \'input\') + sum(structural_cost(c) for c in ast.get(\'args\', []))\\n\\n\\ndef lower_plain(ast, library=None):\\n    validate_ast(ast)\\n    return copy.deepcopy(ast)\\n\\n\\ndef validate_ast(ast, max_depth=12, _depth=0):\\n    if _depth > max_depth or type(ast) is not dict or set(ast) != {\'op\', \'type\', \'args\', \'params\'}:\\n        raise Failure(\'AST_SCHEMA_OR_DEPTH\')\\n    op = ast[\'op\']\\n    if ast[\'type\'] != \'Grid\' or op not in OP_PARAMETER_DOMAINS:\\n        raise Failure(\'AST_TYPE_OR_OPERATION\')\\n    args, params = ast[\'args\'], ast[\'params\']\\n    if type(args) is not list or len(args) != (0 if op == \'input\' else 1) or type(params) is not dict:\\n        raise Failure(\'AST_ARITY\')\\n    domains = OP_PARAMETER_DOMAINS[op]\\n    if set(params) != set(domains):\\n        raise Failure(\'AST_PARAMETER_FIELDS\')\\n    for key, domain in domains.items():\\n        value = params[key]\\n        if op == \'recolor\':\\n            if type(value) is not dict or not value or any(\\n                type(k) is not str or k not in tuple(str(c) for c in range(10)) or\\n                type(v) is not int or not 0 <= v <= 9 for k, v in value.items()\\n            ):\\n                raise Failure(\'AST_PALETTE\')\\n        elif not any(type(value) is type(admitted) and value == admitted for admitted in domain):\\n            raise Failure(\'AST_PARAMETER_DOMAIN\')\\n    for child in args:\\n        validate_ast(child, max_depth, _depth + 1)\\n\\n\\ndef _unique_mode(values):\\n    counts = collections.Counter(values)\\n    if not counts:\\n        raise Failure(\'EMPTY_ROLE\')\\n    maximum = max(counts.values())\\n    winners = [v for v, n in counts.items() if n == maximum]\\n    if len(winners) != 1:\\n        raise Failure(\'AMBIGUOUS_COLOR_ROLE\')\\n    return winners[0]\\n\\n\\ndef background(grid, role):\\n    if role == \'zero\':\\n        return 0\\n    if role == \'mode\':\\n        return _unique_mode(v for row in grid for v in row)\\n    if role == \'border\':\\n        return _unique_mode(v for r, row in enumerate(grid) for c, v in enumerate(row)\\n                            if r in (0, len(grid)-1) or c in (0, len(row)-1))\\n    raise Failure(\'UNKNOWN_BACKGROUND_ROLE\')\\n\\n\\ndef _points(grid, bg):\\n    return [(r, c) for r, row in enumerate(grid) for c, v in enumerate(row) if v != bg]\\n\\n\\ndef _bounds(points):\\n    if not points:\\n        raise Failure(\'EMPTY_FOREGROUND\')\\n    return min(r for r, _ in points), min(c for _, c in points), max(r for r, _ in points), max(c for _, c in points)\\n\\n\\ndef _components(grid, bg, connectivity=4, color_mode=\'foreground\', check=lambda: None):\\n    remaining = set(_points(grid, bg))\\n    offsets = ((-1, 0), (0, -1), (0, 1), (1, 0)) if connectivity == 4 else tuple(\\n        (r, c) for r in (-1, 0, 1) for c in (-1, 0, 1) if r or c)\\n    components = []\\n    while remaining:\\n        check()\\n        start = min(remaining)\\n        remaining.remove(start)\\n        component, todo = [start], [start]\\n        while todo:\\n            r, c = todo.pop()\\n            for dr, dc in offsets:\\n                q = r + dr, c + dc\\n                if q in remaining and (color_mode == \'foreground\' or grid[q[0]][q[1]] == grid[r][c]):\\n                    remaining.remove(q)\\n                    component.append(q)\\n                    todo.append(q)\\n        components.append(component)\\n    return components\\n\\n\\ndef _d4(grid, mode):\\n    if mode == \'identity\':\\n        return [row[:] for row in grid]\\n    if mode == \'hflip\':\\n        return [row[::-1] for row in grid]\\n    if mode == \'vflip\':\\n        return grid[::-1]\\n    if mode == \'r90\':\\n        return [list(row) for row in zip(*grid[::-1])]\\n    if mode == \'r180\':\\n        return [row[::-1] for row in grid[::-1]]\\n    if mode == \'r270\':\\n        return [list(row) for row in zip(*grid)][::-1]\\n    if mode == \'transpose\':\\n        return [list(row) for row in zip(*grid)]\\n    if mode == \'antitranspose\':\\n        return [list(row)[::-1] for row in zip(*grid)][::-1]\\n    raise Failure(\'D4_MODE\')\\n\\n\\ndef _primitive(op, grid, p, check):\\n    if op == \'d4\':\\n        return _d4(grid, p[\'mode\'])\\n    if op == \'recolor\':\\n        return [[p[\'mapping\'].get(str(v), v) for v in row] for row in grid]\\n    if op == \'resize\':\\n        a, b, mode = p[\'row_factor\'], p[\'col_factor\'], p[\'mode\']\\n        h, w = len(grid), len(grid[0])\\n        if mode in (\'scale\', \'tile\'):\\n            if h * a > 30 or w * b > 30:\\n                raise Failure(\'RESIZE_GRID_BOUND\')\\n            if mode == \'scale\':\\n                return [[v for v in row for _ in range(b)] for row in grid for _ in range(a)]\\n            return [row * b for _ in range(a) for row in grid]\\n        if h % a or w % b:\\n            raise Failure(\'REDUCE_DIVISIBILITY\')\\n        nh, nw = h // a, w // b\\n        if mode == \'reduce\':\\n            out = [[grid[r*a][c*b] for c in range(nw)] for r in range(nh)]\\n            if any(grid[r][c] != out[r//a][c//b] for r in range(h) for c in range(w)):\\n                raise Failure(\'NONCONSTANT_SCALE_BLOCK\')\\n            return out\\n        out = [row[:nw] for row in grid[:nh]]\\n        if any(grid[r][c] != out[r % nh][c % nw] for r in range(h) for c in range(w)):\\n            raise Failure(\'NONREPEATED_TILE\')\\n        return out\\n    bg = background(grid, p[\'background\'])\\n    points = _points(grid, bg)\\n    h, w = len(grid), len(grid[0])\\n    if op == \'crop\':\\n        r0, c0, r1, c1 = _bounds(points)\\n        return [row[c0:c1+1] for row in grid[r0:r1+1]]\\n    if op == \'object\':\\n        components = _components(grid, bg, p[\'connectivity\'], p[\'color_mode\'], check)\\n        if not components:\\n            raise Failure(\'NO_OBJECT\')\\n        def key(component):\\n            r0, c0, r1, c1 = _bounds(component)\\n            return {\'largest\': -len(component), \'smallest\': len(component),\\n                    \'top\': r0, \'bottom\': -r1, \'left\': c0, \'right\': -c1}[p[\'selector\']]\\n        best = min(key(component) for component in components)\\n        winners = [component for component in components if key(component) == best]\\n        if len(winners) != 1:\\n            raise Failure(\'AMBIGUOUS_OBJECT_ROLE\')\\n        chosen = winners[0]\\n        if p[\'output\'] == \'canvas\':\\n            out = [[bg] * w for _ in range(h)]\\n            for r, c in chosen:\\n                out[r][c] = grid[r][c]\\n            return out\\n        r0, c0, r1, c1 = _bounds(chosen)\\n        out = [[bg] * (c1-c0+1) for _ in range(r1-r0+1)]\\n        for r, c in chosen:\\n            out[r-r0][c-c0] = grid[r][c]\\n        return out\\n    if op == \'shift\':\\n        out = [[bg] * w for _ in range(h)]\\n        for r, c in points:\\n            nr, nc = r + p[\'dr\'], c + p[\'dc\']\\n            if not (0 <= nr < h and 0 <= nc < w):\\n                raise Failure(\'SHIFT_CLIPS_FOREGROUND\')\\n            out[nr][nc] = grid[r][c]\\n        return out\\n    if op == \'pack\':\\n        direction = p[\'direction\']\\n        if direction in (\'left\', \'right\'):\\n            out = []\\n            for row in grid:\\n                values = [v for v in row if v != bg]\\n                blanks = [bg] * (w-len(values))\\n                out.append(values + blanks if direction == \'left\' else blanks + values)\\n            return out\\n        transposed = [list(row) for row in zip(*grid)]\\n        packed = _primitive(\'pack\', transposed, {\'background\': \'zero\' if bg == 0 else p[\'background\'],\\n                              \'direction\': \'left\' if direction == \'up\' else \'right\'}, check)\\n        # A border role can change under a transpose only if border membership changes;\\n        # transpose preserves border membership, hence this re-evaluation is identical.\\n        return [list(row) for row in zip(*packed)]\\n    if op in (\'fill_bbox\', \'fill_holes\'):\\n        color = _unique_mode(grid[r][c] for r, c in points)\\n        out = [row[:] for row in grid]\\n        if op == \'fill_bbox\':\\n            r0, c0, r1, c1 = _bounds(points)\\n            for r in range(r0, r1+1):\\n                for c in range(c0, c1+1):\\n                    if out[r][c] == bg:\\n                        out[r][c] = color\\n            return out\\n        outside = {(r, c) for r, row in enumerate(grid) for c, v in enumerate(row)\\n                   if v == bg and (r in (0, h-1) or c in (0, w-1))}\\n        todo = list(outside)\\n        while todo:\\n            r, c = todo.pop()\\n            for dr, dc in ((-1, 0), (0, -1), (0, 1), (1, 0)):\\n                q = r+dr, c+dc\\n                if 0 <= q[0] < h and 0 <= q[1] < w and grid[q[0]][q[1]] == bg and q not in outside:\\n                    outside.add(q)\\n                    todo.append(q)\\n        for r, row in enumerate(out):\\n            for c, value in enumerate(row):\\n                if value == bg and (r, c) not in outside:\\n                    out[r][c] = color\\n        return out\\n    if op == \'complete_symmetry\':\\n        out = [row[:] for row in grid]\\n        modes = {\'horizontal\': (\'hflip\',), \'vertical\': (\'vflip\',), \'both\': (\'hflip\', \'vflip\', \'r180\')}[p[\'axis\']]\\n        for mode in modes:\\n            reflected = _d4(grid, mode)\\n            for r in range(h):\\n                for c in range(w):\\n                    value = reflected[r][c]\\n                    if value != bg:\\n                        if out[r][c] not in (bg, value):\\n                            raise Failure(\'SYMMETRY_COLOR_CONFLICT\')\\n                        out[r][c] = value\\n        return out\\n    raise Failure(\'UNKNOWN_PRIMITIVE\')\\n\\n\\ndef execute(program, raw, library=None, check=lambda: None, meter=None):\\n    \\"\\"\\"Typed pure interpreter. Library bodies must be expanded before this call.\\"\\"\\"\\n    validate_ast(program)\\n    valid_grid(raw)\\n    meter = {} if meter is None else meter\\n    meter.setdefault(\'primitive_executions\', 0)\\n    meter.setdefault(\'cell_work\', 0)\\n    def run(ast):\\n        check()\\n        if ast[\'op\'] == \'input\':\\n            return [row[:] for row in raw]\\n        grid = run(ast[\'args\'][0])\\n        check()\\n        if meter.get(\'primitive_limit\') is not None and meter[\'primitive_executions\'] >= meter[\'primitive_limit\']:\\n            raise Budget(\'PRIMITIVE_CAP_UNKNOWN\')\\n        # Every admitted primitive reads and emits at most 30x30 cells.\\n        if meter.get(\'cell_limit\') is not None and meter[\'cell_work\'] + 1800 > meter[\'cell_limit\']:\\n            raise Budget(\'CELL_WORK_CAP_UNKNOWN\')\\n        meter[\'primitive_executions\'] += 1\\n        out = _primitive(ast[\'op\'], grid, ast[\'params\'], check)\\n        valid_grid(out)\\n        meter[\'cell_work\'] += len(grid)*len(grid[0]) + len(out)*len(out[0])\\n        check()\\n        return out\\n    return run(program)\\n\\n\\ndef predict(program, query, library=None, check=lambda: None):\\n    return execute(program, query, library=library, check=check)\\n\\n\\ndef verify(program, train, check=lambda: None, meter=None):\\n    valid_demos(train)\\n    validate_ast(program)\\n    return all(execute(program, pair[\'input\'], check=check, meter=meter) == pair[\'output\'] for pair in train)\\n\\n\\ndef _shape(grid):\\n    return len(grid), len(grid[0])\\n\\n\\ndef _palette_map(outputs, train):\\n    mapping = {}\\n    for grid, pair in zip(outputs, train):\\n        if _shape(grid) != _shape(pair[\'output\']):\\n            return None\\n        for old_row, new_row in zip(grid, pair[\'output\']):\\n            for old, new in zip(old_row, new_row):\\n                if str(old) in mapping and mapping[str(old)] != new:\\n                    return None\\n                mapping[str(old)] = new\\n    return mapping\\n\\n\\ndef _infer_resizes(outputs, train):\\n    for mode in (\'scale\', \'tile\', \'reduce\', \'untile\'):\\n        factors = []\\n        for grid, pair in zip(outputs, train):\\n            h, w = _shape(grid)\\n            oh, ow = _shape(pair[\'output\'])\\n            big, small = ((oh, ow), (h, w)) if mode in (\'scale\', \'tile\') else ((h, w), (oh, ow))\\n            if big[0] % small[0] or big[1] % small[1]:\\n                break\\n            factors.append((big[0] // small[0], big[1] // small[1]))\\n        if len(factors) == len(train) and len(set(factors)) == 1:\\n            a, b = factors[0]\\n            if 1 <= a <= 5 and 1 <= b <= 5 and (a, b) != (1, 1):\\n                yield {\'mode\': mode, \'row_factor\': a, \'col_factor\': b}\\n\\n\\ndef _infer_shifts(outputs, train):\\n    for role in BACKGROUNDS:\\n        values = []\\n        try:\\n            for grid, pair in zip(outputs, train):\\n                target = pair[\'output\']\\n                if _shape(grid) != _shape(target):\\n                    break\\n                # Infer displacement from bounding boxes, then verify actual cell behavior.\\n                old = _bounds(_points(grid, background(grid, role)))\\n                new = _bounds(_points(target, background(target, role)))\\n                if (old[2]-old[0], old[3]-old[1]) != (new[2]-new[0], new[3]-new[1]):\\n                    break\\n                values.append((new[0]-old[0], new[1]-old[1]))\\n            if len(values) == len(train) and len(set(values)) == 1 and values[0] != (0, 0):\\n                dr, dc = values[0]\\n                yield {\'background\': role, \'dr\': dr, \'dc\': dc}\\n        except Failure:\\n            continue\\n\\n\\ndef _families(seed, action):\\n    def geometry():\\n        for mode in D4[1:]:\\n            yield node(\'d4\', seed, mode=mode)\\n    def crop():\\n        for role in BACKGROUNDS:\\n            yield node(\'crop\', seed, background=role)\\n    def objects():\\n        # Round-robin over selectors before changing segmentation, not a large object-root block.\\n        for output, selector, color_mode, connectivity, role in itertools.product(\\n                (\'crop\', \'canvas\'), (\'largest\', \'smallest\', \'top\', \'bottom\', \'left\', \'right\'),\\n                (\'foreground\', \'mono\'), (4, 8), BACKGROUNDS):\\n            yield node(\'object\', seed, background=role, connectivity=connectivity,\\n                       color_mode=color_mode, selector=selector, output=output)\\n    def pack():\\n        for direction, role in itertools.product((\'left\', \'right\', \'up\', \'down\'), BACKGROUNDS):\\n            yield node(\'pack\', seed, background=role, direction=direction)\\n    def bbox():\\n        for role in BACKGROUNDS:\\n            yield node(\'fill_bbox\', seed, background=role, color=\'foreground_mode\')\\n    def symmetry():\\n        for axis, role in itertools.product((\'horizontal\', \'vertical\', \'both\'), BACKGROUNDS):\\n            yield node(\'complete_symmetry\', seed, background=role, axis=axis)\\n    def holes():\\n        for role in BACKGROUNDS:\\n            yield node(\'fill_holes\', seed, background=role, color=\'foreground_mode\')\\n    definitions = [(\'identity\', iter([seed])), (\'geometry\', geometry()), (\'crop\', crop()),\\n                   (\'object\', objects()), (\'pack\', pack()), (\'bbox\', bbox()),\\n                   (\'symmetry\', symmetry()), (\'holes\', holes())]\\n    if action == \'rebind\':\\n        return [(\'identity\', iter([seed]))]\\n    if action == \'change_representation\':\\n        return [(name, it) for name, it in definitions if name in (\'crop\', \'object\')]\\n    if action == \'revise_fragment\':\\n        return [(name, it) for name, it in definitions if name in (\'geometry\', \'pack\', \'bbox\', \'symmetry\', \'holes\')]\\n    if action in (\'compose\', \'library\'):\\n        return [(\'identity\', iter([seed]))]\\n    return definitions\\n\\n\\ndef static_programs(train, caps=None, check=lambda: None, max_programs=16):\\n    \\"\\"\\"No demonstration outputs are used to form these matched static seed programs.\\"\\"\\"\\n    valid_demos(train)\\n    families, out = _families(inp(), None), []\\n    while families and len(out) < max_programs:\\n        next_round = []\\n        for name, iterator in families:\\n            check()\\n            try:\\n                out.append(next(iterator))\\n                next_round.append((name, iterator))\\n            except StopIteration:\\n                pass\\n            if len(out) >= max_programs:\\n                break\\n        families = next_round\\n    return out\\n\\n\\ndef _residual(outputs, train):\\n    shape_errors = sum(_shape(grid) != _shape(pair[\'output\']) for grid, pair in zip(outputs, train))\\n    pixels, total = 0, 0\\n    for grid, pair in zip(outputs, train):\\n        target = pair[\'output\']\\n        total += max(len(grid)*len(grid[0]), len(target)*len(target[0]))\\n        if _shape(grid) == _shape(target):\\n            pixels += sum(v != target[r][c] for r, row in enumerate(grid) for c, v in enumerate(row))\\n        else:\\n            pixels += max(len(grid)*len(grid[0]), len(target)*len(target[0]))\\n    return shape_errors, pixels / max(1, total)\\n\\n\\ndef _expanded_seeds(library, limit):\\n    if type(library) is not dict:\\n        raise Failure(\'LIBRARY_SCHEMA\')\\n    if not library:\\n        return []\\n    if \'expanded\' in library:\\n        seeds = library[\'expanded\']\\n        if type(seeds) is not list:\\n            raise Failure(\'LIBRARY_EXPANDED_SCHEMA\')\\n    else:\\n        seeds = list(library.values())\\n    out = []\\n    for ast in seeds[:limit]:\\n        validate_ast(ast)\\n        out.append(copy.deepcopy(ast))\\n    return out\\n\\n\\ndef propose(train, library=None, caps=None, check=lambda: None,\\n            diagnosis=\'authentic\', revision_action=None, seed_AST=None):\\n    \\"\\"\\"Construct closed programs only from retained demonstrations.\\n\\n    Diagnosis acts through measured residuals and actual bounded candidate branches.\\n    It does not select a task ID, intended task family or hidden answer. The blind\\n    control uses the same primitives and inferred parameters, with FIFO repair order.\\n    \\"\\"\\"\\n    valid_demos(train)\\n    if diagnosis not in (\'authentic\', \'blind\') or revision_action not in (None,) + ACTIONS:\\n        raise Failure(\'DIAGNOSIS_OR_ACTION\')\\n    limits = dict(DEFAULT_CAPS)\\n    if caps:\\n        caps = dict(caps)\\n        for alias, target in ((\'executions\', \'primitive_executions\'), (\'max_programs\', \'selected_programs\'), (\'generated\', \'generated_states\')):\\n            if alias in caps:\\n                if target in caps:\\n                    raise Failure(\'DUPLICATE_CAP_ALIAS\')\\n                caps[target] = caps.pop(alias)\\n        if any(k not in limits for k in caps):\\n            raise Failure(\'UNKNOWN_CAP\')\\n        limits.update(caps)\\n    if any((type(v) not in (int, float) if k == \'CPU_s\' else type(v) is not int) or not math.isfinite(v) or v <= 0 for k, v in limits.items()):\\n        raise Failure(\'CAP_VALUE\')\\n    start = time.process_time()\\n    meter = {\'primitive_executions\': 0, \'cell_work\': 0,\\n             \'primitive_limit\': limits[\'primitive_executions\'], \'cell_limit\': limits[\'cell_work\']}\\n    counts = collections.Counter()\\n    family_counts = {}\\n    selected, selected_records, seen_ast, seen_semantics = [], [], set(), {}\\n    evaluated_ast = set()\\n    pending, serial = [], 0\\n    best_error = 1.0\\n    seed = inp() if seed_AST is None else copy.deepcopy(seed_AST)\\n    validate_ast(seed, limits[\'max_depth\'])\\n    seeds = _expanded_seeds(library or {}, int(limits[\'library_seeds\']))\\n    status = \'COMPLETE_FINITE_SEARCH\'\\n    reason = None\\n\\n    def bounded():\\n        check()\\n        if time.process_time() - start >= limits[\'CPU_s\']:\\n            raise Budget(\'CPU_CAP_UNKNOWN\')\\n        if meter[\'primitive_executions\'] > limits[\'primitive_executions\']:\\n            raise Budget(\'PRIMITIVE_CAP_UNKNOWN\')\\n        if meter[\'cell_work\'] > limits[\'cell_work\']:\\n            raise Budget(\'CELL_WORK_CAP_UNKNOWN\')\\n\\n    def push(ast, family, priority):\\n        nonlocal serial\\n        if structural_cost(ast) > limits[\'max_depth\']:\\n            return\\n        key = digest(ast)\\n        if key in seen_ast:\\n            counts[\'syntactic_duplicates\'] += 1\\n            return\\n        if counts[\'generated_states\'] >= limits[\'generated_states\']:\\n            raise Budget(\'GENERATION_CAP_UNKNOWN\')\\n        seen_ast.add(key)\\n        counts[\'generated_states\'] += 1\\n        serial += 1\\n        heapq.heappush(pending, ((priority if diagnosis == \'authentic\' else (0,)), serial, ast, family))\\n\\n    def repairs(ast, outputs, family):\\n        shape_errors, pixel_error = _residual(outputs, train)\\n        # Palette fitting is a shared exact constraint solver, including static/library seeds.\\n        mapping = _palette_map(outputs, train)\\n        if mapping is not None:\\n            if any(int(k) != v for k, v in mapping.items()):\\n                push(node(\'recolor\', ast, mapping=mapping), \'rebind_palette\', (0, pixel_error, structural_cost(ast)))\\n        elif not shape_errors:\\n            counts[\'palette_inconsistent\'] += 1\\n        if revision_action == \'change_representation\':\\n            return\\n        for params in _infer_resizes(outputs, train):\\n            push(node(\'resize\', ast, **params), \'rebind_resize\', (1, pixel_error, structural_cost(ast)))\\n        for params in _infer_shifts(outputs, train):\\n            push(node(\'shift\', ast, **params), \'rebind_shift\', (1, pixel_error, structural_cost(ast)))\\n        if revision_action == \'rebind\':\\n            return\\n        if structural_cost(ast) < limits[\'max_depth\'] and ast[\'op\'] not in (\'d4\', \'recolor\'):\\n            # Shape constraints eliminate incompatible symmetry roots before execution.\\n            for mode in D4[1:]:\\n                swapped = mode in (\'r90\', \'r270\', \'transpose\', \'antitranspose\')\\n                if all((_shape(g)[::-1] if swapped else _shape(g)) == _shape(p[\'output\'])\\n                       for g, p in zip(outputs, train)):\\n                    push(node(\'d4\', ast, mode=mode), \'compose_geometry\', (2, pixel_error, structural_cost(ast)))\\n        if revision_action in (\'compose\', \'library\') and structural_cost(ast) <= 2:\\n            for role in BACKGROUNDS:\\n                push(node(\'crop\', ast, background=role), \'compose_crop\', (2, pixel_error, structural_cost(ast)))\\n\\n    def evaluate(ast, family):\\n        nonlocal best_error\\n        bounded()\\n        ast_key = digest(ast)\\n        if ast_key in evaluated_ast:\\n            counts[\'queued_already_executed\'] += 1\\n            return\\n        evaluated_ast.add(ast_key)\\n        stats = family_counts.setdefault(family, collections.Counter())\\n        stats[\'executed\'] += 1\\n        outputs = []\\n        try:\\n            for pair in train:\\n                outputs.append(execute(ast, pair[\'input\'], check=bounded, meter=meter))\\n        except Failure as error:\\n            counts[\'primitive_failures\'] += 1\\n            stats[\'precondition_failure\'] += 1\\n            if \'AMBIGUOUS\' in str(error):\\n                counts[\'ambiguity_failures\'] += 1\\n            return\\n        shape_errors, error = _residual(outputs, train)\\n        best_error = min(best_error, error)\\n        if not shape_errors and error == 0:\\n            # An exact closed witness is checked before any semantic pruning.\\n            # Replay is fresh and charged to the same primitive/cell/CPU meter.\\n            if not verify(ast, train, check=bounded, meter=meter):\\n                raise Failure(\'INDEPENDENT_REPLAY_DISAGREEMENT\')\\n            selected.append(copy.deepcopy(ast))\\n            selected_records.append({\'program_sha256\': digest(ast), \'structural_cost\': structural_cost(ast),\\n                                     \'retained_count\': len(train), \'exact_reverified\': True, \'family\': family})\\n            stats[\'exact_fit\'] += 1\\n            counts[\'exact_fit\'] += 1\\n            return\\n        signature = tuple(tuple(tuple(row) for row in grid) for grid in outputs)\\n        cost = structural_cost(ast)\\n        previous_cost = seen_semantics.get(signature)\\n        if previous_cost is not None and previous_cost <= cost:\\n            counts[\'semantic_duplicates\'] += 1\\n            stats[\'semantic_duplicate\'] += 1\\n            return\\n        if previous_cost is None:\\n            if counts[\'semantic_classes\'] >= limits[\'semantic_classes\']:\\n                raise Budget(\'SEMANTIC_CLASS_CAP_UNKNOWN\')\\n            counts[\'semantic_classes\'] += 1\\n            stats[\'new_semantic_class\'] += 1\\n        else:\\n            counts[\'cheaper_semantic_representative\'] += 1\\n        # Retain depth room when a cheaper representative arrives later.\\n        # Equal retained behavior is still not a proof of equal query behavior.\\n        seen_semantics[signature] = cost\\n        counts[\'shape_mismatch\'] += bool(shape_errors)\\n        counts[\'same_shape_residual\'] += bool(not shape_errors and error)\\n        repairs(ast, outputs, family)\\n\\n    try:\\n        if revision_action != \'abstain\':\\n            for ast in (seeds if revision_action in (None, \'library\') else []):\\n                push(ast, \'library_seed\', (-1, structural_cost(ast), 0))\\n            families = [] if revision_action == \'library\' else _families(seed, revision_action)\\n            # First complete round gives each primitive family a root validation.\\n            # Pending repairs are interleaved afterwards, so no root can monopolize search.\\n            round_number = 0\\n            while families or pending:\\n                bounded()\\n                next_round = []\\n                for family, iterator in families:\\n                    try:\\n                        ast = next(iterator)\\n                        next_round.append((family, iterator))\\n                        key = digest(ast)\\n                        if key not in seen_ast:\\n                            if counts[\'generated_states\'] >= limits[\'generated_states\']:\\n                                raise Budget(\'GENERATION_CAP_UNKNOWN\')\\n                            seen_ast.add(key)\\n                            counts[\'generated_states\'] += 1\\n                            evaluate(ast, family)\\n                        else:\\n                            counts[\'syntactic_duplicates\'] += 1\\n                            # A pending repair cannot suppress a family\'s fair root validation.\\n                            if key not in evaluated_ast:\\n                                evaluate(ast, family)\\n                    except StopIteration:\\n                        pass\\n                    if selected:\\n                        break\\n                families = next_round\\n                if selected:\\n                    status, reason = \'UNKNOWN\', \'EXACT_WITNESS_FOUND_SEARCH_INCOMPLETE\'\\n                    break\\n                if round_number:\\n                    for _ in range(3):\\n                        if not pending:\\n                            break\\n                        _, _, ast, family = heapq.heappop(pending)\\n                        evaluate(ast, family)\\n                        if selected:\\n                            break\\n                round_number += 1\\n                if selected:\\n                    status, reason = \'UNKNOWN\', \'EXACT_WITNESS_FOUND_SEARCH_INCOMPLETE\'\\n                    break\\n        else:\\n            reason = \'EXPLICIT_ABSTENTION\'\\n    except Budget as error:\\n        status, reason = \'UNKNOWN\', str(error)\\n    # This internal sample excludes receipt serialization; the outer protocol must\\n    # measure actual through-return CPU and reject any over-return witness.\\n    used = time.process_time() - start\\n    if used >= limits[\'CPU_s\']:\\n        status, reason = \'UNKNOWN\', \'CPU_CAP_UNKNOWN\'\\n        selected, selected_records = [], []\\n        counts[\'over_budget_witnesses_discarded\'] = 1\\n    requests = []\\n    if not selected:\\n        if counts[\'shape_mismatch\']:\\n            requests.append(\'representation_or_output_extent\')\\n        if counts[\'palette_inconsistent\']:\\n            requests.append(\'conditional_color_or_object_relation\')\\n        if counts[\'ambiguity_failures\']:\\n            requests.append(\'relational_object_disambiguation\')\\n        if counts[\'same_shape_residual\']:\\n            requests.append(\'additional_composition_or_local_rule\')\\n    diagnosis_record = {\'schema\': \'CORA_R25_DIAGNOSIS_V1\', \'mode\': diagnosis,\\n                        \'revision_action\': revision_action, \'status\': status, \'stop_reason\': reason,\\n                        \'counts\': dict(counts), \'families\': {k: dict(v) for k, v in family_counts.items()},\\n                        \'best_retained_pixel_error\': best_error, \'next_capability_requests\': requests,\\n                        \'meter\': meter, \'CPU_s\': used, \'caps\': limits}\\n    result = {\'schema\': SCHEMA, \'programs\': selected, \'program_records\': selected_records,\\n              \'diagnosis\': diagnosis_record}\\n    result[\'proposal_sha256\'] = digest(result)\\n    return result\\n\\n\\ndef get_failure_evidence(train, seed_AST=None, caps=None, check=lambda: None):\\n    \\"\\"\\"Observed retained execution residuals, normalized and shared across actions.\\n\\n    No intended-family label is an input. The caller can supervise a revision model\\n    using subsequently executed action outcomes; this function never trains itself.\\n    \\"\\"\\"\\n    valid_demos(train)\\n    seed = inp() if seed_AST is None else seed_AST\\n    accumulated = [0.0] * len(EVIDENCE_FEATURES)\\n    for pair in train:\\n        check()\\n        target = pair[\'output\']\\n        try:\\n            out = execute(seed, pair[\'input\'], check=check)\\n        except Failure:\\n            accumulated[7] += 1\\n            accumulated[6] += 1\\n            continue\\n        equal_shape = _shape(out) == _shape(target)\\n        accumulated[0] += not equal_shape\\n        accumulated[1] += _residual([out], [pair])[1]\\n        p, q = set(v for row in out for v in row), set(v for row in target for v in row)\\n        accumulated[2] += len(p ^ q) / max(1, len(p | q))\\n        bg_out = background(out, \'zero\')\\n        bg_target = background(target, \'zero\')\\n        points, desired = _points(out, bg_out), _points(target, bg_target)\\n        accumulated[3] += abs(len(points)-len(desired)) / max(1, len(out)*len(out[0]), len(target)*len(target[0]))\\n        try:\\n            a, b = _bounds(points), _bounds(desired)\\n            accumulated[4] += sum(abs(x-y) for x, y in zip(a, b)) / 116.0\\n        except Failure:\\n            accumulated[4] += bool(points) != bool(desired)\\n        old_components = _components(out, bg_out, check=check)\\n        new_components = _components(target, bg_target, check=check)\\n        accumulated[5] += abs(len(old_components)-len(new_components)) / max(1, len(old_components), len(new_components))\\n        accumulated[6] += out != target\\n        for grid in (out, target):\\n            count = collections.Counter(v for row in grid for v in row)\\n            maximum = max(count.values())\\n            accumulated[8] += (sum(n == maximum for n in count.values()) > 1) / 2.0\\n        accumulated[9] += min(1.0, structural_cost(seed) / float((caps or {}).get(\'max_depth\', DEFAULT_CAPS[\'max_depth\'])))\\n    values = [min(1.0, max(0.0, v/len(train))) for v in accumulated]\\n    return {\'schema\': \'CORA_R25_EVIDENCE_V1\', \'feature_names\': list(EVIDENCE_FEATURES),\\n            \'features\': values, \'retained_count\': len(train), \'seed_sha256\': digest(seed)}\\n", "r29_cpu_package_20261101/vendor/experiments/independent_synthesis_20261029/round26/coverage_constructor/engine.py": "\\"\\"\\"CORA R26: bounded executable grammar extension over frozen R25.\\n\\nOnly retained demonstrations enter construction. New mechanisms are modular\\nperiod completion, panel mask algebra, relational object alignment, and axis\\naggregation. An exact retained witness is evidence of fit, not generalization.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport collections\\nimport copy\\nimport hashlib\\nimport heapq\\nimport importlib.util\\nimport itertools\\nimport math\\nfrom pathlib import Path\\nimport time\\n\\n_BASE_PATH = Path(__file__).resolve().parents[2] / \'round25\' / \'diagnosis_constructor\' / \'engine.py\'\\nBASE_SHA256 = \'009788d7071929240a5231032eaeb3e783b1313e92aea14a9a81043dccb56559\'\\nif hashlib.sha256(_BASE_PATH.read_bytes()).hexdigest() != BASE_SHA256:\\n    raise RuntimeError(\'FROZEN_R25_DEPENDENCY_HASH\')\\n_spec = importlib.util.spec_from_file_location(\'_cora_r26_frozen_r25\', _BASE_PATH)\\nBASE = importlib.util.module_from_spec(_spec)\\n_spec.loader.exec_module(BASE)\\n\\nFailure, Budget = BASE.Failure, BASE.Budget\\ncanonical, digest = BASE.canonical, BASE.digest\\nvalid_grid, valid_demos = BASE.valid_grid, BASE.valid_demos\\nnode, inp, input_ast = BASE.node, BASE.inp, BASE.inp\\nstructural_cost = BASE.structural_cost\\nBACKGROUNDS, D4, ACTIONS = BASE.BACKGROUNDS, BASE.D4, BASE.ACTIONS\\nEVIDENCE_FEATURES = BASE.EVIDENCE_FEATURES\\nDEFAULT_CAPS = dict(BASE.DEFAULT_CAPS)\\nSCHEMA = \'CORA_R26_COVERAGE_PROPOSAL_V1\'\\nEXTENSION_DOMAINS = {\\n    \'periodic_complete\': {\'background\': BACKGROUNDS, \'row_period\': tuple(range(1, 16)),\\n                          \'col_period\': tuple(range(1, 16))},\\n    \'panel_boolean\': {\'background\': BACKGROUNDS, \'axis\': (\'rows\', \'columns\'), \'gap\': (0, 1),\\n                      \'combine\': (\'union\', \'intersection\', \'xor\', \'left_minus_right\', \'right_minus_left\'),\\n                      \'color\': tuple(range(10))},\\n    \'align_objects\': {\'background\': BACKGROUNDS, \'connectivity\': (4, 8),\\n                      \'color_mode\': (\'mono\', \'foreground\'), \'anchor\': (\'largest\', \'smallest\'),\\n                      \'edge\': (\'top\', \'bottom\', \'left\', \'right\')},\\n    \'axis_aggregate\': {\'background\': BACKGROUNDS, \'axis\': (\'rows\', \'columns\'),\\n                       \'statistic\': (\'foreground_mode\', \'foreground_count\', \'color_count\'),\\n                       \'color\': tuple(range(10))},\\n}\\nOP_PARAMETER_DOMAINS = dict(BASE.OP_PARAMETER_DOMAINS, **EXTENSION_DOMAINS)\\n\\n\\ndef validate_ast(ast, max_depth=12, _depth=0):\\n    if type(max_depth) is not int or max_depth < 0 or _depth > max_depth:\\n        raise Failure(\'AST_DEPTH\')\\n    if type(ast) is not dict or set(ast) != {\'op\', \'type\', \'args\', \'params\'}:\\n        raise Failure(\'AST_SCHEMA\')\\n    op, args, params = ast[\'op\'], ast[\'args\'], ast[\'params\']\\n    if ast[\'type\'] != \'Grid\' or type(op) is not str or op not in OP_PARAMETER_DOMAINS:\\n        raise Failure(\'AST_TYPE_OR_OPERATION\')\\n    if type(args) is not list or len(args) != (0 if op == \'input\' else 1) or type(params) is not dict:\\n        raise Failure(\'AST_ARITY\')\\n    domains = OP_PARAMETER_DOMAINS[op]\\n    if set(params) != set(domains):\\n        raise Failure(\'AST_PARAMETER_FIELDS\')\\n    for key, domain in domains.items():\\n        value = params[key]\\n        if op == \'recolor\':\\n            if type(value) is not dict or not value or any(\\n                    type(k) is not str or k not in tuple(str(c) for c in range(10)) or\\n                    type(v) is not int or not 0 <= v <= 9 for k, v in value.items()):\\n                raise Failure(\'AST_PALETTE\')\\n        elif not any(type(value) is type(item) and value == item for item in domain):\\n            raise Failure(\'AST_PARAMETER_DOMAIN\')\\n    for child in args:\\n        validate_ast(child, max_depth, _depth + 1)\\n\\n\\ndef lower_plain(ast, library=None):\\n    validate_ast(ast)\\n    return copy.deepcopy(ast)\\n\\n\\ndef _components(grid, bg, connectivity, color_mode, check):\\n    \\"\\"\\"Linear grid scan and flood fill; no quadratic repeated minimum selection.\\"\\"\\"\\n    h, w = len(grid), len(grid[0])\\n    seen, out = set(), []\\n    directions = ((-1, 0), (0, -1), (0, 1), (1, 0)) if connectivity == 4 else tuple(\\n        (dr, dc) for dr in (-1, 0, 1) for dc in (-1, 0, 1) if dr or dc)\\n    for r in range(h):\\n        check()\\n        for c in range(w):\\n            if grid[r][c] == bg or (r, c) in seen:\\n                continue\\n            seen.add((r, c))\\n            todo, component = [(r, c)], []\\n            while todo:\\n                check()\\n                x, y = todo.pop()\\n                component.append((x, y))\\n                for dr, dc in directions:\\n                    a, b = x + dr, y + dc\\n                    if (0 <= a < h and 0 <= b < w and (a, b) not in seen and\\n                            grid[a][b] != bg and\\n                            (color_mode == \'foreground\' or grid[a][b] == grid[x][y])):\\n                        seen.add((a, b))\\n                        todo.append((a, b))\\n            out.append(component)\\n    return out\\n\\n\\ndef _extension(op, grid, p, check):\\n    h, w = len(grid), len(grid[0])\\n    bg = BASE.background(grid, p[\'background\'])\\n    if op == \'periodic_complete\':\\n        rp, cp = p[\'row_period\'], p[\'col_period\']\\n        if rp > h or cp > w:\\n            raise Failure(\'PERIOD_EXCEEDS_GRID\')\\n        residues = {}\\n        for r, row in enumerate(grid):\\n            check()\\n            for c, value in enumerate(row):\\n                if value == bg:\\n                    continue\\n                key = r % rp, c % cp\\n                if key in residues and residues[key] != value:\\n                    raise Failure(\'PERIODIC_COLOR_CONFLICT\')\\n                residues[key] = value\\n        if not residues:\\n            raise Failure(\'NO_PERIODIC_EVIDENCE\')\\n        return [[residues.get((r % rp, c % cp), bg) for c in range(w)] for r in range(h)]\\n    if op == \'panel_boolean\':\\n        axis, gap = p[\'axis\'], p[\'gap\']\\n        size = h if axis == \'rows\' else w\\n        if size <= gap or (size-gap) % 2:\\n            raise Failure(\'PANEL_EQUAL_HALVES\')\\n        half = (size-gap) // 2\\n        if gap:\\n            separator = grid[half] if axis == \'rows\' else [row[half] for row in grid]\\n            if len(set(separator)) != 1:\\n                raise Failure(\'PANEL_SEPARATOR_NOT_CONSTANT\')\\n        if axis == \'rows\':\\n            left, right = grid[:half], grid[half+gap:]\\n        else:\\n            left = [row[:half] for row in grid]\\n            right = [row[half+gap:] for row in grid]\\n        out = []\\n        for arow, brow in zip(left, right):\\n            check()\\n            values = []\\n            for a, b in zip(arow, brow):\\n                x, y = a != bg, b != bg\\n                selected = {\'union\': x or y, \'intersection\': x and y, \'xor\': x != y,\\n                            \'left_minus_right\': x and not y,\\n                            \'right_minus_left\': y and not x}[p[\'combine\']]\\n                values.append(p[\'color\'] if selected else bg)\\n            out.append(values)\\n        return out\\n    if op == \'align_objects\':\\n        components = _components(grid, bg, p[\'connectivity\'], p[\'color_mode\'], check)\\n        if len(components) < 2:\\n            raise Failure(\'ALIGN_NEEDS_MULTIPLE_OBJECTS\')\\n        size = (max if p[\'anchor\'] == \'largest\' else min)(len(obj) for obj in components)\\n        anchors = [obj for obj in components if len(obj) == size]\\n        if len(anchors) != 1:\\n            raise Failure(\'AMBIGUOUS_OBJECT_ANCHOR\')\\n        anchor = anchors[0]\\n        edge_index = {\'top\': 0, \'left\': 1, \'bottom\': 2, \'right\': 3}[p[\'edge\']]\\n        target = BASE._bounds(anchor)[edge_index]\\n        out = [[bg] * w for _ in range(h)]\\n        occupied = set()\\n        for obj in components:\\n            check()\\n            delta = target - BASE._bounds(obj)[edge_index]\\n            dr, dc = (delta, 0) if p[\'edge\'] in (\'top\', \'bottom\') else (0, delta)\\n            for r, c in obj:\\n                nr, nc = r + dr, c + dc\\n                if not (0 <= nr < h and 0 <= nc < w):\\n                    raise Failure(\'ALIGN_CLIPS_OBJECT\')\\n                if (nr, nc) in occupied:\\n                    raise Failure(\'ALIGN_OBJECT_COLLISION\')\\n                occupied.add((nr, nc))\\n                out[nr][nc] = grid[r][c]\\n        return out\\n    if op == \'axis_aggregate\':\\n        rows = grid if p[\'axis\'] == \'rows\' else [list(column) for column in zip(*grid)]\\n        values = []\\n        for row in rows:\\n            check()\\n            foreground = [v for v in row if v != bg]\\n            if p[\'statistic\'] == \'foreground_mode\':\\n                value = BASE._unique_mode(foreground) if foreground else bg\\n            elif p[\'statistic\'] == \'foreground_count\':\\n                value = len(foreground)\\n            else:\\n                value = sum(v == p[\'color\'] for v in row)\\n            if not 0 <= value <= 9:\\n                raise Failure(\'AGGREGATE_COUNT_EXCEEDS_COLOR_DOMAIN\')\\n            values.append(value)\\n        return [[value] for value in values] if p[\'axis\'] == \'rows\' else [values]\\n    raise Failure(\'UNKNOWN_EXTENSION\')\\n\\n\\ndef execute(program, raw, library=None, check=lambda: None, meter=None):\\n    \\"\\"\\"Pure mixed-grammar interpreter with precharged conservative work units.\\n\\n    Every attempted non-input primitive is charged before execution, including\\n    explicit precondition failures. Original work units count maximum read/write\\n    cell capacity, as in R25; extensions also debit their extra scans. This is a\\n    finite work proxy, not instruction counting. CPU must be metered separately.\\n    External check exceptions propagate.\\n    \\"\\"\\"\\n    check()\\n    validate_ast(program)\\n    valid_grid(raw)\\n    meter = {} if meter is None else meter\\n    meter.setdefault(\'primitive_executions\', 0)\\n    meter.setdefault(\'cell_work\', 0)\\n    def run(ast):\\n        check()\\n        if ast[\'op\'] == \'input\':\\n            return [row[:] for row in raw]\\n        grid = run(ast[\'args\'][0])\\n        check()\\n        # Original operations use a fixed 1800-cell budget as R25 does. New\\n        # operations include scans/neighbor tests/output in their upper bound.\\n        n = len(grid)*len(grid[0])\\n        charge = (24*n + 1800 if ast[\'op\'] == \'align_objects\' else\\n                  8*n + 1800 if ast[\'op\'] in EXTENSION_DOMAINS else 1800)\\n        if meter.get(\'primitive_limit\') is not None and meter[\'primitive_executions\'] + 1 > meter[\'primitive_limit\']:\\n            raise Budget(\'PRIMITIVE_CAP_UNKNOWN\')\\n        if meter.get(\'cell_limit\') is not None and meter[\'cell_work\'] + charge > meter[\'cell_limit\']:\\n            raise Budget(\'CELL_WORK_CAP_UNKNOWN\')\\n        meter[\'primitive_executions\'] += 1\\n        meter[\'cell_work\'] += charge\\n        out = (_extension(ast[\'op\'], grid, ast[\'params\'], check) if ast[\'op\'] in EXTENSION_DOMAINS else\\n               BASE._primitive(ast[\'op\'], grid, ast[\'params\'], check))\\n        valid_grid(out)\\n        check()\\n        return out\\n    return run(program)\\n\\n\\ndef predict(program, query, library=None, check=lambda: None):\\n    return execute(program, query, library=library, check=check)\\n\\n\\ndef verify(program, train, check=lambda: None, meter=None):\\n    valid_demos(train)\\n    validate_ast(program)\\n    return all(execute(program, pair[\'input\'], check=check, meter=meter) == pair[\'output\'] for pair in train)\\n\\n\\ndef _extension_families(seed, outputs, train, check):\\n    \\"\\"\\"Finite candidate constraints from all visible demos; no query or IDs.\\"\\"\\"\\n    def periodic():\\n        if any(BASE._shape(g) != BASE._shape(p[\'output\']) for g, p in zip(outputs, train)):\\n            return\\n        targets = [p[\'output\'] for p in train]\\n        mh, mw = min(len(g) for g in targets), min(len(g[0]) for g in targets)\\n        for rp, cp in itertools.product(range(1, min(15, mh)+1), range(1, min(15, mw)+1)):\\n            check()\\n            if all(all(g[r][c] == g[r % rp][c % cp] for r in range(len(g)) for c in range(len(g[0])))\\n                   for g in targets):\\n                for role in BACKGROUNDS:\\n                    yield node(\'periodic_complete\', seed, background=role, row_period=rp, col_period=cp)\\n    def panel():\\n        for axis, gap, role in itertools.product((\'columns\', \'rows\'), (0, 1), BACKGROUNDS):\\n            check()\\n            if not all((len(g), (len(g[0])-gap)//2) == BASE._shape(p[\'output\']) and\\n                       len(g[0]) > gap and (len(g[0])-gap) % 2 == 0\\n                       if axis == \'columns\' else\\n                       ((len(g)-gap)//2, len(g[0])) == BASE._shape(p[\'output\']) and\\n                       len(g) > gap and (len(g)-gap) % 2 == 0 for g, p in zip(outputs, train)):\\n                continue\\n            colors = set(range(10))\\n            try:\\n                for grid, pair in zip(outputs, train):\\n                    bg = BASE.background(grid, role)\\n                    colors &= set(v for row in pair[\'output\'] for v in row if v != bg)\\n            except Failure:\\n                continue\\n            for combine, color in itertools.product(EXTENSION_DOMAINS[\'panel_boolean\'][\'combine\'], sorted(colors)):\\n                yield node(\'panel_boolean\', seed, background=role, axis=axis, gap=gap, combine=combine, color=color)\\n    def align():\\n        if any(BASE._shape(g) != BASE._shape(p[\'output\']) for g, p in zip(outputs, train)):\\n            return\\n        for edge, anchor, role, cm, conn in itertools.product(\\n                (\'top\', \'bottom\', \'left\', \'right\'), (\'largest\', \'smallest\'), BACKGROUNDS,\\n                (\'foreground\', \'mono\'), (4, 8)):\\n            yield node(\'align_objects\', seed, background=role, connectivity=conn,\\n                       color_mode=cm, anchor=anchor, edge=edge)\\n    def aggregate():\\n        for axis, statistic, role in itertools.product((\'rows\', \'columns\'),\\n                EXTENSION_DOMAINS[\'axis_aggregate\'][\'statistic\'], BACKGROUNDS):\\n            if not all((len(g), 1) == BASE._shape(p[\'output\']) if axis == \'rows\' else\\n                       (1, len(g[0])) == BASE._shape(p[\'output\']) for g, p in zip(outputs, train)):\\n                continue\\n            for color in (range(10) if statistic == \'color_count\' else (0,)):\\n                yield node(\'axis_aggregate\', seed, background=role, axis=axis, statistic=statistic, color=color)\\n    return [(\'periodic\', periodic()), (\'panel\', panel()), (\'alignment\', align()), (\'aggregate\', aggregate())]\\n\\n\\ndef _limits(caps):\\n    values = dict(DEFAULT_CAPS)\\n    supplied = dict(caps or {})\\n    for alias, target in ((\'executions\', \'primitive_executions\'), (\'max_programs\', \'selected_programs\'),\\n                          (\'generated\', \'generated_states\')):\\n        if alias in supplied:\\n            if target in supplied:\\n                raise Failure(\'DUPLICATE_CAP_ALIAS\')\\n            supplied[target] = supplied.pop(alias)\\n    if any(k not in values for k in supplied):\\n        raise Failure(\'UNKNOWN_CAP\')\\n    values.update(supplied)\\n    if any((type(v) not in (int, float) if k == \'CPU_s\' else type(v) is not int) or\\n           not math.isfinite(v) or v <= 0 for k, v in values.items()):\\n        raise Failure(\'CAP_VALUE\')\\n    return values\\n\\n\\ndef static_programs(train, caps=None, check=lambda: None, max_programs=16):\\n    \\"\\"\\"R25 input-only static seed API; no learned/extension bodies supplied.\\"\\"\\"\\n    return BASE.static_programs(train, caps=caps, check=check, max_programs=max_programs)\\n\\n\\ndef propose(train, library=None, caps=None, check=lambda: None,\\n            diagnosis=\'authentic\', revision_action=None, seed_AST=None):\\n    start = time.process_time()\\n    check()\\n    valid_demos(train)\\n    if diagnosis not in (\'authentic\', \'blind\') or revision_action not in (None,) + ACTIONS:\\n        raise Failure(\'DIAGNOSIS_OR_ACTION\')\\n    limits = _limits(caps)\\n    meter = {\'primitive_executions\': 0, \'cell_work\': 0,\\n             \'primitive_limit\': limits[\'primitive_executions\'], \'cell_limit\': limits[\'cell_work\']}\\n    counts, family_counts = collections.Counter(), {}\\n    selected, records, seen, evaluated, semantics = [], [], set(), set(), {}\\n    pending, serial, best_error = [], 0, 1.0\\n    external_cancellation = []\\n    status, reason = \'COMPLETE_FINITE_SEARCH\', None\\n    seed = inp() if seed_AST is None else copy.deepcopy(seed_AST)\\n    validate_ast(seed, limits[\'max_depth\'])\\n    raw_library = library or {}\\n    if type(raw_library) is not dict:\\n        raise Failure(\'LIBRARY_SCHEMA\')\\n    seeds = raw_library.get(\'expanded\', list(raw_library.values())) if raw_library else []\\n    if type(seeds) is not list:\\n        raise Failure(\'LIBRARY_EXPANDED_SCHEMA\')\\n    seeds = [copy.deepcopy(ast) for ast in seeds[:limits[\'library_seeds\']]]\\n    for ast in seeds:\\n        validate_ast(ast, limits[\'max_depth\'])\\n\\n    def bounded():\\n        try:\\n            check()\\n        except BaseException as error:\\n            # A caller may itself use the exported Budget type for cancellation.\\n            # Preserve that exception rather than treating it as internal UNKNOWN.\\n            external_cancellation.append(error)\\n            raise\\n        if time.process_time() - start >= limits[\'CPU_s\']:\\n            raise Budget(\'CPU_CAP_UNKNOWN\')\\n\\n    def queue(ast, family, priority):\\n        nonlocal serial\\n        if structural_cost(ast) > limits[\'max_depth\']:\\n            return\\n        key = digest(ast)\\n        if key in seen:\\n            counts[\'syntactic_duplicates\'] += 1\\n            return\\n        if counts[\'generated_states\'] >= limits[\'generated_states\']:\\n            raise Budget(\'GENERATION_CAP_UNKNOWN\')\\n        seen.add(key)\\n        counts[\'generated_states\'] += 1\\n        serial += 1\\n        heapq.heappush(pending, ((priority if diagnosis == \'authentic\' else (0,)), serial, ast, family))\\n\\n    def repairs(ast, outputs):\\n        shape_error, error = BASE._residual(outputs, train)\\n        mapping = BASE._palette_map(outputs, train)\\n        if mapping is not None and any(int(k) != v for k, v in mapping.items()):\\n            queue(node(\'recolor\', ast, mapping=mapping), \'rebind_palette\', (0, error, structural_cost(ast)))\\n        elif mapping is None and not shape_error:\\n            counts[\'palette_inconsistent\'] += 1\\n        for params in BASE._infer_resizes(outputs, train):\\n            queue(node(\'resize\', ast, **params), \'rebind_resize\', (1, error, structural_cost(ast)))\\n        for params in BASE._infer_shifts(outputs, train):\\n            queue(node(\'shift\', ast, **params), \'rebind_shift\', (1, error, structural_cost(ast)))\\n        if structural_cost(ast) < limits[\'max_depth\'] and ast[\'op\'] not in (\'d4\', \'recolor\'):\\n            for mode in D4[1:]:\\n                swapped = mode in (\'r90\', \'r270\', \'transpose\', \'antitranspose\')\\n                if all((BASE._shape(g)[::-1] if swapped else BASE._shape(g)) == BASE._shape(p[\'output\'])\\n                       for g, p in zip(outputs, train)):\\n                    queue(node(\'d4\', ast, mode=mode), \'compose_geometry\', (2, error, structural_cost(ast)))\\n        if revision_action in (\'compose\', \'library\') and structural_cost(ast) <= 2:\\n            for role in BACKGROUNDS:\\n                queue(node(\'crop\', ast, background=role), \'compose_crop\', (2, error, structural_cost(ast)))\\n\\n    def evaluate(ast, family):\\n        nonlocal best_error\\n        bounded()\\n        key = digest(ast)\\n        if key in evaluated:\\n            return\\n        evaluated.add(key)\\n        stats = family_counts.setdefault(family, collections.Counter())\\n        stats[\'executed\'] += 1\\n        try:\\n            outputs = [execute(ast, pair[\'input\'], check=bounded, meter=meter) for pair in train]\\n        except Failure as error:\\n            if external_cancellation and external_cancellation[-1] is error:\\n                raise\\n            stats[\'precondition_failure\'] += 1\\n            counts[\'primitive_failures\'] += 1\\n            counts[\'ambiguity_failures\'] += \'AMBIGUOUS\' in str(error)\\n            return\\n        shape_errors, error = BASE._residual(outputs, train)\\n        best_error = min(best_error, error)\\n        if not shape_errors and error == 0:\\n            if not verify(ast, train, check=bounded, meter=meter):\\n                raise Failure(\'INDEPENDENT_REPLAY_DISAGREEMENT\')\\n            selected.append(copy.deepcopy(ast))\\n            records.append({\'program_sha256\': digest(ast), \'structural_cost\': structural_cost(ast),\\n                            \'retained_count\': len(train), \'exact_reverified\': True, \'family\': family})\\n            stats[\'exact_fit\'] += 1\\n            counts[\'exact_fit\'] += 1\\n            return\\n        signature = canonical(outputs)\\n        cost, previous = structural_cost(ast), semantics.get(signature)\\n        if previous is not None and previous <= cost:\\n            counts[\'semantic_duplicates\'] += 1\\n            return\\n        if previous is None:\\n            if counts[\'semantic_classes\'] >= limits[\'semantic_classes\']:\\n                raise Budget(\'SEMANTIC_CLASS_CAP_UNKNOWN\')\\n            counts[\'semantic_classes\'] += 1\\n        semantics[signature] = cost\\n        counts[\'shape_mismatch\'] += bool(shape_errors)\\n        counts[\'same_shape_residual\'] += bool(not shape_errors and error)\\n        repairs(ast, outputs)\\n\\n    try:\\n        if revision_action != \'abstain\':\\n            # Every original and new family gets one root turn per round. Guidance\\n            # changes ordering but cannot erase the ordinary grammar coverage floor.\\n            families = BASE._families(seed, None)\\n            seed_outputs = [execute(seed, pair[\'input\'], check=bounded, meter=meter) for pair in train]\\n            families += _extension_families(seed, seed_outputs, train, bounded)\\n            if seeds:\\n                families.append((\'library_seed\', iter(seeds)))\\n            preferred = {\'rebind\': (\'identity\',), \'change_representation\': (\'crop\', \'object\', \'panel\', \'aggregate\'),\\n                         \'revise_fragment\': (\'periodic\', \'alignment\', \'geometry\', \'pack\', \'bbox\', \'symmetry\', \'holes\'),\\n                         \'compose\': (\'identity\', \'periodic\', \'panel\', \'alignment\', \'aggregate\'),\\n                         \'library\': (\'library_seed\',)}.get(revision_action, ())\\n            if diagnosis == \'authentic\' and preferred:\\n                families.sort(key=lambda item: item[0] not in preferred)\\n            round_number = 0\\n            while families or pending:\\n                bounded()\\n                next_round = []\\n                for family, iterator in families:\\n                    bounded()\\n                    try:\\n                        ast = next(iterator)\\n                    except StopIteration:\\n                        continue\\n                    next_round.append((family, iterator))\\n                    key = digest(ast)\\n                    if key not in seen:\\n                        if counts[\'generated_states\'] >= limits[\'generated_states\']:\\n                            raise Budget(\'GENERATION_CAP_UNKNOWN\')\\n                        seen.add(key)\\n                        counts[\'generated_states\'] += 1\\n                    evaluate(ast, family)\\n                    if selected:\\n                        break\\n                families = next_round\\n                if not selected and round_number:\\n                    for _ in range(3):\\n                        if not pending:\\n                            break\\n                        _, _, ast, family = heapq.heappop(pending)\\n                        evaluate(ast, family)\\n                        if selected:\\n                            break\\n                round_number += 1\\n                if selected:\\n                    status, reason = \'UNKNOWN\', \'EXACT_WITNESS_FOUND_SEARCH_INCOMPLETE\'\\n                    break\\n        else:\\n            reason = \'EXPLICIT_ABSTENTION\'\\n    except Budget as error:\\n        if external_cancellation and external_cancellation[-1] is error:\\n            raise\\n        status, reason = \'UNKNOWN\', str(error)\\n    used = time.process_time() - start\\n    if used >= limits[\'CPU_s\']:\\n        status, reason = \'UNKNOWN\', \'CPU_CAP_UNKNOWN\'\\n        selected, records = [], []\\n        counts[\'over_budget_witnesses_discarded\'] += 1\\n    requests = []\\n    if not selected:\\n        if counts[\'shape_mismatch\']:\\n            requests.append(\'representation_or_output_extent\')\\n        if counts[\'palette_inconsistent\']:\\n            requests.append(\'conditional_color_or_object_relation\')\\n        if counts[\'ambiguity_failures\']:\\n            requests.append(\'relational_object_disambiguation\')\\n        if counts[\'same_shape_residual\']:\\n            requests.append(\'additional_composition_or_local_rule\')\\n    receipt = {\'schema\': \'CORA_R26_DIAGNOSIS_V1\', \'mode\': diagnosis, \'revision_action\': revision_action,\\n               \'guidance_preserves_static_family_floor\': True, \'status\': status, \'stop_reason\': reason,\\n               \'floor_semantics\': \'interleaved_root_turns_no_equal_complete_coverage_guarantee_under_caps\',\\n               \'counts\': dict(counts), \'families\': {k: dict(v) for k, v in family_counts.items()},\\n               \'best_retained_pixel_error\': best_error, \'next_capability_requests\': requests,\\n               \'meter\': meter, \'work_unit_semantics\': \'precharged_cell_capacity_and_extension_scan_proxy\',\\n               \'CPU_s\': used, \'caps\': limits, \'frozen_base_sha256\': BASE_SHA256}\\n    result = {\'schema\': SCHEMA, \'programs\': selected, \'program_records\': records, \'diagnosis\': receipt}\\n    result[\'proposal_sha256\'] = digest(result)\\n    return result\\n\\n\\ndef get_failure_evidence(train, seed_AST=None, caps=None, check=lambda: None):\\n    \\"\\"\\"R25\'s ten generic residual features, using the new typed interpreter.\\"\\"\\"\\n    valid_demos(train)\\n    seed = inp() if seed_AST is None else seed_AST\\n    accumulated = [0.0] * len(EVIDENCE_FEATURES)\\n    external_cancellation = []\\n    def caller_check():\\n        try:\\n            check()\\n        except BaseException as error:\\n            external_cancellation.append(error)\\n            raise\\n    for pair in train:\\n        caller_check()\\n        target = pair[\'output\']\\n        try:\\n            out = execute(seed, pair[\'input\'], check=caller_check)\\n        except Failure as error:\\n            if external_cancellation and external_cancellation[-1] is error:\\n                raise\\n            accumulated[6] += 1\\n            accumulated[7] += 1\\n            continue\\n        accumulated[0] += BASE._shape(out) != BASE._shape(target)\\n        accumulated[1] += BASE._residual([out], [pair])[1]\\n        a, b = {v for row in out for v in row}, {v for row in target for v in row}\\n        accumulated[2] += len(a ^ b) / max(1, len(a | b))\\n        points, desired = BASE._points(out, 0), BASE._points(target, 0)\\n        accumulated[3] += abs(len(points)-len(desired)) / max(1, len(out)*len(out[0]), len(target)*len(target[0]))\\n        try:\\n            accumulated[4] += sum(abs(x-y) for x, y in zip(BASE._bounds(points), BASE._bounds(desired))) / 116.0\\n        except Failure:\\n            accumulated[4] += bool(points) != bool(desired)\\n        old = _components(out, 0, 4, \'foreground\', caller_check)\\n        new = _components(target, 0, 4, \'foreground\', caller_check)\\n        accumulated[5] += abs(len(old)-len(new)) / max(1, len(old), len(new))\\n        accumulated[6] += out != target\\n        for grid in (out, target):\\n            count = collections.Counter(v for row in grid for v in row)\\n            accumulated[8] += (sum(n == max(count.values()) for n in count.values()) > 1) / 2.0\\n        accumulated[9] += min(1.0, structural_cost(seed) / float((caps or {}).get(\'max_depth\', DEFAULT_CAPS[\'max_depth\'])))\\n    return {\'schema\': \'CORA_R26_EVIDENCE_V1\', \'feature_names\': list(EVIDENCE_FEATURES),\\n            \'features\': [min(1.0, max(0.0, value / len(train))) for value in accumulated],\\n            \'retained_count\': len(train), \'seed_sha256\': digest(seed)}\\n", "r29_cpu_package_20261101/vendor/experiments/independent_synthesis_20261029/round27/learned_local_rules/model.py": "\\"\\"\\"A bounded learned conditional Color program over a fixed pixel feature DSL.\\n\\nThe constructor learns a serializable decision tree from retained grid pairs.\\nNo sklearn, task IDs, absolute row/column keys, grid lookup table, generated\\nPython, query inputs or answers enter training. The language is engineered;\\nlearning a new conditional program is not invention of new primitive semantics.\\nConceptual predecessors in the user\'s read-only Reasoning_Project code are\\nconcept_grammar.py (SHA256 0a9ad226fc00c2edaf8f709b652ce79768e99a137f155feb05009b80f0f1fb8e)\\nand composable_reasoner.py (bf99d4bbaeeaa05ef29b7e83ae0dada1cdd20b7fc4dd867f17e53d832d166210).\\nThis implementation is independently authored and imports neither file.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport collections\\nimport copy\\nimport hashlib\\nimport json\\nimport math\\nimport time\\n\\nFEATURES = (\'center\', \'north\', \'south\', \'west\', \'east\', \'row_foreground_count\',\\n            \'column_foreground_count\', \'border_distance\', \'component_area\',\\n            \'component_height\', \'component_width\')\\nNUMERIC = frozenset(FEATURES[5:])\\nDEFAULT_CAPS = {\'CPU_s\': 2.0, \'max_depth\': 4, \'max_nodes\': 31,\\n                \'min_support\': 2, \'min_demos\': 2, \'max_samples\': 9000,\\n                \'predicate_work\': 2000000, \'feature_cell_work\': 2000000}\\n\\n\\nclass Failure(ValueError):\\n    pass\\n\\n\\nclass Abstention(ValueError):\\n    pass\\n\\n\\nclass _Limit(TimeoutError):\\n    pass\\n\\n\\ndef canonical(value):\\n    return json.dumps(value, sort_keys=True, separators=(\',\', \':\'), allow_nan=False)\\n\\n\\ndef digest(value):\\n    return hashlib.sha256(canonical(value).encode()).hexdigest()\\n\\n\\ndef valid_grid(grid):\\n    if (type(grid) is not list or not 1 <= len(grid) <= 30 or type(grid[0]) is not list\\n            or not 1 <= len(grid[0]) <= 30 or any(type(row) is not list or len(row) != len(grid[0])\\n                or any(type(value) is not int or not 0 <= value <= 9 for value in row) for row in grid)):\\n        raise Failure(\'GRID_SCHEMA\')\\n\\n\\ndef _limits(caps):\\n    result = dict(DEFAULT_CAPS)\\n    supplied = dict(caps or {})\\n    if any(key not in result for key in supplied):\\n        raise Failure(\'UNKNOWN_CAP\')\\n    result.update(supplied)\\n    for key, value in result.items():\\n        good_type = type(value) in (int, float) if key == \'CPU_s\' else type(value) is int\\n        if not good_type or not math.isfinite(value) or value <= 0:\\n            raise Failure(\'CAP_VALUE\')\\n    if result[\'max_depth\'] > 4 or result[\'max_nodes\'] > 31 or result[\'min_support\'] < 2 or result[\'min_demos\'] < 2:\\n        raise Failure(\'DECLARED_LANGUAGE_OR_SUPPORT_BOUND\')\\n    return result\\n\\n\\nclass _Budget:\\n    def __init__(self, limits, check):\\n        self.limits, self.check, self.started = limits, check, time.process_time()\\n        self.used = {\'predicate_work\': 0, \'feature_cell_work\': 0}\\n        self.external = []\\n\\n    def guard(self):\\n        try:\\n            self.check()\\n        except BaseException as error:\\n            self.external.append(error)\\n            raise\\n        if time.process_time() - self.started >= self.limits[\'CPU_s\']:\\n            raise _Limit(\'CPU_CAP_UNKNOWN\')\\n\\n    def spend(self, name, value):\\n        self.guard()\\n        if self.used[name] + value > self.limits[name]:\\n            raise _Limit(name.upper() + \'_CAP_UNKNOWN\')\\n        self.used[name] += value\\n\\n\\ndef _features(grid, budget):\\n    valid_grid(grid)\\n    h, w = len(grid), len(grid[0])\\n    budget.spend(\'feature_cell_work\', 64 * h * w)\\n    area, height, width = {}, {}, {}\\n    seen = set()\\n    for r in range(h):\\n        budget.guard()\\n        for c in range(w):\\n            if not grid[r][c] or (r, c) in seen:\\n                continue\\n            todo, component = [(r, c)], []\\n            seen.add((r, c))\\n            while todo:\\n                budget.guard()\\n                point = todo.pop()\\n                component.append(point)\\n                for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\\n                    nr, nc = point[0] + dr, point[1] + dc\\n                    if 0 <= nr < h and 0 <= nc < w and grid[nr][nc] and (nr, nc) not in seen:\\n                        seen.add((nr, nc)); todo.append((nr, nc))\\n            bh = max(p[0] for p in component) - min(p[0] for p in component) + 1\\n            bw = max(p[1] for p in component) - min(p[1] for p in component) + 1\\n            for point in component:\\n                area[point], height[point], width[point] = len(component), bh, bw\\n    rows = [sum(value != 0 for value in row) for row in grid]\\n    cols = [sum(grid[r][c] != 0 for r in range(h)) for c in range(w)]\\n    result = []\\n    for r in range(h):\\n        budget.guard()\\n        for c in range(w):\\n            neighbors = [grid[r + dr][c + dc] if 0 <= r + dr < h and 0 <= c + dc < w else 10\\n                         for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1))]\\n            result.append((grid[r][c], *neighbors, rows[r], cols[c],\\n                           min(r, c, h - 1 - r, w - 1 - c), area.get((r, c), 0),\\n                           height.get((r, c), 0), width.get((r, c), 0)))\\n    return result\\n\\n\\ndef _emissions(colors):\\n    return ([{\'kind\': \'feature\', \'feature\': feature} for feature in FEATURES[:5]]\\n            + [{\'kind\': \'constant\', \'color\': color} for color in sorted(colors)])\\n\\n\\ndef _emit(spec, feature):\\n    return feature[FEATURES.index(spec[\'feature\'])] if spec[\'kind\'] == \'feature\' else spec[\'color\']\\n\\n\\ndef _fit_emit(indices, examples, emissions, budget):\\n    best = None\\n    for order, spec in enumerate(emissions):\\n        budget.spend(\'predicate_work\', len(indices))\\n        errors = sum(_emit(spec, examples[i][0]) != examples[i][1] for i in indices)\\n        # Copy/neighborhood roles cost less than a literal color; stable ties\\n        # favor copy, making unchanged behavior a compact executable rule.\\n        key = (errors, 1 if spec[\'kind\'] == \'feature\' else 2, order)\\n        if best is None or key < best[0]:\\n            best = (key, spec)\\n    return best[0][0], copy.deepcopy(best[1])\\n\\n\\ndef _support(indices, examples, limits):\\n    return (len(indices) >= limits[\'min_support\'] and\\n            len({examples[i][2] for i in indices}) >= limits[\'min_demos\'])\\n\\n\\ndef _construct(indices, examples, emissions, budget, depth, state):\\n    budget.guard()\\n    if state[\'nodes\'] >= budget.limits[\'max_nodes\']:\\n        raise _Limit(\'NODE_CAP_UNKNOWN\')\\n    state[\'nodes\'] += 1\\n    if not _support(indices, examples, budget.limits):\\n        return None\\n    error, emission = _fit_emit(indices, examples, emissions, budget)\\n    info = {\'support\': len(indices), \'support_demos\': len({examples[i][2] for i in indices})}\\n    if error == 0:\\n        return dict(info, kind=\'emit\', type=\'Color\', emission=emission)\\n    if depth >= budget.limits[\'max_depth\']:\\n        state[\'unresolved_leaves\'] += 1\\n        return None\\n    best = None\\n    for feature_index, name in enumerate(FEATURES):\\n        budget.guard()\\n        values = sorted({examples[i][0][feature_index] for i in indices})\\n        predicates = [(\'eq\', value) for value in values]\\n        if name in NUMERIC:\\n            predicates += [(\'le\', value) for value in values[:-1]]\\n        for operation, value in predicates:\\n            budget.spend(\'predicate_work\', len(indices))\\n            yes = [i for i in indices if (examples[i][0][feature_index] == value if operation == \'eq\'\\n                                         else examples[i][0][feature_index] <= value)]\\n            yes_set = set(yes)\\n            no = [i for i in indices if i not in yes_set]\\n            if not _support(yes, examples, budget.limits) or not _support(no, examples, budget.limits):\\n                continue\\n            left_error, _ = _fit_emit(yes, examples, emissions, budget)\\n            right_error, _ = _fit_emit(no, examples, emissions, budget)\\n            key = (left_error + right_error, 0 if operation == \'eq\' else 1, feature_index, value)\\n            if best is None or key < best[0]:\\n                best = (key, name, operation, value, values, yes, no)\\n    if best is None:\\n        state[\'unresolved_leaves\'] += 1\\n        return None\\n    _, name, operation, value, values, yes_indices, no_indices = best\\n    yes = _construct(yes_indices, examples, emissions, budget, depth + 1, state)\\n    no = _construct(no_indices, examples, emissions, budget, depth + 1, state)\\n    if yes is None or no is None:\\n        return None\\n    return dict(info, kind=\'if\', type=\'Color\',\\n                predicate={\'feature\': name, \'operation\': operation, \'value\': value,\\n                           \'observed_values\': values}, yes=yes, no=no)\\n\\n\\ndef validate_model(model):\\n    if type(model) is not dict or set(model) != {\'schema\', \'features\', \'background\', \'tree\', \'training_demo_sha256\', \'model_sha256\'}:\\n        raise Failure(\'MODEL_SCHEMA\')\\n    if (model[\'schema\'] != \'CORA_CONDITIONAL_COLOR_TREE_V1\' or model[\'features\'] != list(FEATURES)\\n            or type(model[\'background\']) is not int or model[\'background\'] != 0):\\n        raise Failure(\'MODEL_FEATURE_CONTRACT\')\\n    if type(model[\'training_demo_sha256\']) is not str or len(model[\'training_demo_sha256\']) != 64:\\n        raise Failure(\'MODEL_PROVENANCE\')\\n    count = 0\\n    def walk(node, depth):\\n        nonlocal count\\n        count += 1\\n        if depth > 4 or count > 31 or type(node) is not dict or node.get(\'type\') != \'Color\':\\n            raise Failure(\'TREE_TYPE_OR_SIZE\')\\n        if (type(node.get(\'support\')) is not int or node[\'support\'] < 2 or\\n                type(node.get(\'support_demos\')) is not int or node[\'support_demos\'] < 2):\\n            raise Failure(\'TREE_SUPPORT\')\\n        if node.get(\'kind\') == \'emit\':\\n            if set(node) != {\'kind\', \'type\', \'emission\', \'support\', \'support_demos\'}:\\n                raise Failure(\'EMIT_FIELDS\')\\n            spec = node[\'emission\']\\n            if type(spec) is not dict:\\n                raise Failure(\'EMISSION_SCHEMA\')\\n            if spec.get(\'kind\') == \'feature\':\\n                if set(spec) != {\'kind\', \'feature\'} or spec[\'feature\'] not in FEATURES[:5]:\\n                    raise Failure(\'EMISSION_FEATURE\')\\n            elif spec.get(\'kind\') == \'constant\':\\n                if set(spec) != {\'kind\', \'color\'} or type(spec[\'color\']) is not int or not 0 <= spec[\'color\'] <= 9:\\n                    raise Failure(\'EMISSION_COLOR\')\\n            else:\\n                raise Failure(\'EMISSION_KIND\')\\n        elif node.get(\'kind\') == \'if\':\\n            if set(node) != {\'kind\', \'type\', \'predicate\', \'yes\', \'no\', \'support\', \'support_demos\'}:\\n                raise Failure(\'IF_FIELDS\')\\n            pred = node[\'predicate\']\\n            if (type(pred) is not dict or set(pred) != {\'feature\', \'operation\', \'value\', \'observed_values\'}\\n                    or pred[\'feature\'] not in FEATURES or pred[\'operation\'] not in (\'eq\', \'le\')\\n                    or pred[\'operation\'] == \'le\' and pred[\'feature\'] not in NUMERIC\\n                    or type(pred[\'value\']) is not int or type(pred[\'observed_values\']) is not list\\n                    or not pred[\'observed_values\'] or any(type(v) is not int for v in pred[\'observed_values\'])\\n                    or pred[\'observed_values\'] != sorted(set(pred[\'observed_values\']))\\n                    or pred[\'value\'] not in pred[\'observed_values\']):\\n                raise Failure(\'PREDICATE_SCHEMA\')\\n            walk(node[\'yes\'], depth + 1); walk(node[\'no\'], depth + 1)\\n        else:\\n            raise Failure(\'TREE_KIND\')\\n    walk(model[\'tree\'], 0)\\n    payload = {key: value for key, value in model.items() if key != \'model_sha256\'}\\n    if digest(payload) != model[\'model_sha256\']:\\n        raise Failure(\'MODEL_HASH\')\\n    return count\\n\\n\\ndef _apply_features(model, features, budget):\\n    values = []\\n    for feature in features:\\n        budget.guard()\\n        node = model[\'tree\']\\n        while node[\'kind\'] == \'if\':\\n            budget.spend(\'predicate_work\', 1)\\n            pred = node[\'predicate\']; value = feature[FEATURES.index(pred[\'feature\'])]\\n            if value not in pred[\'observed_values\']:\\n                raise Abstention(\'UNSEEN_ROUTING_FEATURE_VALUE\')\\n            truth = value == pred[\'value\'] if pred[\'operation\'] == \'eq\' else value <= pred[\'value\']\\n            node = node[\'yes\'] if truth else node[\'no\']\\n        budget.spend(\'predicate_work\', 1)\\n        color = _emit(node[\'emission\'], feature)\\n        if not 0 <= color <= 9:\\n            raise Abstention(\'UNOBSERVED_BOUNDARY_EMISSION\')\\n        values.append(color)\\n    return values\\n\\n\\ndef predict(model, grid, caps=None, check=lambda: None):\\n    limits = _limits(caps); budget = _Budget(limits, check)\\n    validate_model(model)\\n    prediction, reason = None, \'PREDICTED_ALL_PIXELS\'\\n    try:\\n        features = _features(grid, budget)\\n        values = _apply_features(model, features, budget)\\n        prediction = [values[r * len(grid[0]):(r + 1) * len(grid[0])] for r in range(len(grid))]\\n    except (Abstention, _Limit) as error:\\n        if budget.external and budget.external[-1] is error:\\n            raise\\n        reason = str(error)\\n    elapsed = time.process_time() - budget.started\\n    if elapsed >= limits[\'CPU_s\']:\\n        prediction, reason = None, \'THROUGH_RETURN_CPU_CAP_UNKNOWN\'\\n    return {\'prediction\': prediction, \'status\': \'PREDICTED\' if prediction is not None else \'ABSTAINED\',\\n            \'reason\': reason, \'model_sha256\': model[\'model_sha256\'],\\n            \'cost\': dict(budget.used, CPU_s=elapsed), \'query_truth_received\': False}\\n\\n\\ndef train(retained, caps=None, check=lambda: None):\\n    limits = _limits(caps); budget = _Budget(limits, check)\\n    if type(retained) is not list or not 2 <= len(retained) <= 10:\\n        raise Failure(\'AT_LEAST_TWO_RETAINED_DEMOS_REQUIRED\')\\n    examples, emissions, state, model = [], None, {\'nodes\': 0, \'unresolved_leaves\': 0}, None\\n    reason = \'NO_EXACT_TREE_WITHIN_DECLARED_SEARCH\'\\n    try:\\n        colors = set()\\n        for pair in retained:\\n            budget.guard()\\n            if type(pair) is not dict or set(pair) != {\'input\', \'output\'}:\\n                raise Failure(\'RETAINED_DEMO_FIELDS_ONLY\')\\n            valid_grid(pair[\'input\']); valid_grid(pair[\'output\'])\\n            if (len(pair[\'input\']), len(pair[\'input\'][0])) != (len(pair[\'output\']), len(pair[\'output\'][0])):\\n                raise Failure(\'SAME_SHAPED_PAIRS_REQUIRED\')\\n            features = _features(pair[\'input\'], budget)\\n            targets = [value for row in pair[\'output\'] for value in row]\\n            if len(examples) + len(features) > limits[\'max_samples\']:\\n                raise _Limit(\'SAMPLE_CAP_UNKNOWN\')\\n            # A duplicate pair does not manufacture independent leaf support.\\n            demo_identity = digest(pair)\\n            examples.extend((feature, target, demo_identity) for feature, target in zip(features, targets))\\n            colors.update(targets)\\n        emissions = _emissions(colors)\\n        tree = _construct(list(range(len(examples))), examples, emissions, budget, 0, state)\\n        if tree is not None:\\n            payload = {\'schema\': \'CORA_CONDITIONAL_COLOR_TREE_V1\', \'features\': list(FEATURES),\\n                       \'background\': 0, \'tree\': tree, \'training_demo_sha256\': digest(retained)}\\n            candidate = dict(payload, model_sha256=digest(payload))\\n            validate_model(candidate)\\n            # Fresh feature extraction/execution for every retained pair shares\\n            # the original training budget; a fitted table is not a verifier.\\n            for pair in retained:\\n                actual = _apply_features(candidate, _features(pair[\'input\'], budget), budget)\\n                desired = [value for row in pair[\'output\'] for value in row]\\n                if actual != desired:\\n                    raise Failure(\'FRESH_REPLAY_DISAGREEMENT\')\\n            model, reason = candidate, \'EXACT_ALL_DEMO_CONDITIONAL_PROGRAM\'\\n    except _Limit as error:\\n        if budget.external and budget.external[-1] is error:\\n            raise\\n        reason = str(error)\\n    elapsed = time.process_time() - budget.started\\n    if elapsed >= limits[\'CPU_s\']:\\n        model, reason = None, \'THROUGH_RETURN_CPU_CAP_UNKNOWN\'\\n    return {\'model\': model, \'status\': \'VERIFIED_RETAINED_FIT\' if model else \'UNKNOWN\',\\n            \'reason\': reason, \'samples\': len(examples), \'constructed_nodes\': state[\'nodes\'],\\n            \'unresolved_leaves\': state[\'unresolved_leaves\'], \'cost\': dict(budget.used, CPU_s=elapsed),\\n            \'caps\': limits, \'query_or_task_id_received\': False, \'primitive_semantics_learned\': False,\\n            \'learned_capability\': \'conditional_Color_program_over_fixed_features\'}\\n", "r29_cpu_package_20261101/worker.py": "\\"\\"\\"Isolated child entry: install running CPU/wall guards before solver imports.\\"\\"\\"\\nimport sys\\nimport time\\nimport signal\\nimport resource\\n\\n# Minimal guard bootstrap CPU is included in process_time(), and parent wall\\n# deadline covers interpreter startup. No task or heavy dependency is read here.\\n_boot_cpu = time.process_time()\\nclass GuardTimeout(BaseException):\\n    pass\\n\\ndef _alarm(*unused):\\n    raise GuardTimeout(\'RUNNING_PROCESS_CAP\')\\n\\nsignal.signal(signal.SIGPROF, _alarm)\\nsignal.signal(signal.SIGALRM, _alarm)\\n# Supplied caps are available before JSON/path/solver imports.\\n_cpu_s, _wall_deadline = float(sys.argv[3]), float(sys.argv[4])\\nif _cpu_s <= time.process_time() or _wall_deadline <= time.monotonic():\\n    raise SystemExit(3)\\nresource.setrlimit(resource.RLIMIT_CPU, (max(1, int(_cpu_s) + 1), max(2, int(_cpu_s) + 2)))\\nsignal.setitimer(signal.ITIMER_PROF, max(.000001, _cpu_s - time.process_time()))\\nsignal.setitimer(signal.ITIMER_REAL, max(.000001, _wall_deadline - time.monotonic()))\\n\\nimport os\\nimport json\\nimport hashlib\\nfrom pathlib import Path\\n\\n\\ndef main():\\n    request, response = map(Path, sys.argv[1:3])\\n    raw = json.loads(request.read_text())\\n    cpu_s, deadline = _cpu_s, _wall_deadline\\n    def check():\\n        if time.process_time() >= cpu_s or time.monotonic() >= deadline:\\n            raise GuardTimeout(\'RUNNING_PROCESS_CAP\')\\n    here = Path(__file__).resolve().parent\\n    manifest = json.loads((here / \'MANIFEST.json\').read_text())\\n    for relative, expected in manifest[\'files\'].items():\\n        check()\\n        path = here / relative\\n        if path.is_symlink() or not path.resolve().is_relative_to(here) or hashlib.sha256(path.read_bytes()).hexdigest() != expected:\\n            raise ValueError(\'PACKAGE_MANIFEST_MISMATCH\')\\n    sys.path.insert(0, str(here / \'vendor\'))\\n    sys.path.insert(0, str(here))\\n    import solver\\n    predictions, report = solver.solve(raw[\'task\'], check)\\n    check()\\n    result = {\'predictions\': predictions, \'report\': dict(report, worker_process_CPU_s=time.process_time(),\\n                                                       minimal_guard_bootstrap_CPU_s=_boot_cpu,\\n                                                       manifest_identity=hashlib.sha256((here / \'MANIFEST.json\').read_bytes()).hexdigest())}\\n    with response.open(\'x\') as stream:\\n        json.dump(result, stream, separators=(\',\', \':\'), allow_nan=False)\\n        stream.flush(); os.fsync(stream.fileno())\\n    check()\\n\\nif __name__ == \'__main__\':\\n    try:\\n        main()\\n    except BaseException:\\n        raise SystemExit(3)\\n", "sfb/__init__.py": "\\"\\"\\"Configuration B\'s side code: the CAST side process (cast_side) and the attempt-2 merge (merge).\\"\\"\\"\\n", "sfb/cast_side.py": "\\"\\"\\"Configuration B: the CAST side process. It runs on CPU next to the neural GPU workers. It calls the CAST package\\nnamed in the build manifest once per puzzle, in a freshly forked child, and writes one result file per puzzle.\\n\\n  python -m sfb.cast_side --challenges C --out DIR --package NAME --entry FUNC --deadline UNIX_TS\\n      [--mode grids|evidence] [--task-ids a,b,...] [--cpu-s 15] [--wall-s 30] [--grace-s 10] [--nice 10]\\n\\nModes:\\n- grids: the entry returns one grid or None per test input, or a details dict with \\"grids\\" (cast_v1\'s\\n  cast_candidates and the constructor-package contract).\\n- evidence: the entry is cast_selective_v2\'s evaluate(task, cpu_s, wall_s). The child turns its report into plain\\n  JSON: status, and per query the first complete candidate and each policy\'s accepted grid. The parent re-validates\\n  every grid. The merge later applies the package\'s own insert_attempt2 to that report.\\n\\nWhat it guarantees (each item is tested in configs/tests/test_cfg_b.py):\\n- State resets between puzzles. CAST code runs only inside a child forked for that puzzle, and the parent never calls\\n  it. So nothing a call leaves behind reaches the next puzzle: module caches, globals, timers, signal handlers or\\n  memory.\\n- Caps are enforced at process level, not only configured.\\n  - The package\'s own caps are passed through: `cpu_s` and `wall_s`.\\n  - Backstop 1: the child gets RLIMIT_CPU of cpu_s + grace_s seconds, enforced by the kernel.\\n  - Backstop 2: the parent SIGKILLs the child at wall_s + grace_s seconds of wall time.\\n  - The child is always reaped (wait4), and its CPU time is recorded.\\n- Total budget: a puzzle starts only if it can finish (wall_s + grace_s) before --deadline. The notebook additionally\\n  kills this whole process group at the deadline.\\n- Output is re-validated here: one entry per test input, each None or a rectangular grid of 1..30 x 1..30 ints in\\n  0..9. Anything else becomes None and is counted as malformed. Result files are written atomically.\\n- Fail-safe: no exception leaves this process. A failure leaves that puzzle without a CAST answer, so it keeps\\n  configuration A\'s attempts.\\n- Only the puzzle\'s demonstrations and test inputs are passed to CAST (the task dict is rebuilt from those fields).\\n\\"\\"\\"\\nimport argparse\\nimport json\\nimport os\\nimport select\\nimport signal\\nimport sys\\nimport time\\n\\nfor _v in (\\"OMP_NUM_THREADS\\", \\"OPENBLAS_NUM_THREADS\\", \\"MKL_NUM_THREADS\\", \\"NUMEXPR_NUM_THREADS\\"):\\n    os.environ.setdefault(_v, \\"1\\")                    # one core for CAST; set before numpy is imported\\n\\nMAX_PAYLOAD = 4 << 20                                  # bytes; a larger reply is treated as malformed\\n\\n\\ndef valid_grid(g):\\n    \\"\\"\\"A valid ARC grid: a non-empty rectangular list of lists of ints 0..9, at most 30 x 30 (bools rejected).\\"\\"\\"\\n    if not isinstance(g, list) or not 1 <= len(g) <= 30:\\n        return False\\n    w = None\\n    for row in g:\\n        if not isinstance(row, list) or not 1 <= len(row) <= 30:\\n            return False\\n        if w is None:\\n            w = len(row)\\n        elif len(row) != w:\\n            return False\\n        for v in row:\\n            if type(v) is not int or not 0 <= v <= 9:\\n                return False\\n    return True\\n\\n\\ndef clean_task(t):\\n    \\"\\"\\"Only what the solver may see: demonstration pairs and test inputs.\\"\\"\\"\\n    return {\\"train\\": [{\\"input\\": p[\\"input\\"], \\"output\\": p[\\"output\\"]} for p in t[\\"train\\"]],\\n            \\"test\\": [{\\"input\\": x[\\"input\\"]} for x in t[\\"test\\"]]}\\n\\n\\ndef atomic_json(path, obj):\\n    tmp = f\\"{path}.tmp{os.getpid()}\\"\\n    with open(tmp, \\"w\\") as f:\\n        json.dump(obj, f)\\n        f.flush()\\n        os.fsync(f.fileno())\\n    os.replace(tmp, path)\\n\\n\\ndef _to_grid(g):\\n    \\"\\"\\"A grid as a list of lists of ints, None, or the marker \\"MALFORMED\\".\\"\\"\\"\\n    if g is None:\\n        return None\\n    try:\\n        import numpy as np\\n        a = np.asarray(g)\\n        if a.ndim != 2 or a.dtype.kind not in \\"iub\\":\\n            return \\"MALFORMED\\"\\n        return [[int(v) for v in row] for row in a.tolist()]\\n    except Exception:                                  # noqa: BLE001\\n        return \\"MALFORMED\\"\\n\\n\\ndef compact_evidence(rep):\\n    \\"\\"\\"The parts of an evaluate() report that the merge needs, as plain JSON (computed in the child).\\"\\"\\"\\n    if not isinstance(rep, dict):\\n        return {\\"status\\": \\"MALFORMED_REPORT\\", \\"queries\\": None}\\n    out = {\\"status\\": str(rep.get(\\"status\\"))[:60], \\"version\\": str(rep.get(\\"version\\"))[:60],\\n           \\"n_demos\\": rep.get(\\"n_demos\\") if isinstance(rep.get(\\"n_demos\\"), int) else None,\\n           \\"cpu_s\\": rep.get(\\"cpu_s\\") if isinstance(rep.get(\\"cpu_s\\"), (int, float)) else None,\\n           \\"wall_s\\": rep.get(\\"wall_s\\") if isinstance(rep.get(\\"wall_s\\"), (int, float)) else None, \\"queries\\": []}\\n    qs = rep.get(\\"queries\\")\\n    if not isinstance(qs, list):\\n        out.update(status=\\"MALFORMED_REPORT\\", queries=None)\\n        return out\\n    for q in qs:\\n        if not isinstance(q, dict):\\n            out[\\"queries\\"].append(None)\\n            continue\\n        acc = q.get(\\"accepted\\") if isinstance(q.get(\\"accepted\\"), dict) else {}\\n        accm = q.get(\\"accepted_module\\") if isinstance(q.get(\\"accepted_module\\"), dict) else {}\\n        cm = q.get(\\"candidate_module\\")\\n        out[\\"queries\\"].append({\\"candidate\\": _to_grid(q.get(\\"candidate\\")),\\n                               \\"candidate_module\\": None if cm is None else str(cm)[:20],\\n                               \\"accepted\\": {str(k)[:10]: _to_grid(v) for k, v in acc.items()},\\n                               \\"accepted_module\\": {str(k)[:10]: (None if v is None else str(v)[:20])\\n                                                   for k, v in accm.items()}})\\n    return out\\n\\n\\ndef _child(fn, task, kwargs, w, cpu_limit, mode=\\"grids\\"):\\n    \\"\\"\\"Runs in the forked child: never returns.\\"\\"\\"\\n    code = 0\\n    try:\\n        import resource\\n        resource.setrlimit(resource.RLIMIT_CORE, (0, 0))      # a CPU-limit kill leaves no core file\\n        resource.setrlimit(resource.RLIMIT_CPU, (cpu_limit, cpu_limit + 5))\\n        res = fn(task, **kwargs)\\n        if mode == \\"evidence\\":\\n            res = compact_evidence(res)\\n        payload = json.dumps({\\"ok\\": True, \\"res\\": res}, default=str).encode()\\n    except BaseException as e:                         # noqa: BLE001  fail-safe by contract\\n        payload = json.dumps({\\"ok\\": False, \\"error\\": f\\"{type(e).__name__}: {str(e)[:200]}\\"}).encode()\\n        code = 3\\n    try:\\n        view = memoryview(payload)\\n        while view:\\n            n = os.write(w, view)\\n            view = view[n:]\\n    except BaseException:                              # noqa: BLE001\\n        code = 4\\n    os._exit(code)\\n\\n\\ndef run_one(fn, task, kwargs, wall_limit, cpu_limit, mode=\\"grids\\"):\\n    \\"\\"\\"Fork, run, collect. Returns (reply dict or None, child record).\\"\\"\\"\\n    r, w = os.pipe()\\n    t0 = time.time()\\n    pid = os.fork()\\n    if pid == 0:                                       # child\\n        os.close(r)\\n        _child(fn, task, kwargs, w, cpu_limit, mode)\\n    os.close(w)\\n    buf, killed, too_big = bytearray(), False, False\\n    end = t0 + wall_limit\\n    while True:\\n        left = end - time.time()\\n        if left <= 0:\\n            os.kill(pid, signal.SIGKILL)\\n            killed = True\\n            break\\n        ready, _, _ = select.select([r], [], [], min(left, 1.0))\\n        if not ready:\\n            continue\\n        chunk = os.read(r, 1 << 16)\\n        if not chunk:                                  # end of file: the child has finished writing\\n            break\\n        buf += chunk\\n        if len(buf) > MAX_PAYLOAD:\\n            os.kill(pid, signal.SIGKILL)\\n            killed = too_big = True\\n            break\\n    os.close(r)\\n    _, status, ru = os.wait4(pid, 0)                   # always reap\\n    rec = {\\"wall_s\\": round(time.time() - t0, 3), \\"cpu_s\\": round(ru.ru_utime + ru.ru_stime, 3),\\n           \\"exit\\": os.WEXITSTATUS(status) if os.WIFEXITED(status) else None,\\n           \\"signal\\": os.WTERMSIG(status) if os.WIFSIGNALED(status) else None,\\n           \\"killed_by_parent\\": killed, \\"too_big\\": too_big}\\n    reply = None\\n    if not killed and buf:\\n        try:\\n            reply = json.loads(bytes(buf))\\n        except ValueError:\\n            reply = None\\n    return reply, rec\\n\\n\\ndef interpret(reply, rec, n_test):\\n    \\"\\"\\"The reply as {grids, status, module, program, ...}; malformed parts become None.\\"\\"\\"\\n    out = {\\"grids\\": [None] * n_test, \\"module\\": [None] * n_test, \\"program\\": [None] * n_test, \\"malformed\\": 0}\\n    if rec[\\"killed_by_parent\\"]:\\n        out[\\"status\\"] = \\"KILLED_TOO_BIG\\" if rec[\\"too_big\\"] else \\"KILLED_WALL\\"\\n        return out\\n    if rec[\\"signal\\"] is not None:\\n        out[\\"status\\"] = \\"KILLED_CPU\\" if rec[\\"signal\\"] in (signal.SIGXCPU, signal.SIGKILL) else f\\"SIGNAL_{rec[\'signal\']}\\"\\n        return out\\n    if not isinstance(reply, dict) or not reply.get(\\"ok\\"):\\n        out[\\"status\\"] = \\"ERROR_CHILD\\" if isinstance(reply, dict) else \\"ERROR_NO_REPLY\\"\\n        out[\\"error\\"] = reply.get(\\"error\\") if isinstance(reply, dict) else None\\n        return out\\n    res = reply.get(\\"res\\")\\n    grids = res.get(\\"grids\\") if isinstance(res, dict) else res\\n    out[\\"status\\"] = str(res.get(\\"status\\", \\"OK\\"))[:60] if isinstance(res, dict) else \\"OK\\"\\n    if isinstance(res, dict):\\n        out[\\"version\\"] = str(res.get(\\"version\\"))[:60]\\n        out[\\"cast_cpu_s\\"] = res.get(\\"cpu_s\\") if isinstance(res.get(\\"cpu_s\\"), (int, float)) else None\\n    if not isinstance(grids, list) or len(grids) != n_test:\\n        out[\\"malformed\\"] = n_test\\n        out[\\"status\\"] = \\"MALFORMED_SHAPE\\"\\n        return out\\n    for j, g in enumerate(grids):\\n        if g is None:\\n            continue\\n        if valid_grid(g):\\n            out[\\"grids\\"][j] = g\\n            if isinstance(res, dict):\\n                for k in (\\"module\\", \\"program\\"):\\n                    v = res.get(k)\\n                    if isinstance(v, list) and j < len(v):\\n                        out[k][j] = str(v[j])[:300] if v[j] is not None else None\\n        else:\\n            out[\\"malformed\\"] += 1\\n    return out\\n\\n\\ndef interpret_evidence(reply, rec, n_test):\\n    \\"\\"\\"An evidence reply: kill and error handling as for grids; every grid in the report re-validated.\\"\\"\\"\\n    out = {\\"grids\\": [None] * n_test, \\"module\\": [None] * n_test, \\"program\\": [None] * n_test, \\"malformed\\": 0,\\n           \\"evidence\\": None}\\n    if rec[\\"killed_by_parent\\"]:\\n        out[\\"status\\"] = \\"KILLED_TOO_BIG\\" if rec[\\"too_big\\"] else \\"KILLED_WALL\\"\\n        return out\\n    if rec[\\"signal\\"] is not None:\\n        out[\\"status\\"] = \\"KILLED_CPU\\" if rec[\\"signal\\"] in (signal.SIGXCPU, signal.SIGKILL) else f\\"SIGNAL_{rec[\'signal\']}\\"\\n        return out\\n    if not isinstance(reply, dict) or not reply.get(\\"ok\\"):\\n        out[\\"status\\"] = \\"ERROR_CHILD\\" if isinstance(reply, dict) else \\"ERROR_NO_REPLY\\"\\n        out[\\"error\\"] = reply.get(\\"error\\") if isinstance(reply, dict) else None\\n        return out\\n    ev = reply.get(\\"res\\")\\n    if not isinstance(ev, dict) or not isinstance(ev.get(\\"queries\\"), list) or len(ev[\\"queries\\"]) != n_test:\\n        out.update(status=\\"MALFORMED_SHAPE\\", malformed=n_test)\\n        return out\\n    clean_q = []\\n    for j, q in enumerate(ev[\\"queries\\"]):\\n        if not isinstance(q, dict):\\n            clean_q.append({\\"candidate\\": None, \\"candidate_module\\": None, \\"accepted\\": {}, \\"accepted_module\\": {}})\\n            out[\\"malformed\\"] += 1\\n            continue\\n        c = q.get(\\"candidate\\")\\n        if c is not None and not valid_grid(c):\\n            out[\\"malformed\\"] += 1\\n            c = None\\n        acc = {}\\n        for k, g in (q.get(\\"accepted\\") or {}).items():\\n            if g is not None and not valid_grid(g):\\n                out[\\"malformed\\"] += 1\\n                g = None\\n            acc[k] = g\\n        clean_q.append({\\"candidate\\": c, \\"candidate_module\\": q.get(\\"candidate_module\\"), \\"accepted\\": acc,\\n                        \\"accepted_module\\": q.get(\\"accepted_module\\") or {}})\\n        out[\\"grids\\"][j] = c\\n        out[\\"module\\"][j] = q.get(\\"candidate_module\\") if c is not None else None\\n    out[\\"status\\"] = str(ev.get(\\"status\\"))[:60]\\n    out[\\"version\\"] = ev.get(\\"version\\")\\n    out[\\"cast_cpu_s\\"] = ev.get(\\"cpu_s\\")\\n    out[\\"evidence\\"] = dict(ev, queries=clean_q)\\n    return out\\n\\n\\ndef main(argv=None):\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\"--challenges\\", required=True)\\n    ap.add_argument(\\"--out\\", required=True)\\n    ap.add_argument(\\"--package\\", required=True)\\n    ap.add_argument(\\"--entry\\", required=True)\\n    ap.add_argument(\\"--deadline\\", type=float, required=True)\\n    ap.add_argument(\\"--task-ids\\", default=\\"\\")\\n    ap.add_argument(\\"--cpu-s\\", type=float, default=15.0)\\n    ap.add_argument(\\"--wall-s\\", type=float, default=30.0)\\n    ap.add_argument(\\"--grace-s\\", type=float, default=10.0)\\n    ap.add_argument(\\"--nice\\", type=int, default=10)\\n    ap.add_argument(\\"--details\\", type=int, default=1)\\n    ap.add_argument(\\"--mode\\", default=\\"grids\\", choices=[\\"grids\\", \\"evidence\\"])\\n    a = ap.parse_args(argv)\\n    os.makedirs(a.out, exist_ok=True)\\n    t_start = time.time()\\n    st = {\\"t_start\\": t_start, \\"deadline\\": a.deadline, \\"package\\": a.package, \\"entry\\": a.entry, \\"cpu_s\\": a.cpu_s,\\n          \\"wall_s\\": a.wall_s, \\"grace_s\\": a.grace_s, \\"done\\": 0, \\"answered_tasks\\": 0, \\"answered_outputs\\": 0,\\n          \\"malformed_outputs\\": 0, \\"statuses\\": {}, \\"child_cpu_s\\": 0.0, \\"stopped\\": None, \\"error\\": None,\\n          \\"python\\": sys.version.split()[0], \\"pid\\": os.getpid()}\\n    status_path = os.path.join(a.out, \\"status.json\\")\\n    try:\\n        try:\\n            os.nice(a.nice)\\n        except OSError:\\n            pass\\n        import importlib\\n        mod = importlib.import_module(a.package)\\n        fn = getattr(mod, a.entry)\\n        st[\\"version\\"] = str(getattr(mod, \\"VERSION\\", None))\\n        try:\\n            import numpy\\n            st[\\"numpy\\"] = numpy.__version__\\n        except Exception:                              # noqa: BLE001\\n            st[\\"numpy\\"] = None\\n        ch = json.load(open(a.challenges))\\n        ids = [x for x in a.task_ids.split(\\",\\") if x in ch] if a.task_ids else list(ch)\\n        st[\\"n_tasks\\"] = len(ids)\\n        atomic_json(status_path, st)\\n        kwargs = {\\"cpu_s\\": a.cpu_s, \\"wall_s\\": a.wall_s}\\n        if a.details and a.mode == \\"grids\\":\\n            kwargs[\\"details\\"] = True\\n        st[\\"mode\\"] = a.mode\\n        wall_limit = a.wall_s + a.grace_s\\n        cpu_limit = int(a.cpu_s + a.grace_s) + 1\\n        for tid in ids:\\n            if time.time() + wall_limit > a.deadline:\\n                st[\\"stopped\\"] = \\"budget\\"\\n                break\\n            t = clean_task(ch[tid])\\n            n_test = len(t[\\"test\\"])\\n            try:\\n                reply, rec = run_one(fn, t, kwargs, wall_limit, cpu_limit, a.mode)\\n                res = (interpret_evidence if a.mode == \\"evidence\\" else interpret)(reply, rec, n_test)\\n            except Exception as e:                     # noqa: BLE001  (e.g. fork failed): no answer for this task\\n                rec = {\\"error\\": f\\"{type(e).__name__}: {e}\\"[:200]}\\n                res = {\\"grids\\": [None] * n_test, \\"module\\": [None] * n_test, \\"program\\": [None] * n_test,\\n                       \\"malformed\\": 0, \\"status\\": \\"ERROR_PARENT\\"}\\n            res.update(task=tid, n_test=n_test, child=rec)\\n            atomic_json(os.path.join(a.out, f\\"{tid}.json\\"), res)\\n            st[\\"done\\"] += 1\\n            k = sum(g is not None for g in res[\\"grids\\"])\\n            st[\\"answered_outputs\\"] += k\\n            st[\\"answered_tasks\\"] += int(k > 0)\\n            st[\\"malformed_outputs\\"] += res.get(\\"malformed\\", 0)\\n            st[\\"statuses\\"][res[\\"status\\"]] = st[\\"statuses\\"].get(res[\\"status\\"], 0) + 1\\n            st[\\"child_cpu_s\\"] = round(st[\\"child_cpu_s\\"] + float(rec.get(\\"cpu_s\\", 0.0) or 0.0), 3)\\n            st[\\"t_now\\"] = time.time()\\n            atomic_json(status_path, st)\\n        else:\\n            st[\\"stopped\\"] = \\"finished\\"\\n    except BaseException as e:                         # noqa: BLE001  fail-safe by contract\\n        st[\\"error\\"] = f\\"{type(e).__name__}: {str(e)[:300]}\\"\\n        st[\\"stopped\\"] = st[\\"stopped\\"] or \\"error\\"\\n    st[\\"wall_s\\"] = round(time.time() - t_start, 2)\\n    try:\\n        atomic_json(status_path, st)\\n    except BaseException:                              # noqa: BLE001\\n        pass\\n    print(json.dumps({k: st.get(k) for k in (\\"done\\", \\"n_tasks\\", \\"answered_tasks\\", \\"answered_outputs\\", \\"stopped\\",\\n                                              \\"error\\", \\"wall_s\\")}), flush=True)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "sfo/__init__.py": "\\"\\"\\"Configuration O: the CORA-only CPU entry (CAST v1 core alone, no neural model).\\"\\"\\"\\n", "sfo/driver_plus.py": "\\"\\"\\"Configuration O+: CORA-only CPU entry with an ordered list of constructor packages (the constructor hook).\\n\\n  python -m sfo.driver_plus --challenges C --out RUN_DIR --submission SUB --packages JSON\\n      [--deadline-s 39600] [--reserve-s 600] [--shards N] [--grace-s 10] [--poll-s 20]\\n\\n`--packages`: a JSON list in priority order, each {\\"name\\", \\"entry\\", \\"cpu_s\\", \\"wall_s\\", \\"details\\"}. Example: cast_v1\\n(cast_candidates, 15/30, details 1) then the reviewed R29 package (candidates, 12/24, details 0; its API has no\\ndetails argument).\\n\\nAttempts, per query:\\n- attempt_1 is the first package\'s valid answer, in priority order;\\n- attempt_2 is the next distinct valid answer from another package;\\n- otherwise each attempt is the fixed fallback, a copy of the test input.\\n\\nPackages run one after another, each over every task in `--shards` side processes (`sfb.cast_side`). Each puzzle\\nruns in a fresh forked child, with the package\'s own caps and process-level backstops. The submission is valid\\nbefore any work starts, is rebuilt atomically every poll, and is written once more at the end. Total-time guard: no\\ntask starts after deadline minus reserve, and at the deadline every side process group is killed. A package that\\nhas not finished contributes the results it has. Any failure keeps the last valid file.\\n\\"\\"\\"\\nimport argparse\\nimport json\\nimport os\\nimport signal\\nimport subprocess\\nimport sys\\nimport time\\n\\nfrom sfb.cast_side import atomic_json, valid_grid\\n\\n\\ndef copy_of(g):\\n    return [[int(v) for v in row] for row in g]\\n\\n\\ndef package_answers(r, n):\\n    if not isinstance(r, dict) or r.get(\\"status\\") != \\"OK\\":\\n        return [None] * n\\n    g = r.get(\\"grids\\")\\n    if not isinstance(g, list) or len(g) != n:\\n        return [None] * n\\n    return [x if x is not None and valid_grid(x) else None for x in g]\\n\\n\\ndef build_submission(ch, results):\\n    \\"\\"\\"results: one {task_id: cast_side result} per package, in priority order.\\"\\"\\"\\n    sub, counts = {}, {\\"outputs\\": 0, \\"attempt_1_from_package\\": 0, \\"attempt_2_from_package\\": 0,\\n                       \\"per_package_answers\\": [0] * len(results)}\\n    for tid, t in ch.items():\\n        n = len(t[\\"test\\"])\\n        per = [package_answers(res.get(tid), n) for res in results]\\n        entry = []\\n        for j, x in enumerate(t[\\"test\\"]):\\n            fb = copy_of(x[\\"input\\"])\\n            got = []\\n            for k, ans in enumerate(per):\\n                if ans[j] is not None:\\n                    counts[\\"per_package_answers\\"][k] += 1\\n                    got.append(ans[j])\\n            a1 = got[0] if got else fb\\n            a2 = next((g for g in got[1:] if g != a1), None)\\n            counts[\\"attempt_1_from_package\\"] += bool(got)\\n            counts[\\"attempt_2_from_package\\"] += a2 is not None\\n            entry.append({\\"attempt_1\\": a1, \\"attempt_2\\": a2 if a2 is not None else copy_of(x[\\"input\\"])})\\n            counts[\\"outputs\\"] += 1\\n        sub[tid] = entry\\n    return sub, counts\\n\\n\\ndef schema_problems(sub, ch):\\n    problems = []\\n    for tid, t in ch.items():\\n        e = sub.get(tid)\\n        if not isinstance(e, list) or len(e) != len(t[\\"test\\"]):\\n            problems.append(f\\"{tid}: wrong entry count\\")\\n            continue\\n        for j, a in enumerate(e):\\n            if not isinstance(a, dict) or set(a) != {\\"attempt_1\\", \\"attempt_2\\"} or \\\\\\n                    not valid_grid(a[\\"attempt_1\\"]) or not valid_grid(a[\\"attempt_2\\"]):\\n                problems.append(f\\"{tid}[{j}]: bad entry\\")\\n    if set(sub) - set(ch):\\n        problems.append(\\"unknown task ids\\")\\n    return problems\\n\\n\\ndef read_results(dirs, ids):\\n    res = {}\\n    for d in dirs:\\n        for tid in ids:\\n            p = os.path.join(d, f\\"{tid}.json\\")\\n            if tid in res or not os.path.exists(p):\\n                continue\\n            try:\\n                with open(p) as f:\\n                    res[tid] = json.load(f)\\n            except (OSError, ValueError):\\n                pass\\n    return res\\n\\n\\ndef main(argv=None):\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\"--challenges\\", required=True)\\n    ap.add_argument(\\"--out\\", required=True)\\n    ap.add_argument(\\"--submission\\", required=True)\\n    ap.add_argument(\\"--packages\\", required=True)\\n    ap.add_argument(\\"--deadline-s\\", type=float, default=11 * 3600)\\n    ap.add_argument(\\"--reserve-s\\", type=float, default=600)\\n    ap.add_argument(\\"--shards\\", type=int, default=0)\\n    ap.add_argument(\\"--grace-s\\", type=float, default=10.0)\\n    ap.add_argument(\\"--poll-s\\", type=float, default=20.0)\\n    ap.add_argument(\\"--code-dir\\", default=os.path.dirname(os.path.dirname(os.path.abspath(__file__))))\\n    a = ap.parse_args(argv)\\n    t0 = time.time()\\n    pkgs = json.loads(a.packages)\\n    os.makedirs(a.out, exist_ok=True)\\n    with open(a.challenges) as f:\\n        ch = json.load(f)\\n    ids = list(ch)\\n    sub, _ = build_submission(ch, [{} for _ in pkgs])\\n    atomic_json(a.submission, sub)                             # valid before any CORA work\\n    n = a.shards or min(4, os.cpu_count() or 1)\\n    n = max(1, min(n, len(ids) or 1))\\n    shards = [ids[k::n] for k in range(n)]\\n    deadline = t0 + a.deadline_s\\n    cast_deadline = deadline - a.reserve_s\\n    env = dict(os.environ, CUDA_VISIBLE_DEVICES=\\"\\", OMP_NUM_THREADS=\\"1\\", OPENBLAS_NUM_THREADS=\\"1\\",\\n               MKL_NUM_THREADS=\\"1\\", NUMEXPR_NUM_THREADS=\\"1\\", PYTHONHASHSEED=\\"0\\", PYTHONUNBUFFERED=\\"1\\")\\n    st = {\\"t_start\\": t0, \\"deadline\\": deadline, \\"shards\\": n, \\"n_tasks\\": len(ids), \\"packages\\": pkgs,\\n          \\"python\\": sys.version.split()[0], \\"cpu_count\\": os.cpu_count(), \\"per_package\\": []}\\n    dirs_per_pkg = []\\n    stopped = None\\n    for k, pk in enumerate(pkgs):\\n        dirs, procs = [], []\\n        t_pk = time.time()\\n        for s, part in enumerate(shards):\\n            d = os.path.join(a.out, f\\"pkg{k}_{pk[\'name\']}\\", f\\"shard_{s}\\")\\n            os.makedirs(d, exist_ok=True)\\n            dirs.append(d)\\n            cmd = [sys.executable, \\"-m\\", \\"sfb.cast_side\\", \\"--challenges\\", a.challenges, \\"--out\\", d,\\n                   \\"--package\\", pk[\\"name\\"], \\"--entry\\", pk[\\"entry\\"], \\"--deadline\\", str(cast_deadline),\\n                   \\"--cpu-s\\", str(pk[\\"cpu_s\\"]), \\"--wall-s\\", str(pk[\\"wall_s\\"]), \\"--grace-s\\", str(a.grace_s),\\n                   \\"--details\\", str(int(pk.get(\\"details\\", 1))), \\"--nice\\", \\"0\\", \\"--task-ids\\", \\",\\".join(part)]\\n            log = open(d + \\".log\\", \\"w\\")\\n            procs.append(subprocess.Popen(cmd, cwd=a.code_dir, env=env, stdout=log, stderr=log,\\n                                          start_new_session=True))\\n        dirs_per_pkg.append(dirs)\\n        while True:\\n            t_poll = time.time() + a.poll_s\\n            while time.time() < t_poll and any(p.poll() is None for p in procs):\\n                time.sleep(1.0)\\n            try:\\n                sub, counts = build_submission(ch, [read_results(ds, ids) for ds in dirs_per_pkg])\\n                if not schema_problems(sub, ch):\\n                    atomic_json(a.submission, sub)\\n            except Exception as e:                             # noqa: BLE001  keep the last valid file\\n                st[\\"poll_error\\"] = repr(e)[:200]\\n            st.update(t=time.time(), package_running=k, alive=[p.poll() is None for p in procs])\\n            atomic_json(os.path.join(a.out, \\"status.json\\"), st)\\n            if not any(p.poll() is None for p in procs):\\n                break\\n            if time.time() > deadline:\\n                stopped = \\"deadline reached; side processes killed\\"\\n                break\\n        for p in procs:\\n            if p.poll() is None:\\n                try:\\n                    os.killpg(p.pid, signal.SIGKILL)\\n                except OSError:\\n                    pass\\n            try:\\n                p.wait(timeout=30)\\n            except subprocess.TimeoutExpired:\\n                pass\\n        res_k = read_results(dirs, ids)\\n        statuses = {}\\n        for r in res_k.values():\\n            statuses[r.get(\\"status\\")] = statuses.get(r.get(\\"status\\"), 0) + 1\\n        st[\\"per_package\\"].append({\\"name\\": pk[\\"name\\"], \\"wall_s\\": round(time.time() - t_pk, 2),\\n                                  \\"tasks_with_result\\": len(res_k), \\"statuses\\": statuses,\\n                                  \\"child_cpu_s\\": round(sum(float((r.get(\\"child\\") or {}).get(\\"cpu_s\\") or 0)\\n                                                           for r in res_k.values()), 2)})\\n        if stopped:\\n            st[\\"watchdog\\"] = stopped\\n            break\\n    results = [read_results(ds, ids) for ds in dirs_per_pkg] + [{} for _ in range(len(pkgs) - len(dirs_per_pkg))]\\n    sub, counts = build_submission(ch, results)\\n    problems = schema_problems(sub, ch)\\n    if not problems:\\n        atomic_json(a.submission, sub)\\n    st.update(t=time.time(), wall_s=round(time.time() - t0, 2), schema_problems=problems[:20],\\n              n_schema_problems=len(problems), **counts)\\n    atomic_json(os.path.join(a.out, \\"status.json\\"), st)\\n    print(json.dumps({k: st.get(k) for k in (\\"wall_s\\", \\"n_tasks\\", \\"outputs\\", \\"attempt_1_from_package\\",\\n                                              \\"attempt_2_from_package\\", \\"per_package_answers\\",\\n                                              \\"n_schema_problems\\")}), flush=True)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"}')
FILE_SHA = {'cast_v1/__init__.py': 'cc04c63281a2205d741e48d5e1571739ca737ce8518cc6f2953a08990d7e52dd', 'cast_v1/_nd.py': '18e53fa3cbbc6ddae32211ef99d2bb5b1dffec10bc5e4e8dae1422cdf999826f', 'cast_v1/absdom.py': '7a7664b78481a6c7301e329536c36ed109462157303c128207e0db97182394d6', 'cast_v1/api.py': '5c27592033998c799bf9abf5e478efcaf86133d9c234167a65e9e448e1c0f87d', 'cast_v1/candidates.py': '792825e4b6f067494ace0eceff766f9c78788e35a651d254ef22c1f925c200aa', 'cast_v1/grids.py': '42c349f4c672bc506e3bfec30ff40c63fa86056c561a2b7ea33f32dec931a7f0', 'cast_v1/ops.py': '380102ef6cac9196bfc8c8713326196533ed83ccb1d292df459799fc9e80a18e', 'cast_v1/rewrite.py': '86381a6ae3f37215d4919621c3be1a3393c4943174f568b8e27159f1330eb88b', 'cast_v1/search.py': 'a6b6261906ed1be03a823912546df486cd2ea9cacc3a4cca05f06ad0cc27e48d', 'cast_v1/symmetry.py': '0040c861706b6ecf173bb2acb48c0d6aacad64c77fb8e73a90412ee28668b04c', 'r29_cpu_package_20261101/LICENSE_STATUS.md': '961a190f57ed497a78e721803016757e0c6ad4c94643304f7e4de74c390f485c', 'r29_cpu_package_20261101/MANIFEST.json': '11eb2ee9cd10f1507a0ffdcc6ae069de2efed79eb569788888aebcc3fd685ef7', 'r29_cpu_package_20261101/README.md': '9c433733b3940cb438802117b0ad283cfd850cbfd3927a5fb83982f1d76359ea', 'r29_cpu_package_20261101/VENDORED_PROVENANCE.json': '32edf758c7e6032ac962fde6c49bbfc4d53042067ef3ec24105e03cdce728bc5', 'r29_cpu_package_20261101/__init__.py': '101ebb8b163749f38937ca955a046e30832941f4fddcbdf14677926f1f89d556', 'r29_cpu_package_20261101/api.py': '5006260b401e41099c463835635f8af49d859d76c337f81c4ed6c9a8d7162cbc', 'r29_cpu_package_20261101/cli.py': '52e9450ffe7e28c26ddbfb5a3bed202b4ac2502a73e97af41758180e44ad6d8d', 'r29_cpu_package_20261101/solver.py': '21101fb10a55fcafe950a84ad704701c93dce3fae8739d02ac0975d7dd096706', 'r29_cpu_package_20261101/test_package.py': '8f29a7472684c554baa0273f696e27992fb0c30a0ec188cfca594553233eea8f', 'r29_cpu_package_20261101/vendor/cage/__init__.py': '8b671d53028b09a9a552f1b361e1a68a1ef06df067c05c41c2dae7fd4ce6417f', 'r29_cpu_package_20261101/vendor/cage/dependencies.py': 'fa95fe4b143c05363e48f2cf9f2c97a68eede57fff9ed1456985c9d6b424f4df', 'r29_cpu_package_20261101/vendor/cage/programs.py': 'b6ccc7a65a19414a59bc26d101e7f15825c619d1389095bc8a1ae3febbf32a82', 'r29_cpu_package_20261101/vendor/cage/v2b/__init__.py': 'e3c9233781d1c3b917a01e737a436be2d0b0961b19d6d34068e7d7a1c0821a8c', 'r29_cpu_package_20261101/vendor/cage/v2b/conditional_model.py': '39ce0652ce78519c0472b4627bc3e0640e2c1b9aae29f3656d9535f3df2db993', 'r29_cpu_package_20261101/vendor/cage/v2b/goal_constructor.py': 'd067f9d3a5a696be9ec6829e5bdcc2947714bb8ab04ff88e495ef9b948f7092c', 'r29_cpu_package_20261101/vendor/experiments/independent_synthesis_20261029/round25/capability_growth/library.py': '974dab6c9e98df7f024081c7c4599e7b9242be39cc5f0763987fe5e60ea83dc6', 'r29_cpu_package_20261101/vendor/experiments/independent_synthesis_20261029/round25/diagnosis_constructor/engine.py': '009788d7071929240a5231032eaeb3e783b1313e92aea14a9a81043dccb56559', 'r29_cpu_package_20261101/vendor/experiments/independent_synthesis_20261029/round26/coverage_constructor/engine.py': '0e513986bd94565fa0c572f4dbd6dfd5a698a8fa87188f2f876e27f607919a82', 'r29_cpu_package_20261101/vendor/experiments/independent_synthesis_20261029/round27/learned_local_rules/model.py': '110bcb0c19801d22fc27de7f38b34d1c7e7aca65afdbcb7d0b10980b00a3fbd6', 'r29_cpu_package_20261101/worker.py': '703ab07bb55c424d7409cf7c18603ec437624fae1ac5cb650b0f5c87d1e6ac4d', 'sfb/__init__.py': '839f1efcafe73c661e0f7b726f169ace61e0c76c4fc1cc7623fd39533cf59829', 'sfb/cast_side.py': 'c072eccd3595bce1a3cd3fd028cc2f378b6fcf6ada838b4e7ea772dd22494853', 'sfo/__init__.py': '4247997a9b0bf1831bde45b068fecc9a51f28eb782077725b45cf3bd42f25fc0', 'sfo/driver_plus.py': '157d41b2c0caa1cf9d1afb4e5ff50144a150d673bf7f2452787dbd3955dc3077'}
for _name, _text in FILES.items():
    if hashlib.sha256(_text.encode("utf-8")).hexdigest() != FILE_SHA[_name]:
        raise RuntimeError("embedded file hash mismatch: " + _name)
    _p = os.path.join(CODE_DIR, _name)
    os.makedirs(os.path.dirname(_p), exist_ok=True)
    with open(_p, "w", encoding="utf-8") as _f:
        _f.write(_text)
print("code written", CODE_SHA, len(FILES), flush=True)


In [ ]:

# Cell 2 of 2: the constructor packages in priority order on CPU (sfo/driver_plus.py), then a final schema guard.
# attempt_1 = the first package's answer; attempt_2 = the next distinct answer from another package; else a copy
# of the test input.
import glob, json, os, subprocess, sys, time
KROOT = os.environ.get("SF_KAGGLE_ROOT", "/kaggle")
T0 = time.time()
SUB = KROOT + "/working/submission.json"
COMP_DIR = None
for _d in (KROOT + "/input/competitions/arc-prize-2026-arc-agi-2", KROOT + "/input/arc-prize-2026-arc-agi-2"):
    if os.path.isdir(_d):
        COMP_DIR = _d
        break
if COMP_DIR is None:
    COMP_DIR = os.path.dirname(glob.glob(KROOT + "/input/**/arc-agi_test_challenges.json", recursive=True)[0])
if os.environ.get("SF_REHEARSE_ON") == "evaluation":
    CH = os.path.join(COMP_DIR, "arc-agi_evaluation_challenges.json")
else:
    CH = os.path.join(COMP_DIR, "arc-agi_test_challenges.json")
with open(CH) as _f:
    ch = json.load(_f)

def _fallback_all():
    return {t: [{"attempt_1": x["input"], "attempt_2": x["input"]} for x in v["test"]] for t, v in ch.items()}

def _write(obj):
    _tmp = SUB + ".tmp"
    with open(_tmp, "w") as _f:
        json.dump(obj, _f)
        _f.flush()
        os.fsync(_f.fileno())
    os.replace(_tmp, SUB)

def _grid_ok(g):
    if not isinstance(g, list) or not 1 <= len(g) <= 30:
        return False
    w = len(g[0]) if isinstance(g[0], list) else -1
    return 1 <= w <= 30 and all(isinstance(r, list) and len(r) == w and
                                all(type(v) is int and 0 <= v <= 9 for v in r) for r in g)

def _valid(sub):
    return isinstance(sub, dict) and set(sub) == set(ch) and all(
        isinstance(sub[t], list) and len(sub[t]) == len(v["test"]) and
        all(isinstance(a, dict) and set(a) == {"attempt_1", "attempt_2"} and _grid_ok(a["attempt_1"]) and
            _grid_ok(a["attempt_2"]) for a in sub[t]) for t, v in ch.items())

_write(_fallback_all())                                       # valid before anything else runs
info = {"config": "O+", "code_sha256": CODE_SHA, "challenges": CH, "n_tasks": len(ch),
        "rehearsal": os.environ.get("SF_REHEARSE_ON") == "evaluation",
        "rerun_env": os.environ.get("KAGGLE_IS_COMPETITION_RERUN"), "python": sys.version.split()[0],
        "cpu_count": os.cpu_count(), "affinity": len(os.sched_getaffinity(0)),
        "packages": json.loads(os.environ["SF_OP_PACKAGES"])}
for _m in ("numpy", "scipy"):
    try:
        info[_m] = __import__(_m).__version__
    except Exception:
        info[_m] = None
with open(KROOT + "/working/run_info.json", "w") as _f:
    json.dump(info, _f, indent=1)
print(json.dumps(info), flush=True)
_cmd = [sys.executable, "-m", "sfo.driver_plus", "--challenges", CH, "--out", KROOT + "/working/run",
        "--submission", SUB, "--packages", os.environ["SF_OP_PACKAGES"],
        "--deadline-s", os.environ["SF_O_DEADLINE_S"], "--shards", os.environ.get("SF_O_SHARDS", "0"),
        "--grace-s", os.environ["SF_O_GRACE_S"], "--code-dir", CODE_DIR]
try:
    _r = subprocess.run(_cmd, cwd=CODE_DIR, timeout=float(os.environ["SF_O_DEADLINE_S"]) + 900)
    _rc = _r.returncode
except Exception as _e:                                       # the last valid file stays
    _rc = repr(_e)
try:
    with open(SUB) as _f:
        _ok = _valid(json.load(_f))
except Exception:
    _ok = False
if not _ok:
    _write(_fallback_all())
print("driver returncode", _rc, "final file valid", _ok, "wall_s", round(time.time() - T0, 1), flush=True)
